# RoPE 2D-grid versus runtime-1D — MICROBENCH_ONLY

Method labels: native = original Qwen2; old = target-tested runtime-T/stride 1D source; runtime = new runtime-T/stride 2D-batch-grid source. This is not a comparison with the original constexpr kernel. No training, model load, optimizer, score, ownership-share or admission test. This notebook runs only short numerical gates and native/old/runtime RoPE fixtures on the deployed server. It does not install a solution into SOUP_REPO. Upload auto-starts; only the authorized parent may submit.

The exact original CUDA numerical/PTX preflight runs separately before measurements. Every GPU child is capped at 80 seconds, and external service caps stay authoritative. Each method uses a fresh process and private empty Triton cache. First-call synchronized wall includes compilation where needed, allocation/dispatch and device work; it is not a pure compile timer. Warm CUDA-event stream intervals are reported separately. Original telemetry stays enabled. Five observed T values (133,235,317,485,512), synthetic B4/Hq12/Hk2/D128 pairs, CB1 assumed, Q projection-order and K contiguous gradients. Single fixed native/old/runtime order; not a counterbalanced task-speed result.

No downloads, installations, credentials or automatic retries. Exact runtime/source, global GPU lock, guardian cleanup, quarantine and idle gates fail closed. An incomplete job is censored, never success. Full-task profile remains HELD pending target results and review.


In [ ]:
import os,base64,hashlib,tempfile
from pathlib import Path
assert os.environ.get('SOUP_REPO') and os.environ.get('SOUP_RESULTS')
assert Path('/kaggle/working').is_dir()
PAYLOAD = {'solution/__init__.py': 'IiIiT25seSBvcmlnaW5hbCBSb1BFIGFwcGxpY2F0aW9uIGtlcm5lbHM7IG9mZmljaWFsIHN0YXJ0ZXIga2VybmVscyBhcmUgbm90IGNsYWltZWQuIiIiCmRlZiBhcHBseSgpOgogICAgZnJvbSAucm90YXJ5IGltcG9ydCBwYXRjaF9xd2VuMl9yb3RhcnkKICAgIGlmIG5vdCBwYXRjaF9xd2VuMl9yb3RhcnkoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlBpbm5lZCBRd2VuMiByb3Rhcnkgc291cmNlIGd1YXJkIHJlZnVzZWQgaW5zdGFsbGF0aW9uIikKCgpkZWYgcXVldWVfc21va2UoKToKICAgIGZyb20gLmNwdV9jb250cmFjdCBpbXBvcnQgcnVuCiAgICByZXR1cm4gcnVuKCkK', 'solution/cpu_contract.py': 'IiIiU21hbGwgQ1BVLW9ubHkgcXVldWUgc21va2U6IG5vIENVREEgcXVlcmllcywga2VybmVscywgbW9kZWwgbG9hZGluZyBvciBkb3dubG9hZHMuIiIiCmltcG9ydCBpdGVydG9vbHMKaW1wb3J0IHRvcmNoCgoKZGVmIHJlZmVyZW5jZShxLCBrLCBjLCBzKToKICAgIGZyb20gdHJhbnNmb3JtZXJzLm1vZGVscy5xd2VuMi5tb2RlbGluZ19xd2VuMiBpbXBvcnQgYXBwbHlfcm90YXJ5X3Bvc19lbWIKICAgIHdoaWxlIGdldGF0dHIoYXBwbHlfcm90YXJ5X3Bvc19lbWIsICJfb3JpZ2luYWxfcm90YXJ5X2NhbmRpZGF0ZSIsIEZhbHNlKToKICAgICAgICBhcHBseV9yb3RhcnlfcG9zX2VtYiA9IGFwcGx5X3JvdGFyeV9wb3NfZW1iLl9fd3JhcHBlZF9fCiAgICByZXR1cm4gYXBwbHlfcm90YXJ5X3Bvc19lbWIocSwgaywgYywgcykKCgpkZWYgZXhhY3QoYSwgYik6CiAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhLCBiLCBhdG9sPTAsIHJ0b2w9MCwgZXF1YWxfbmFuPVRydWUpCiAgICBtYXNrID0gYiA9PSAwCiAgICBhc3NlcnQgdG9yY2guZXF1YWwodG9yY2guc2lnbmJpdChhW21hc2tdKSwgdG9yY2guc2lnbmJpdChiW21hc2tdKSkKCgpkZWYgaW5wdXRzKGI9MiwgdD03LCBocT0xMiwgaGs9MiwgZD0xMjgsIGNiPTEsIGRldmljZT0iY3B1Iik6CiAgICBnZW5lcmF0b3IgPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPWRldmljZSkubWFudWFsX3NlZWQoNDcxMzEpCiAgICBkZWYgcmFuZChzaGFwZSk6CiAgICAgICAgcmV0dXJuIHRvcmNoLnJhbmRuKHNoYXBlLCBnZW5lcmF0b3I9Z2VuZXJhdG9yLCBkZXZpY2U9ZGV2aWNlKQogICAgcSA9IHJhbmQoKGIsIHQsIGhxLCBkKSkudHJhbnNwb3NlKDEsIDIpLnJlcXVpcmVzX2dyYWRfKCkKICAgIGsgPSByYW5kKChiLCB0LCBoaywgZCkpLnRyYW5zcG9zZSgxLCAyKS5yZXF1aXJlc19ncmFkXygpCiAgICByZXR1cm4gcSwgaywgcmFuZCgoY2IsdCxkKSksIHJhbmQoKGNiLHQsZCkpCgoKZGVmIHJ1bigpOgogICAgZnJvbSAucm90YXJ5IGltcG9ydCBfc3VwcG9ydGVkLCBuYXRpdmVfYWRqb2ludAogICAgYmVmb3JlID0gdG9yY2guZ2V0X3JuZ19zdGF0ZSgpLmNsb25lKCkKICAgIHEsayxjLHMgPSBpbnB1dHMoKQogICAgeSA9IHJlZmVyZW5jZShxLGssYyxzKQogICAgZ3MgPSB0dXBsZSh0b3JjaC5vbmVzX2xpa2UoeCkgKiA2NTUzNiBmb3IgeCBpbiB5KQogICAgZXhwZWN0ZWQgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKHksIChxLGspLCBncykKICAgIGZvciBnLGUgaW4gemlwKGdzLGV4cGVjdGVkKToKICAgICAgICBleGFjdChuYXRpdmVfYWRqb2ludChnLGMscyksIGUpCiAgICBhc3NlcnQgX3N1cHBvcnRlZChxLGssYyxzLGN1ZGFfb25seT1GYWxzZSkKICAgIGFzc2VydCBub3QgX3N1cHBvcnRlZChxLGssYyxzKQogICAgYXNzZXJ0IG5vdCBfc3VwcG9ydGVkKHEscSxjLHMsY3VkYV9vbmx5PUZhbHNlKQogICAgYXNzZXJ0IG5vdCBfc3VwcG9ydGVkKHEuY29udGlndW91cygpLGssYyxzLGN1ZGFfb25seT1GYWxzZSkKICAgIGMucmVxdWlyZXNfZ3JhZF8oKQogICAgYXNzZXJ0IG5vdCBfc3VwcG9ydGVkKHEsayxjLHMsY3VkYV9vbmx5PUZhbHNlKQogICAgYXNzZXJ0IHRvcmNoLmVxdWFsKGJlZm9yZSwgdG9yY2guZ2V0X3JuZ19zdGF0ZSgpKQogICAgcmV0dXJuIGRpY3QocGFzc2VkPVRydWUsIHNjb3BlPSJDUFVfTUVUQURBVEFfQU5EX05BVElWRV9BREpPSU5UX09OTFkiLCBncHVfZXhlY3V0ZWQ9RmFsc2UsCiAgICAgICAgICAgICAgICB0cml0b25fZXhlY3V0ZWQ9RmFsc2UsIHNjb3JlPU5vbmUsIGFkbWlzc2lvbj0iVU5LTk9XTiIsIHRvcmNoPXRvcmNoLl9fdmVyc2lvbl9fKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBpbXBvcnQganNvbgogICAgcHJpbnQoanNvbi5kdW1wcyhydW4oKSwgaW5kZW50PTIpKQo=', 'solution/gpu_preflight.py': 'IiIiRXhwbGljaXQgZnV0dXJlIFQ0IGdhdGUsIG5vdCBydW4gZHVyaW5nIHByZXBhcmF0aW9uLiBTbWFsbCBmaXh0dXJlIG1vZGVscyBvbmx5LgoKcHl0aG9uIC1tIHNvbHV0aW9uLmdwdV9wcmVmbGlnaHQgLS1vdXQgL25ldy9wcmVmbGlnaHQvcGF0aApBbGwgc291cmNlL3ZlcnNpb25zLCBudW1lcmljYWwgYm91bmRhcnkgYW5kIFBUWCBjaGVja3MgZmFpbCBjbG9zZWQuIFRoZXJlIGlzIG5vCnNjb3JlL2FkbWlzc2lvbiBoZXJlLiBLZXJuZWwgZXhlY3V0aW9uIGlzIGEgcHJlZmxpZ2h0IHNlcGFyYXRlIGZyb20gY29sZCB0cmFpbmluZy4KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCByZQppbXBvcnQgdHJhY2ViYWNrCgoKZGVmIG1haW4oKToKICAgIHA9YXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKTtwLmFkZF9hcmd1bWVudCgiLS1vdXQiLHJlcXVpcmVkPVRydWUpO2E9cC5wYXJzZV9hcmdzKCkKICAgIG91dD1QYXRoKGEub3V0KS5yZXNvbHZlKCk7b3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1GYWxzZSkKICAgIHJlc3VsdD1kaWN0KHN0YXR1cz0iRkFJTEVEIixncHVfZXhlY3V0ZWQ9RmFsc2Usc2NvcmU9Tm9uZSxhZG1pc3Npb249IlVOS05PV04iLGNoZWNrcz1bXSkKICAgIHRyeToKICAgICAgICBpZiBvcy5lbnZpcm9uLmdldCgiVFJJVE9OX0lOVEVSUFJFVCIpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkdQVSBwcmVmbGlnaHQgcmVmdXNlcyBpbnRlcnByZXRlciBtb2RlIikKICAgICAgICBpbXBvcnQgdG9yY2gsdHJpdG9uLHRyYW5zZm9ybWVycwogICAgICAgIGZyb20gLiBpbXBvcnQgcm90YXJ5CiAgICAgICAgZnJvbSAuY3B1X2NvbnRyYWN0IGltcG9ydCBpbnB1dHMscmVmZXJlbmNlLGV4YWN0CiAgICAgICAgaWYgdHJpdG9uLl9fdmVyc2lvbl9fICE9ICIzLjYuMCIgb3IgdG9yY2guX192ZXJzaW9uX18gIT0gIjIuMTEuMCtjdTEyOCIgb3IgdHJhbnNmb3JtZXJzLl9fdmVyc2lvbl9fICE9ICI1LjE2LjEiOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkV4YWN0IHRhcmdldCBUb3JjaC9Ucml0b24vVHJhbnNmb3JtZXJzIHZlcnNpb25zIHJlcXVpcmVkIikKICAgICAgICBpZiB0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpIT0xIG9yIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApIT0iVGVzbGEgVDQiOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk9uZSB2aXNpYmxlIFRlc2xhIFQ0IHJlcXVpcmVkIikKICAgICAgICBvcy5lbnZpcm9uWyJTT1VQX1JPVEFSWV9DT0RFR0VOX0RJUiJdPXN0cihvdXQvInB0eCIpCiAgICAgICAgIyBPcmlnaW5hbCBjYWxsYWJsZSBzdGF5cyBpbnRhY3QgdW50aWwgYWxsIGFyaXRobWV0aWMgdGVzdHMgaGF2ZSBydW4uCiAgICAgICAgZm9yIGIsdCxjYiBpbiAoKDEsMSwxKSwoMiwxNywyKSwoNCw1MTEsMSksKDQsNTEyLDQpKToKICAgICAgICAgICAgcSxrLGMscz1pbnB1dHMoYj1iLHQ9dCxjYj1jYixkZXZpY2U9ImN1ZGEiKQogICAgICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KCJjdWRhIixkdHlwZT10b3JjaC5mbG9hdDE2KToKICAgICAgICAgICAgICAgIGV4cGVjdGVkPXJlZmVyZW5jZShxLGssYyxzKQogICAgICAgICAgICAgICAgYWN0dWFsPXJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLGssYyxzKQogICAgICAgICAgICBncz10dXBsZSh0b3JjaC5saW5zcGFjZSgtNjU1MzYsNjU1MzYseC5udW1lbCgpLGRldmljZT0iY3VkYSIpLnJlc2hhcGUoeC5zaGFwZSkgZm9yIHggaW4gYWN0dWFsKQogICAgICAgICAgICBlZz10b3JjaC5hdXRvZ3JhZC5ncmFkKGV4cGVjdGVkLChxLGspLGdzKQogICAgICAgICAgICBhZz10b3JjaC5hdXRvZ3JhZC5ncmFkKGFjdHVhbCwocSxrKSxncykKICAgICAgICAgICAgZm9yIGFhLGVlIGluIHppcCgoKmFjdHVhbCwqYWcpLCgqZXhwZWN0ZWQsKmVnKSk6ZXhhY3QoYWEsZWUpCiAgICAgICAgICAgIGZvciB5LHggaW4gemlwKGFjdHVhbCwocSxrKSk6YXNzZXJ0IHkuc3RyaWRlKCk9PXguc3RyaWRlKCkKICAgICAgICAgICAgcmVzdWx0WyJjaGVja3MiXS5hcHBlbmQoZGljdChuYW1lPSJzY2FsZWRfYm91bmRhcnkiLGI9Yix0PXQsY29lZmZpY2llbnRfYmF0Y2g9Y2IpKQogICAgICAgICMgRXhoYXVzdGl2ZSBzaWduZWQtemVybyBhcml0aG1ldGljOyBjYWxsIG9uZS1pbnB1dCBrZXJuZWwgYm91bmRhcnkgb25seSwKICAgICAgICAjIHdpdGhvdXQgYnJvYWRlbmluZyB0aGUgcHJvZHVjdGlvbiBkaXNwYXRjaGVyIGJleW9uZCBEMTI4L0gxMjoyLgogICAgICAgIGltcG9ydCBpdGVydG9vbHMKICAgICAgICB2PXRvcmNoLnRlbnNvcihsaXN0KGl0ZXJ0b29scy5wcm9kdWN0KFswLiwtMC4sMS4sLTEuXSxyZXBlYXQ9NikpLGRldmljZT0iY3VkYSIpCiAgICAgICAgcT10b3JjaC56ZXJvcygoMSxsZW4odiksMSwyKSxkZXZpY2U9ImN1ZGEiKS50cmFuc3Bvc2UoMSwyKS5yZXF1aXJlc19ncmFkXygpCiAgICAgICAgaz1xLmRldGFjaCgpLmNsb25lKCkucmVxdWlyZXNfZ3JhZF8oKQogICAgICAgIGMscz12WzosOjJdLnJlc2hhcGUoMSxsZW4odiksMikuY29udGlndW91cygpLHZbOiwyOjRdLnJlc2hhcGUoMSxsZW4odiksMikuY29udGlndW91cygpCiAgICAgICAgZz12WzosNDpdLnJlc2hhcGVfYXMocSkKICAgICAgICBlPXJlZmVyZW5jZShxLGssYyxzKQogICAgICAgIGFjdHVhbD10dXBsZShyb3RhcnkuX0Zyb3plblJvUEUuYXBwbHkoeCxjLHMpIGZvciB4IGluIChxLGspKQogICAgICAgIGVnPXRvcmNoLmF1dG9ncmFkLmdyYWQoZSwocSxrKSwoZyxnKSk7YWc9dG9yY2guYXV0b2dyYWQuZ3JhZChhY3R1YWwsKHEsayksKGcsZykpCiAgICAgICAgZm9yIGFhLGVlIGluIHppcCgoKmFjdHVhbCwqYWcpLCgqZSwqZWcpKTpleGFjdChhYSxlZSkKICAgICAgICByZXN1bHRbImNoZWNrcyJdLmFwcGVuZChkaWN0KG5hbWU9ImV4aGF1c3RpdmVfc2lnbmVkX3plcm8iLGNvbWJpbmF0aW9ucz1sZW4odikpKQogICAgICAgICMgRmlyc3Qtb3JkZXIgbm9uZmluaXRlIGNsYXNzZXMvc3Vibm9ybWFsIGFyaXRobWV0aWM6IG5vIE5hTiBwYXlsb2FkIGNsYWltLgogICAgICAgIHY9dG9yY2gudGVuc29yKFswLiwtMC4sZmxvYXQoImluZiIpLGZsb2F0KCItaW5mIiksZmxvYXQoIm5hbiIpLDFlLTQwLC0xZS00MCwxLl0sZGV2aWNlPSJjdWRhIikKICAgICAgICBxPXYucmVwZWF0KDgpLnJlc2hhcGUoMSw4LDEsOCkudHJhbnNwb3NlKDEsMikucmVxdWlyZXNfZ3JhZF8oKTtrPXEuZGV0YWNoKCkuY2xvbmUoKS5yZXF1aXJlc19ncmFkXygpCiAgICAgICAgYz12LnJlcGVhdCg4KS5yZXNoYXBlKDEsOCw4KS5jb250aWd1b3VzKCk7cz1jLmZsaXAoLTEpLmNvbnRpZ3VvdXMoKQogICAgICAgIGU9cmVmZXJlbmNlKHEsayxjLHMpO2FjdHVhbD10dXBsZShyb3RhcnkuX0Zyb3plblJvUEUuYXBwbHkoeCxjLHMpIGZvciB4IGluIChxLGspKQogICAgICAgIGVnPXRvcmNoLmF1dG9ncmFkLmdyYWQoZSwocSxrKSwocS5kZXRhY2goKSxrLmRldGFjaCgpKSk7YWc9dG9yY2guYXV0b2dyYWQuZ3JhZChhY3R1YWwsKHEsayksKHEuZGV0YWNoKCksay5kZXRhY2goKSkpCiAgICAgICAgZm9yIGFhLGVlIGluIHppcCgoKmFjdHVhbCwqYWcpLCgqZSwqZWcpKTpleGFjdChhYSxlZSkKICAgICAgICByZXN1bHRbImNoZWNrcyJdLmFwcGVuZChkaWN0KG5hbWU9Im5vbmZpbml0ZV9jbGFzc2VzX3N1Ym5vcm1hbF9hbmRfc2lnbmVkX3plcm8iKSkKICAgICAgICAjIEV4ZXJjaXNlIHVuY2hhbmdlZCBTRFBBIGF1dG9jYXN0L0dRQSB3aXRoIEZQMzIgcS9rIGlucHV0cywgYm90aCBtYXNrIHJvdXRlcy4KICAgICAgICBmcm9tIHRyYW5zZm9ybWVycy5pbnRlZ3JhdGlvbnMuc2RwYV9hdHRlbnRpb24gaW1wb3J0IHNkcGFfYXR0ZW50aW9uX2ZvcndhcmQKICAgICAgICBmcm9tIHR5cGVzIGltcG9ydCBTaW1wbGVOYW1lc3BhY2UKICAgICAgICBmb3IgbWFza2VkIGluIChGYWxzZSwgVHJ1ZSk6CiAgICAgICAgICAgIHEsayxjLHM9aW5wdXRzKGI9Mix0PTE3LGRldmljZT0iY3VkYSIpCiAgICAgICAgICAgIHY9dG9yY2gucmFuZG4oKDIsMTcsMiwxMjgpLGRldmljZT0iY3VkYSIpLnRyYW5zcG9zZSgxLDIpLnJlcXVpcmVzX2dyYWRfKCkKICAgICAgICAgICAgb3duZXI9U2ltcGxlTmFtZXNwYWNlKG51bV9rZXlfdmFsdWVfZ3JvdXBzPTYsaXNfY2F1c2FsPVRydWUpCiAgICAgICAgICAgIG1hc2s9dG9yY2gub25lcygoMiwxLDE3LDE3KSxkZXZpY2U9ImN1ZGEiLGR0eXBlPXRvcmNoLmJvb2wpLnRyaWwoKSBpZiBtYXNrZWQgZWxzZSBOb25lCiAgICAgICAgICAgIGRlZiBhdHRlbnRpb24oZm4pOgogICAgICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgiY3VkYSIsZHR5cGU9dG9yY2guZmxvYXQxNik6CiAgICAgICAgICAgICAgICAgICAgcXIsa3I9Zm4ocSxrLGMscykKICAgICAgICAgICAgICAgICAgICByZXR1cm4gc2RwYV9hdHRlbnRpb25fZm9yd2FyZChvd25lcixxcixrcix2LG1hc2ssZHJvcG91dD0wLixzY2FsaW5nPTEyOCoqLTAuNSlbMF0KICAgICAgICAgICAgcm5nPXRvcmNoLmN1ZGEuZ2V0X3JuZ19zdGF0ZSgpLmNsb25lKCkKICAgICAgICAgICAgZSxhY3R1YWw9YXR0ZW50aW9uKHJlZmVyZW5jZSksYXR0ZW50aW9uKHJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYikKICAgICAgICAgICAgZXhhY3QoYWN0dWFsLGUpO2Fzc2VydCBhY3R1YWwuZHR5cGU9PXRvcmNoLmZsb2F0MTYKICAgICAgICAgICAgYXNzZXJ0IHRvcmNoLmVxdWFsKHJuZyx0b3JjaC5jdWRhLmdldF9ybmdfc3RhdGUoKSkKICAgICAgICAgICAgZz10b3JjaC5vbmVzX2xpa2UoZSkqNjQKICAgICAgICAgICAgZWc9dG9yY2guYXV0b2dyYWQuZ3JhZChlLChxLGssdiksZyk7YWc9dG9yY2guYXV0b2dyYWQuZ3JhZChhY3R1YWwsKHEsayx2KSxnKQogICAgICAgICAgICBmb3IgYWEsZWUgaW4gemlwKGFnLGVnKTp0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhYSxlZSxydG9sPTJlLTUsYXRvbD0yZS01LGVxdWFsX25hbj1UcnVlKQogICAgICAgICAgICByZXN1bHRbImNoZWNrcyJdLmFwcGVuZChkaWN0KG5hbWU9InVuY2hhbmdlZF9zZHBhX2F1dG9jYXN0X2dxYSIsbWFza2VkPW1hc2tlZCkpCiAgICAgICAgIyBVbmNoYW5nZWQgcmVhbCBRd2VuMkF0dGVudGlvbiBTRFBBIGJvdW5kYXJ5LCBubyBwcmV0cmFpbmVkIG1vZGVsIGxvYWQuCiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxzLnF3ZW4yIGltcG9ydCBtb2RlbGluZ19xd2VuMiBhcyBtb2QKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycy5tb2RlbHMucXdlbjIuY29uZmlndXJhdGlvbl9xd2VuMiBpbXBvcnQgUXdlbjJDb25maWcKICAgICAgICB0b3JjaC5tYW51YWxfc2VlZCg0NzEzMikKICAgICAgICBjZmc9UXdlbjJDb25maWcoaGlkZGVuX3NpemU9MTUzNixudW1fYXR0ZW50aW9uX2hlYWRzPTEyLG51bV9rZXlfdmFsdWVfaGVhZHM9MiwKICAgICAgICAgICAgICAgICAgICAgICBpbnRlcm1lZGlhdGVfc2l6ZT0zMixudW1faGlkZGVuX2xheWVycz0xLGF0dGVudGlvbl9kcm9wb3V0PTAuLAogICAgICAgICAgICAgICAgICAgICAgIHJvcGVfcGFyYW1ldGVycz17InJvcGVfdHlwZSI6ImRlZmF1bHQiLCJyb3BlX3RoZXRhIjoxXzAwMF8wMDAufSkKICAgICAgICBjZmcuX2F0dG5faW1wbGVtZW50YXRpb249InNkcGEiCiAgICAgICAgbW9kdWxlPW1vZC5Rd2VuMkF0dGVudGlvbihjZmcsMCkuY3VkYSgpLnRyYWluKCkKICAgICAgICBlbWI9bW9kLlF3ZW4yUm90YXJ5RW1iZWRkaW5nKGNmZykuY3VkYSgpCiAgICAgICAgZm9yIG1hc2tlZCBpbiAoRmFsc2UsVHJ1ZSk6CiAgICAgICAgICAgIGhpZGRlbj10b3JjaC5yYW5kbigoMiwxNywxNTM2KSxkZXZpY2U9ImN1ZGEiLHJlcXVpcmVzX2dyYWQ9VHJ1ZSkKICAgICAgICAgICAgY29lZmY9ZW1iKGhpZGRlbix0b3JjaC5hcmFuZ2UoMTcsZGV2aWNlPSJjdWRhIikudW5zcXVlZXplKDApKQogICAgICAgICAgICBtYXNrPXRvcmNoLm9uZXMoKDIsMSwxNywxNyksZGV2aWNlPSJjdWRhIixkdHlwZT10b3JjaC5ib29sKS50cmlsKCkgaWYgbWFza2VkIGVsc2UgTm9uZQogICAgICAgICAgICB2YXJpYWJsZXM9KGhpZGRlbiwqbW9kdWxlLnBhcmFtZXRlcnMoKSk7dXBzdHJlYW09dG9yY2gucmFuZG5fbGlrZShoaWRkZW4pKjY0CiAgICAgICAgICAgIG9yaWdpbmFsPW1vZC5hcHBseV9yb3RhcnlfcG9zX2VtYgogICAgICAgICAgICBybmc9dG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkuY2xvbmUoKQogICAgICAgICAgICBlPW1vZHVsZShoaWRkZW4sY29lZmYsbWFzaylbMF07ZWc9dG9yY2guYXV0b2dyYWQuZ3JhZChlLHZhcmlhYmxlcyx1cHN0cmVhbSkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgYXNzZXJ0IHJvdGFyeS5wYXRjaF9xd2VuMl9yb3RhcnkoKQogICAgICAgICAgICAgICAgY291bnQ9cm90YXJ5Ll9DT1VOVFNbImZvcndhcmRfbGF1bmNoX3JldHVybmVkIl0KICAgICAgICAgICAgICAgIGFjdHVhbD1tb2R1bGUoaGlkZGVuLGNvZWZmLG1hc2spWzBdO2FnPXRvcmNoLmF1dG9ncmFkLmdyYWQoYWN0dWFsLHZhcmlhYmxlcyx1cHN0cmVhbSkKICAgICAgICAgICAgICAgIGFzc2VydCByb3RhcnkuX0NPVU5UU1siZm9yd2FyZF9sYXVuY2hfcmV0dXJuZWQiXT09Y291bnQrMgogICAgICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICAgICAgbW9kLmFwcGx5X3JvdGFyeV9wb3NfZW1iPW9yaWdpbmFsCiAgICAgICAgICAgIGFzc2VydCB0b3JjaC5lcXVhbChybmcsdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkpCiAgICAgICAgICAgIGV4YWN0KGFjdHVhbCxlKQogICAgICAgICAgICAjIFVwc3RyZWFtIHNoYXJlZC1hbmNlc3RyeSBzY2hlZHVsaW5nIGFuZCBTRFBBIHJlZHVjdGlvbnMgY2FuIGRpZmZlcjsKICAgICAgICAgICAgIyByYXcgc2NhbGVkIGdyYWRpZW50cyBhcmUgY29tcGFyZWQsIG5vdCBkaXZpZGVkIHRvIGhpZGUgYSBtaXNtYXRjaC4KICAgICAgICAgICAgZm9yIGFhLGVlIGluIHppcChhZyxlZyk6dG9yY2gudGVzdGluZy5hc3NlcnRfY2xvc2UoYWEsZWUscnRvbD0yZS01LGF0b2w9MmUtNSxlcXVhbF9uYW49VHJ1ZSkKICAgICAgICAgICAgcmVzdWx0WyJjaGVja3MiXS5hcHBlbmQoZGljdChuYW1lPSJyZWFsX3F3ZW4yX3NkcGFfcm91dGUiLG1hc2tlZD1tYXNrZWQpKQogICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTtyZXN1bHRbImdwdV9leGVjdXRlZCJdPVRydWUKICAgICAgICBwdHhmaWxlcz1zb3J0ZWQoKG91dC8icHR4IikuZ2xvYigiKi5wdHgiKSk7YXNzZXJ0IHB0eGZpbGVzCiAgICAgICAgcHR4Y2hlY2tzPVtdCiAgICAgICAgZm9yIHBhdGggaW4gcHR4ZmlsZXM6CiAgICAgICAgICAgIHRleHQ9cGF0aC5yZWFkX3RleHQoKQogICAgICAgICAgICBhc3NlcnQgbm90IHJlLnNlYXJjaChyIlxiZm1hKD86XC58XHMpIix0ZXh0KSxwYXRoLm5hbWUKICAgICAgICAgICAgYXNzZXJ0IG5vdCByZS5zZWFyY2gociJcYig/Om11bHxhZGQpXC5bXlxuO10qZnR6Iix0ZXh0KSxwYXRoLm5hbWUKICAgICAgICAgICAgYXNzZXJ0IHJlLnNlYXJjaChyIlxibXVsKD86XC5ybik/XC5mMzIiLHRleHQpLHBhdGgubmFtZQogICAgICAgICAgICBhc3NlcnQgcmUuc2VhcmNoKHIiXGJhZGQoPzpcLnJuKT9cLmYzMiIsdGV4dCkscGF0aC5uYW1lCiAgICAgICAgICAgIHB0eGNoZWNrcy5hcHBlbmQoZGljdChmaWxlPXBhdGgubmFtZSxzaGEyNTY9aGFzaGxpYi5zaGEyNTYocGF0aC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpKSkKICAgICAgICByZXN1bHQudXBkYXRlKHN0YXR1cz0iUEFTU0VEX1BSRUZMSUdIVF9PTkxZIixwdHhfY2hlY2tzPXB0eGNoZWNrcyxjb3VudHM9ZGljdChyb3RhcnkuX0NPVU5UUykpCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShlcnJvcj1yZXByKGV4YyksdHJhY2ViYWNrPXRyYWNlYmFjay5mb3JtYXRfZXhjKCkpCiAgICBmaW5hbGx5OgogICAgICAgIChvdXQvInByZWZsaWdodC5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdCxpbmRlbnQ9MikrIlxuIikKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0LGluZGVudD0yKSkKICAgIHJhaXNlIFN5c3RlbUV4aXQoMCBpZiByZXN1bHRbInN0YXR1cyJdPT0iUEFTU0VEX1BSRUZMSUdIVF9PTkxZIiBlbHNlIDEpCgppZiBfX25hbWVfXz09Il9fbWFpbl9fIjptYWluKCkK', 'solution/rotary.py': 'IiIiT3JpZ2luYWwgb3V0LW9mLXBsYWNlIEZQMzIgaGFsZi1zcGxpdCBSb1BFOyBubyBub3JtYWxpemF0aW9uIG9yIGF0dGVudGlvbiBwYXRjaC4KClByb2R1Y3Rpb24gZGlzcGF0Y2ggaXMgQ1VEQS1vbmx5IGFuZCBmaXhlZC10YXNrIGdlb21ldHJ5LiBDUFUgaW50ZXJwcmV0YXRpb24gaXMKYW4gZXhwbGljaXRseSByZXF1ZXN0ZWQgdGVzdCBwYXRoLCBuZXZlciBhIHBlcmZvcm1hbmNlIG9yIEdQVSB2YWxpZGF0aW9uIGNsYWltLgoiIiIKaW1wb3J0IGZ1bmN0b29scwppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW5zcGVjdAppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKCmltcG9ydCB0b3JjaAppbXBvcnQgdHJpdG9uCmltcG9ydCB0cml0b24ubGFuZ3VhZ2UgYXMgdGwKCkVYUEVDVEVEX1FXRU5fQkxPQiA9ICJiNGZlZDk4ZmFmYWU4ZTRlZTM5YmUzYjhjZDc2NmIxYzg1YjA3OWNkIgpfQ09VTlRTID0gQ291bnRlcigpCgoKZGVmIF9ldmVudChraW5kLCAqKmV4dHJhKToKICAgIF9DT1VOVFNba2luZF0gKz0gMQogICAgbiA9IF9DT1VOVFNba2luZF0KICAgIGlmIG4gPT0gMSBvciBuICUgMjU2ID09IDA6CiAgICAgICAgcHJpbnQoIlJPVEFSWV9FVklERU5DRSAiICsganNvbi5kdW1wcyhkaWN0KGV2ZW50PWtpbmQsIGNvdW50PW4sICoqZXh0cmEpLCBzb3J0X2tleXM9VHJ1ZSksIGZsdXNoPVRydWUpCgoKZGVmIF9zYXZlX2NvZGVnZW4oY29tcGlsZWQsIG5hbWUpOgogICAgIyBPcHQtaW4gcHJlZmxpZ2h0LW9ubHkgZXZpZGVuY2UuIE1lYXN1cmVkIHJ1bnMgbmV2ZXIgc2V0IHRoaXMgZW52aXJvbm1lbnQga2V5LgogICAgZGlyZWN0b3J5ID0gb3MuZW52aXJvbi5nZXQoIlNPVVBfUk9UQVJZX0NPREVHRU5fRElSIikKICAgIGlmIGRpcmVjdG9yeSBhbmQgY29tcGlsZWQgaXMgbm90IE5vbmU6CiAgICAgICAgcHR4ID0gY29tcGlsZWQuYXNtLmdldCgicHR4IiwgIiIpCiAgICAgICAgaWYgbm90IHB0eDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJObyBnZW5lcmF0ZWQgUFRYIGF2YWlsYWJsZSBmb3IgcmVxdWlyZWQgcHJlZmxpZ2h0IGluc3BlY3Rpb24iKQogICAgICAgIHJvb3QgPSBQYXRoKGRpcmVjdG9yeSkKICAgICAgICByb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihwdHguZW5jb2RlKCkpLmhleGRpZ2VzdCgpCiAgICAgICAgKHJvb3QgLyAobmFtZSArICJfIiArIGRpZ2VzdFs6MTZdICsgIi5wdHgiKSkud3JpdGVfdGV4dChwdHgpCgoKQHRyaXRvbi5qaXQKZGVmIF9zaWduZmxpcF9mMzIoeCk6CiAgICAjIEV4YWN0IHVuYXJ5IG5lZ2F0aW9uLCBpbmNsdWRpbmcgLTAuMC4gTm90IHRoZSBub24tZXF1aXZhbGVudCBleHByZXNzaW9uIDAteC4KICAgIHJldHVybiAoeC50byh0bC51aW50MzIsIGJpdGNhc3Q9VHJ1ZSkgXiAweDgwMDAwMDAwKS50byh0bC5mbG9hdDMyLCBiaXRjYXN0PVRydWUpCgoKQHRyaXRvbi5qaXQoZG9fbm90X3NwZWNpYWxpemU9WyJUIiwgIk4iXSkKZGVmIHJvdGFyeV9hcHBseV9mMzIoWCwgQywgUywgWSwgSDogdGwuY29uc3RleHByLCBULAogICAgICAgICAgICAgICAgICAgICBEOiB0bC5jb25zdGV4cHIsIENCOiB0bC5jb25zdGV4cHIsIE4sCiAgICAgICAgICAgICAgICAgICAgIEJMT0NLOiB0bC5jb25zdGV4cHIpOgogICAgIyBGbGF0dGVuIHRoZSBkZW5zZSBCLFQsSCwoRC8yKSBwaHlzaWNhbCBvcmRlciBvZiB0aGUgcHJvamVjdGlvbiBzdG9yYWdlLgogICAgaSA9IHRsLnByb2dyYW1faWQoMCkgKiBCTE9DSyArIHRsLmFyYW5nZSgwLCBCTE9DSykKICAgIGhhbGYgPSBEIC8vIDIKICAgIGogPSBpICUgaGFsZgogICAgaCA9IChpIC8vIGhhbGYpICUgSAogICAgdCA9IGkgLy8gKGhhbGYgKiBIKQogICAgYiA9IHRsLnByb2dyYW1faWQoMSkKICAgIHhwID0gKChiICogVCArIHQpICogSCArIGgpICogRCArIGoKICAgIGNwID0gKChiIGlmIENCICE9IDEgZWxzZSAwKSAqIFQgKyB0KSAqIEQgKyBqCiAgICBtID0gaSA8IE4KICAgIGEgPSB0bC5sb2FkKFggKyB4cCwgbSwgb3RoZXI9MCkKICAgIHogPSB0bC5sb2FkKFggKyB4cCArIGhhbGYsIG0sIG90aGVyPTApCiAgICBjMCA9IHRsLmxvYWQoQyArIGNwLCBtLCBvdGhlcj0wKQogICAgYzEgPSB0bC5sb2FkKEMgKyBjcCArIGhhbGYsIG0sIG90aGVyPTApCiAgICBzMCA9IHRsLmxvYWQoUyArIGNwLCBtLCBvdGhlcj0wKQogICAgczEgPSB0bC5sb2FkKFMgKyBjcCArIGhhbGYsIG0sIG90aGVyPTApCiAgICAjIGVuYWJsZV9mcF9mdXNpb249RmFsc2UgYXQgbGF1bmNoOiB0d28gcm91bmRlZCBwcm9kdWN0cyB0aGVuIHJvdW5kZWQgYWRkLgogICAgeTAgPSB0bC5hZGQodGwubXVsKGEsIGMwKSwgdGwubXVsKF9zaWduZmxpcF9mMzIoeiksIHMwKSkKICAgIHkxID0gdGwuYWRkKHRsLm11bCh6LCBjMSksIHRsLm11bChhLCBzMSkpCiAgICB0bC5zdG9yZShZICsgeHAsIHkwLCBtKQogICAgdGwuc3RvcmUoWSArIHhwICsgaGFsZiwgeTEsIG0pCgoKQHRyaXRvbi5qaXQoZG9fbm90X3NwZWNpYWxpemU9WyJUIiwgIk4iLCAiR0IiLCAiR0giLCAiR1QiLCAiR0QiXSkKZGVmIHJvdGFyeV9hZGpvaW50X2YzMihHLCBDLCBTLCBEWCwgSDogdGwuY29uc3RleHByLCBULAogICAgICAgICAgICAgICAgICAgICAgIEQ6IHRsLmNvbnN0ZXhwciwgQ0I6IHRsLmNvbnN0ZXhwciwgTiwKICAgICAgICAgICAgICAgICAgICAgICBHQiwgR0gsIEdULAogICAgICAgICAgICAgICAgICAgICAgIEdELCBCTE9DSzogdGwuY29uc3RleHByKToKICAgIGkgPSB0bC5wcm9ncmFtX2lkKDApICogQkxPQ0sgKyB0bC5hcmFuZ2UoMCwgQkxPQ0spCiAgICBoYWxmID0gRCAvLyAyCiAgICBqID0gaSAlIGhhbGYKICAgIGggPSAoaSAvLyBoYWxmKSAlIEgKICAgIHQgPSBpIC8vIChoYWxmICogSCkKICAgIGIgPSB0bC5wcm9ncmFtX2lkKDEpCiAgICB4cCA9ICgoYiAqIFQgKyB0KSAqIEggKyBoKSAqIEQgKyBqCiAgICBncCA9IGIgKiBHQiArIGggKiBHSCArIHQgKiBHVCArIGogKiBHRAogICAgY3AgPSAoKGIgaWYgQ0IgIT0gMSBlbHNlIDApICogVCArIHQpICogRCArIGoKICAgIG0gPSBpIDwgTgogICAgZzAgPSB0bC5sb2FkKEcgKyBncCwgbSwgb3RoZXI9MCkKICAgIGcxID0gdGwubG9hZChHICsgZ3AgKyBoYWxmICogR0QsIG0sIG90aGVyPTApCiAgICBjMCA9IHRsLmxvYWQoQyArIGNwLCBtLCBvdGhlcj0wKQogICAgYzEgPSB0bC5sb2FkKEMgKyBjcCArIGhhbGYsIG0sIG90aGVyPTApCiAgICBzMCA9IHRsLmxvYWQoUyArIGNwLCBtLCBvdGhlcj0wKQogICAgczEgPSB0bC5sb2FkKFMgKyBjcCArIGhhbGYsIG0sIG90aGVyPTApCiAgICAjIE5hdGl2ZSBzbGljZS1iYWNrd2FyZCBwYWRzIHdpdGggKzAgdGhlbiBhY2N1bXVsYXRlcyB0d28gZGlzam9pbnQgc2xpY2VzLgogICAgIyBJdCBjYW5vbmljYWxpemVzIGEgemVyby12YWx1ZWQgcm90YXRlZCBjb250cmlidXRpb24gdG8gcG9zaXRpdmUgemVyby4KICAgIHowID0gdGwubXVsKGcxLCBzMSkKICAgIHoxID0gX3NpZ25mbGlwX2YzMih0bC5tdWwoZzAsIHMwKSkKICAgIHowID0gdGwud2hlcmUoejAgPT0gMC4wLCAwLjAsIHowKQogICAgejEgPSB0bC53aGVyZSh6MSA9PSAwLjAsIDAuMCwgejEpCiAgICBkeDAgPSB0bC5hZGQodGwubXVsKGcwLCBjMCksIHowKQogICAgZHgxID0gdGwuYWRkKHRsLm11bChnMSwgYzEpLCB6MSkKICAgIHRsLnN0b3JlKERYICsgeHAsIGR4MCwgbSkKICAgIHRsLnN0b3JlKERYICsgeHAgKyBoYWxmLCBkeDEsIG0pCgoKZGVmIF9kZW5zZV9wcm9qZWN0aW9uKHgpOgogICAgaWYgeC5uZGltICE9IDQ6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICBiLCBoLCB0LCBkID0geC5zaGFwZQogICAgcmV0dXJuIGIgPiAwIGFuZCBoID4gMCBhbmQgdCA+IDAgYW5kIGQgPiAwIGFuZCBkICUgMiA9PSAwIGFuZCB4LnN0cmlkZSgpID09ICh0KmgqZCwgZCwgaCpkLCAxKQoKCmRlZiBfc3VwcG9ydF9yZWFzb24ocSwgaywgY29zLCBzaW4sIHVuc3F1ZWV6ZV9kaW09MSwgKiwgY3VkYV9vbmx5PVRydWUsIHRhc2tfb25seT1UcnVlKToKICAgIHhzID0gKHEsIGssIGNvcywgc2luKQogICAgaWYgYW55KHR5cGUoeCkgaXMgbm90IHRvcmNoLlRlbnNvciBmb3IgeCBpbiB4cyk6CiAgICAgICAgcmV0dXJuICJ0ZW5zb3Jfc3ViY2xhc3MiCiAgICBpZiB1bnNxdWVlemVfZGltICE9IDE6CiAgICAgICAgcmV0dXJuICJheGlzIgogICAgaWYgYW55KHguZHR5cGUgIT0gdG9yY2guZmxvYXQzMiBvciB4LmRldmljZSAhPSBxLmRldmljZSBvciB4LmlzX25lZygpIG9yIHguaXNfY29uaigpIGZvciB4IGluIHhzKToKICAgICAgICByZXR1cm4gImR0eXBlX2RldmljZV9vcl9sYXp5X3ZpZXciCiAgICBpZiBjdWRhX29ubHkgYW5kIHEuZGV2aWNlLnR5cGUgIT0gImN1ZGEiOgogICAgICAgIHJldHVybiAibm90X2N1ZGEiCiAgICBpZiBhbnkoeC5sYXlvdXQgIT0gdG9yY2guc3RyaWRlZCBmb3IgeCBpbiB4cyk6CiAgICAgICAgcmV0dXJuICJsYXlvdXQiCiAgICBpZiBub3QgX2RlbnNlX3Byb2plY3Rpb24ocSkgb3Igbm90IF9kZW5zZV9wcm9qZWN0aW9uKGspOgogICAgICAgIHJldHVybiAicHJvamVjdGlvbl9zdHJpZGUiCiAgICBiLCBocSwgdCwgZCA9IHEuc2hhcGUKICAgIGlmIGsuc2hhcGVbMF0gIT0gYiBvciBrLnNoYXBlWzI6XSAhPSAodCwgZCk6CiAgICAgICAgcmV0dXJuICJxa19nZW9tZXRyeSIKICAgIGlmIHRhc2tfb25seSBhbmQgbm90IChiIDw9IDQgYW5kIHQgPD0gNTEyIGFuZCBocSA9PSAxMiBhbmQgay5zaGFwZVsxXSA9PSAyIGFuZCBkID09IDEyOCk6CiAgICAgICAgcmV0dXJuICJvdXRzaWRlX2ZpeGVkX3Rhc2tfZ2VvbWV0cnkiCiAgICBpZiBjb3MubmRpbSAhPSAzIG9yIHNpbi5zaGFwZSAhPSBjb3Muc2hhcGUgb3IgY29zLnNoYXBlWzBdIG5vdCBpbiAoMSwgYikgb3IgY29zLnNoYXBlWzE6XSAhPSAodCwgZCk6CiAgICAgICAgcmV0dXJuICJjb2VmZmljaWVudF9nZW9tZXRyeSIKICAgIGlmIGNvcy5yZXF1aXJlc19ncmFkIG9yIHNpbi5yZXF1aXJlc19ncmFkOgogICAgICAgIHJldHVybiAidHJhaW5hYmxlX2NvZWZmaWNpZW50IgogICAgaWYgbm90IGNvcy5pc19jb250aWd1b3VzKCkgb3Igbm90IHNpbi5pc19jb250aWd1b3VzKCk6CiAgICAgICAgcmV0dXJuICJjb2VmZmljaWVudF9zdHJpZGUiCiAgICBzdG9yYWdlID0gW3gudW50eXBlZF9zdG9yYWdlKCkuZGF0YV9wdHIoKSBmb3IgeCBpbiB4c10KICAgIGlmIHN0b3JhZ2VbMF0gPT0gc3RvcmFnZVsxXSBvciBhbnkocyBpbiBzdG9yYWdlWzI6XSBmb3IgcyBpbiBzdG9yYWdlWzoyXSk6CiAgICAgICAgcmV0dXJuICJzaGFyZWRfYWN0aXZhdGlvbl9zdG9yYWdlIgogICAgcmV0dXJuIE5vbmUKCgpkZWYgX3N1cHBvcnRlZCgqYXJncywgKiprd2FyZ3MpOgogICAgcmV0dXJuIF9zdXBwb3J0X3JlYXNvbigqYXJncywgKiprd2FyZ3MpIGlzIE5vbmUKCgpkZWYgbmF0aXZlX2Fkam9pbnQoZywgY29zLCBzaW4pOgogICAgIiIiRGlmZmVyZW50aWFibGUgZmFsbGJhY2sgZm9yIGhpZ2hlci1vcmRlciByZXF1ZXN0czsgZXhwbGljaXQgbmF0aXZlIHNsaWNlIGdyYXBoLiIiIgogICAgZCA9IGcuc2hhcGVbLTFdIC8vIDIKICAgIGMsIHMgPSBjb3MudW5zcXVlZXplKDEpLCBzaW4udW5zcXVlZXplKDEpCiAgICByb3RhdGVkX2dyYWQgPSBnICogcwogICAgYSwgYiA9IHJvdGF0ZWRfZ3JhZFsuLi4sIDpkXSwgcm90YXRlZF9ncmFkWy4uLiwgZDpdCiAgICAjIFR3byBzbGljZS1iYWNrd2FyZHMgcmVhbGx5IGRvIGFkZCBwYWRkZWQgdGVuc29yczsgcHJlc2VydmUgdGhlaXIgemVybyBzaWducy4KICAgIHphID0gdG9yY2guY2F0KCh0b3JjaC56ZXJvc19saWtlKGEpLCAtYSksIGRpbT0tMSkKICAgIHpiID0gdG9yY2guY2F0KChiLCB0b3JjaC56ZXJvc19saWtlKGIpKSwgZGltPS0xKQogICAgcmV0dXJuIGcgKiBjICsgKHphICsgemIpCgoKY2xhc3MgX0Zyb3plblJvUEUodG9yY2guYXV0b2dyYWQuRnVuY3Rpb24pOgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGZvcndhcmQoY3R4LCB4LCBjb3MsIHNpbik6CiAgICAgICAgY3R4LnNldF9tYXRlcmlhbGl6ZV9ncmFkcyhGYWxzZSkKICAgICAgICBjdHguc2F2ZV9mb3JfYmFja3dhcmQoY29zLCBzaW4pCiAgICAgICAgY3R4LnNoYXBlLCBjdHguc3RyaWRlID0gdHVwbGUoeC5zaGFwZSksIHguc3RyaWRlKCkKICAgICAgICB5ID0gdG9yY2guZW1wdHlfc3RyaWRlZCh4LnNoYXBlLCB4LnN0cmlkZSgpLCBkdHlwZT14LmR0eXBlLCBkZXZpY2U9eC5kZXZpY2UpCiAgICAgICAgYiwgaCwgdCwgZCA9IHguc2hhcGUKICAgICAgICBuID0geC5udW1lbCgpIC8vICgyICogYikKICAgICAgICBjb21waWxlZCA9IHJvdGFyeV9hcHBseV9mMzJbKHRyaXRvbi5jZGl2KG4sIDI1NiksIGIpXSh4LCBjb3MsIHNpbiwgeSwgaCwgdCwgZCwgY29zLnNoYXBlWzBdLCBuLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBCTE9DSz0yNTYsIG51bV93YXJwcz00LCBlbmFibGVfZnBfZnVzaW9uPUZhbHNlKQogICAgICAgIF9zYXZlX2NvZGVnZW4oY29tcGlsZWQsICJyb3RhcnlfYXBwbHlfZjMyIikKICAgICAgICBfZXZlbnQoImZvcndhcmRfbGF1bmNoX3JldHVybmVkIiwgc2hhcGU9bGlzdCh4LnNoYXBlKSwgZHR5cGU9c3RyKHguZHR5cGUpLCBzdHJpZGU9bGlzdCh4LnN0cmlkZSgpKSkKICAgICAgICByZXR1cm4geQoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBiYWNrd2FyZChjdHgsIGcpOgogICAgICAgIGlmIGcgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUsIE5vbmUKICAgICAgICBjb3MsIHNpbiA9IGN0eC5zYXZlZF90ZW5zb3JzCiAgICAgICAgaWYgdG9yY2guaXNfZ3JhZF9lbmFibGVkKCk6CiAgICAgICAgICAgIF9ldmVudCgiaGlnaGVyX29yZGVyX25hdGl2ZV9hZGpvaW50IikKICAgICAgICAgICAgcmV0dXJuIG5hdGl2ZV9hZGpvaW50KGcsIGNvcywgc2luKSwgTm9uZSwgTm9uZQogICAgICAgICMgUG9pbnRlciBsb2FkcyBkbyBub3QgaW1wbGVtZW50IFB5VG9yY2gncyBsYXp5IG5lZ2F0aXZlL2Nvbmp1Z2F0ZSBiaXRzLgogICAgICAgIGcgPSBnLnJlc29sdmVfbmVnKCkucmVzb2x2ZV9jb25qKCkKICAgICAgICBkeCA9IHRvcmNoLmVtcHR5X3N0cmlkZWQoY3R4LnNoYXBlLCBjdHguc3RyaWRlLCBkdHlwZT1nLmR0eXBlLCBkZXZpY2U9Zy5kZXZpY2UpCiAgICAgICAgYiwgaCwgdCwgZCA9IGN0eC5zaGFwZQogICAgICAgIG4gPSBoKnQqZC8vMgogICAgICAgIGNvbXBpbGVkID0gcm90YXJ5X2Fkam9pbnRfZjMyWyh0cml0b24uY2RpdihuLCAyNTYpLCBiKV0oZywgY29zLCBzaW4sIGR4LCBoLCB0LCBkLCBjb3Muc2hhcGVbMF0sIG4sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAqZy5zdHJpZGUoKSwgQkxPQ0s9MjU2LCBudW1fd2FycHM9NCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVuYWJsZV9mcF9mdXNpb249RmFsc2UpCiAgICAgICAgX3NhdmVfY29kZWdlbihjb21waWxlZCwgInJvdGFyeV9hZGpvaW50X2YzMiIpCiAgICAgICAgX2V2ZW50KCJiYWNrd2FyZF9sYXVuY2hfcmV0dXJuZWQiLCBzaGFwZT1saXN0KGN0eC5zaGFwZSksIHVwc3RyZWFtX3N0cmlkZT1saXN0KGcuc3RyaWRlKCkpKQogICAgICAgIHJldHVybiBkeCwgTm9uZSwgTm9uZQoKCmRlZiBmdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLCBrLCBjb3MsIHNpbiwgdW5zcXVlZXplX2RpbT0xKToKICAgIGludGVycHJldGluZyA9IG9zLmVudmlyb24uZ2V0KCJUUklUT05fSU5URVJQUkVUIikgPT0gIjEiCiAgICByZWFzb24gPSBfc3VwcG9ydF9yZWFzb24ocSwgaywgY29zLCBzaW4sIHVuc3F1ZWV6ZV9kaW0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3VkYV9vbmx5PW5vdCBpbnRlcnByZXRpbmcsIHRhc2tfb25seT1ub3QgaW50ZXJwcmV0aW5nKQogICAgaWYgcmVhc29uOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuc3VwcG9ydGVkIGZyb3plbiBGUDMyIHJvdGFyeSBib3VuZGFyeTogIiArIHJlYXNvbikKICAgICMgU2VwYXJhdGUgRnVuY3Rpb25zIHByZXNlcnZlIGluZGVwZW5kZW50bHkgZnJvemVuL3VudXNlZCBRL0sgYnJhbmNoZXMuCiAgICByZXR1cm4gX0Zyb3plblJvUEUuYXBwbHkocSwgY29zLCBzaW4pLCBfRnJvemVuUm9QRS5hcHBseShrLCBjb3MsIHNpbikKCgpkZWYgcGF0Y2hfcXdlbjJfcm90YXJ5KCk6CiAgICBmcm9tIHRyYW5zZm9ybWVycy5tb2RlbHMucXdlbjIgaW1wb3J0IG1vZGVsaW5nX3F3ZW4yIGFzIG1vZHVsZQogICAgb3JpZ2luYWwgPSBtb2R1bGUuYXBwbHlfcm90YXJ5X3Bvc19lbWIKICAgIGlmIGdldGF0dHIob3JpZ2luYWwsICJfb3JpZ2luYWxfcm90YXJ5X2NhbmRpZGF0ZSIsIEZhbHNlKToKICAgICAgICByZXR1cm4gVHJ1ZQogICAgcmF3ID0gb3BlbihpbnNwZWN0LmdldGZpbGUobW9kdWxlKSwgInJiIikucmVhZCgpCiAgICBibG9iID0gaGFzaGxpYi5zaGExKGIiYmxvYiAiICsgc3RyKGxlbihyYXcpKS5lbmNvZGUoKSArIGIiXDAiICsgcmF3KS5oZXhkaWdlc3QoKQogICAgaWYgYmxvYiAhPSBFWFBFQ1RFRF9RV0VOX0JMT0I6CiAgICAgICAgX2V2ZW50KCJwYXRjaF9yZWZ1c2VkX3NvdXJjZV9taXNtYXRjaCIsIG9ic2VydmVkPWJsb2IpCiAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgQGZ1bmN0b29scy53cmFwcyhvcmlnaW5hbCkKICAgIGRlZiBndWFyZGVkKHEsIGssIGNvcywgc2luLCB1bnNxdWVlemVfZGltPTEpOgogICAgICAgIHJlYXNvbiA9IF9zdXBwb3J0X3JlYXNvbihxLCBrLCBjb3MsIHNpbiwgdW5zcXVlZXplX2RpbSkKICAgICAgICBpZiByZWFzb24gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIF9ldmVudCgibmF0aXZlX2ZhbGxiYWNrXyIgKyByZWFzb24pCiAgICAgICAgICAgIHJldHVybiBvcmlnaW5hbChxLCBrLCBjb3MsIHNpbiwgdW5zcXVlZXplX2RpbT11bnNxdWVlemVfZGltKQogICAgICAgIHJldHVybiBmdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLCBrLCBjb3MsIHNpbiwgdW5zcXVlZXplX2RpbSkKCiAgICBndWFyZGVkLl9vcmlnaW5hbF9yb3RhcnlfY2FuZGlkYXRlID0gVHJ1ZQogICAgbW9kdWxlLmFwcGx5X3JvdGFyeV9wb3NfZW1iID0gZ3VhcmRlZAogICAgX2V2ZW50KCJwYXRjaF9pbnN0YWxsZWQiLCBzb3VyY2VfYmxvYj1ibG9iLCBvd25lZF9rZXJuZWxzPVsicm90YXJ5X2FwcGx5X2YzMiIsICJyb3RhcnlfYWRqb2ludF9mMzIiXSkKICAgIHJldHVybiBUcnVlCg==', 'solution/test_rotary.py': 'IiIiT3JpZ2luYWwgZml4dHVyZXM7IGludGVycHJldGVyIHJlc3VsdHMgYXJlIG5vdCBDVURBIGNvZGUtZ2VuZXJhdGlvbiBldmlkZW5jZS4iIiIKaW1wb3J0IGl0ZXJ0b29scwppbXBvcnQgb3MKaW1wb3J0IHB5dGVzdAppbXBvcnQgdG9yY2gKZnJvbSAuY3B1X2NvbnRyYWN0IGltcG9ydCBpbnB1dHMsIHJlZmVyZW5jZSwgZXhhY3QKZnJvbSAuIGltcG9ydCByb3RhcnkKCklOVEVSUFJFVCA9IG9zLmVudmlyb24uZ2V0KCJUUklUT05fSU5URVJQUkVUIikgPT0gIjEiCktFUk5FTCA9IHB5dGVzdC5tYXJrLnNraXBpZihub3QgSU5URVJQUkVULCByZWFzb249IkNQVSBpbnRlcnByZXRlciBleHBsaWNpdGx5IGRpc2FibGVkIikKCgpAcHl0ZXN0Lm1hcmsucGFyYW1ldHJpemUoImNiIiwgWzEsMl0pCkBweXRlc3QubWFyay5wYXJhbWV0cml6ZSgic2NhbGUiLCBbMS4sNjU1MzYuXSkKQEtFUk5FTApkZWYgdGVzdF9maW5pdGVfc3RhZ2VkX2FyaXRobWV0aWMoY2Isc2NhbGUpOgogICAgcSxrLGMscyA9IGlucHV0cyhjYj1jYikKICAgIHZlcnNpb25zID0gW3guX3ZlcnNpb24gZm9yIHggaW4gKHEsayxjLHMpXQogICAgc25hcHNob3RzID0gW3guZGV0YWNoKCkuY2xvbmUoKSBmb3IgeCBpbiAocSxrLGMscyldCiAgICBybmcgPSB0b3JjaC5nZXRfcm5nX3N0YXRlKCkuY2xvbmUoKQogICAgZXhwZWN0ZWQgPSByZWZlcmVuY2UocSxrLGMscykKICAgIGFjdHVhbCA9IHJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLGssYyxzKQogICAgZ3MgPSB0dXBsZSh0b3JjaC5saW5zcGFjZSgtc2NhbGUsc2NhbGUseC5udW1lbCgpKS5yZXNoYXBlKHguc2hhcGUpIGZvciB4IGluIGV4cGVjdGVkKQogICAgZWcgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKGV4cGVjdGVkLChxLGspLGdzKQogICAgYWcgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKGFjdHVhbCwocSxrKSxncykKICAgIGZvciBhLGUseCBpbiB6aXAoYWN0dWFsLGV4cGVjdGVkLChxLGspKToKICAgICAgICBleGFjdChhLGUpCiAgICAgICAgYXNzZXJ0IGEuc3RyaWRlKCkgPT0geC5zdHJpZGUoKQogICAgICAgIGFzc2VydCBhLmRhdGFfcHRyKCkgIT0geC5kYXRhX3B0cigpCiAgICBmb3IgYSxlIGluIHppcChhZyxlZyk6IGV4YWN0KGEsZSkKICAgIGZvciB4LHZlcnNpb24sc25hcCBpbiB6aXAoKHEsayxjLHMpLHZlcnNpb25zLHNuYXBzaG90cyk6CiAgICAgICAgYXNzZXJ0IHguX3ZlcnNpb24gPT0gdmVyc2lvbgogICAgICAgIGV4YWN0KHgsc25hcCkKICAgIGFzc2VydCB0b3JjaC5lcXVhbChybmcsdG9yY2guZ2V0X3JuZ19zdGF0ZSgpKQoKCkBLRVJORUwKQHB5dGVzdC5tYXJrLnBhcmFtZXRyaXplKCJsYXlvdXQiLCBbImNvbnRpZ3VvdXMiLCJleHBhbmRlZCIsInNsaWNlZCIsIm5lZ2F0aXZlX3ZpZXciXSkKZGVmIHRlc3RfdXBzdHJlYW1fbGF5b3V0KGxheW91dCk6CiAgICBxLGssYyxzID0gaW5wdXRzKHQ9MykKICAgIGRlZiBncmFkaWVudCh4KToKICAgICAgICBpZiBsYXlvdXQgPT0gImV4cGFuZGVkIjogcmV0dXJuIHRvcmNoLm9uZXMoKDEsMSwxLDEpKS5leHBhbmRfYXMoeCkKICAgICAgICBpZiBsYXlvdXQgPT0gImNvbnRpZ3VvdXMiOiByZXR1cm4gdG9yY2gub25lcyh4LnNoYXBlKQogICAgICAgIGcgPSB0b3JjaC5vbmVzKCgqeC5zaGFwZVs6LTFdLCB4LnNoYXBlWy0xXSoyKSlbLi4uLDo6Ml0KICAgICAgICByZXR1cm4gdG9yY2guX25lZ192aWV3KGcpIGlmIGxheW91dCA9PSAibmVnYXRpdmVfdmlldyIgZWxzZSBnCiAgICBncyA9IHR1cGxlKGdyYWRpZW50KHgpIGZvciB4IGluIChxLGspKQogICAgZSA9IHRvcmNoLmF1dG9ncmFkLmdyYWQocmVmZXJlbmNlKHEsayxjLHMpLChxLGspLGdzKQogICAgYSA9IHRvcmNoLmF1dG9ncmFkLmdyYWQocm90YXJ5LmZ1c2VkX2FwcGx5X3JvdGFyeV9wb3NfZW1iKHEsayxjLHMpLChxLGspLGdzKQogICAgZm9yIGFhLGVlIGluIHppcChhLGUpOiBleGFjdChhYSxlZSkKCgpAS0VSTkVMCkBweXRlc3QubWFyay5wYXJhbWV0cml6ZSgicnEscmssdXNlZCIsIFsoVHJ1ZSxUcnVlLDApLChUcnVlLFRydWUsMSksKEZhbHNlLFRydWUsMSksKFRydWUsRmFsc2UsMCldKQpkZWYgdGVzdF9pbmRlcGVuZGVudF91bnVzZWRfb3V0cHV0cyhycSxyayx1c2VkKToKICAgIHEsayxjLHMgPSBpbnB1dHMoYj0xLHQ9MSkKICAgIHEucmVxdWlyZXNfZ3JhZF8ocnEpO2sucmVxdWlyZXNfZ3JhZF8ocmspCiAgICB2YXJpYWJsZXMgPSB0dXBsZSh4IGZvciB4IGluIChxLGspIGlmIHgucmVxdWlyZXNfZ3JhZCkKICAgIGV4cGVjdGVkLGFjdHVhbCA9IHJlZmVyZW5jZShxLGssYyxzKSxyb3RhcnkuZnVzZWRfYXBwbHlfcm90YXJ5X3Bvc19lbWIocSxrLGMscykKICAgIGFzc2VydCBbeC5yZXF1aXJlc19ncmFkIGZvciB4IGluIGFjdHVhbF0gPT0gW3JxLHJrXQogICAgZSA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoZXhwZWN0ZWRbdXNlZF0uc3VtKCksIHZhcmlhYmxlcywgYWxsb3dfdW51c2VkPVRydWUpCiAgICBhID0gdG9yY2guYXV0b2dyYWQuZ3JhZChhY3R1YWxbdXNlZF0uc3VtKCksIHZhcmlhYmxlcywgYWxsb3dfdW51c2VkPVRydWUpCiAgICBmb3IgYWEsZWUgaW4gemlwKGEsZSk6CiAgICAgICAgaWYgZWUgaXMgTm9uZTogYXNzZXJ0IGFhIGlzIE5vbmUKICAgICAgICBlbHNlOiBleGFjdChhYSxlZSkKCgpAS0VSTkVMCmRlZiB0ZXN0X2V4aGF1c3RpdmVfc2lnbmVkX3plcm9fbmF0aXZlX3NsaWNlX2Fkam9pbnQoKToKICAgIHZhbHVlcyA9IHRvcmNoLnRlbnNvcihsaXN0KGl0ZXJ0b29scy5wcm9kdWN0KFswLiwtMC4sMS4sLTEuXSxyZXBlYXQ9NikpKQogICAgbiA9IGxlbih2YWx1ZXMpCiAgICBxID0gdG9yY2guemVyb3MoKDEsbiwxLDIpKS50cmFuc3Bvc2UoMSwyKS5yZXF1aXJlc19ncmFkXygpCiAgICBrID0gcS5kZXRhY2goKS5jbG9uZSgpLnJlcXVpcmVzX2dyYWRfKCkKICAgIGMscyA9IHZhbHVlc1s6LDoyXS5yZXNoYXBlKDEsbiwyKS5jb250aWd1b3VzKCksIHZhbHVlc1s6LDI6NF0ucmVzaGFwZSgxLG4sMikuY29udGlndW91cygpCiAgICBnID0gdmFsdWVzWzosNDpdLnJlc2hhcGVfYXMocSkKICAgIGUsYSA9IHJlZmVyZW5jZShxLGssYyxzKSxyb3RhcnkuZnVzZWRfYXBwbHlfcm90YXJ5X3Bvc19lbWIocSxrLGMscykKICAgIGZvciBhYSxlZSBpbiB6aXAoYSxlKTogZXhhY3QoYWEsZWUpCiAgICBlZyA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoZSwocSxrKSwoZyxnKSkKICAgIGFnID0gdG9yY2guYXV0b2dyYWQuZ3JhZChhLChxLGspLChnLGcpKQogICAgZm9yIGFhLGVlIGluIHppcChhZyxlZyk6IGV4YWN0KGFhLGVlKQogICAgZXhhY3Qocm90YXJ5Lm5hdGl2ZV9hZGpvaW50KGcsYyxzKSxlZ1swXSkKCgpAS0VSTkVMCmRlZiB0ZXN0X25vbmZpbml0ZV9jbGFzc2VzX2FuZF9zdWJub3JtYWxzKCk6CiAgICB2YWxzID0gdG9yY2gudGVuc29yKFswLiwtMC4sZmxvYXQoJ2luZicpLGZsb2F0KCctaW5mJyksZmxvYXQoJ25hbicpLDFlLTQwLC0xZS00MCwxLl0pCiAgICBxID0gdmFscy5yZXBlYXQoOCkucmVzaGFwZSgxLDgsMSw4KS50cmFuc3Bvc2UoMSwyKS5yZXF1aXJlc19ncmFkXygpCiAgICBrID0gcS5kZXRhY2goKS5jbG9uZSgpLnJlcXVpcmVzX2dyYWRfKCkKICAgIGMgPSB2YWxzLnJlcGVhdCg4KS5yZXNoYXBlKDEsOCw4KS5jb250aWd1b3VzKCkKICAgIHMgPSB0b3JjaC5mbGlwKGMsWy0xXSkuY29udGlndW91cygpCiAgICBlLGEgPSByZWZlcmVuY2UocSxrLGMscykscm90YXJ5LmZ1c2VkX2FwcGx5X3JvdGFyeV9wb3NfZW1iKHEsayxjLHMpCiAgICBncyA9IChxLmRldGFjaCgpLGsuZGV0YWNoKCkpCiAgICBlZyxhZyA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoZSwocSxrKSxncyksdG9yY2guYXV0b2dyYWQuZ3JhZChhLChxLGspLGdzKQogICAgZm9yIGFhLGVlIGluIHppcCgoKmEsKmFnKSwoKmUsKmVnKSk6IGV4YWN0KGFhLGVlKQoKCkBLRVJORUwKZGVmIHRlc3RfY29lZmZpY2llbnRfbXV0YXRpb25fcmVqZWN0ZWRfYnlfYXV0b2dyYWQoKToKICAgIHEsayxjLHMgPSBpbnB1dHModD0xKQogICAgYSA9IHJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLGssYyxzKQogICAgYy5hZGRfKDEpCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoUnVudGltZUVycm9yLG1hdGNoPSJtb2RpZmllZCBieSBhbiBpbnBsYWNlIG9wZXJhdGlvbiIpOgogICAgICAgIGFbMF0uc3VtKCkuYmFja3dhcmQoKQoKCkBLRVJORUwKZGVmIHRlc3RfaGlnaGVyX29yZGVyX25hdGl2ZV9mYWxsYmFjaygpOgogICAgcSxrLGMscyA9IGlucHV0cyh0PTEpCiAgICBnID0gdG9yY2gub25lc19saWtlKHEscmVxdWlyZXNfZ3JhZD1UcnVlKQogICAgYSA9IHJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLGssYyxzKVswXQogICAgZSA9IHJlZmVyZW5jZShxLGssYyxzKVswXQogICAgYWcsID0gdG9yY2guYXV0b2dyYWQuZ3JhZChhLHEsZyxjcmVhdGVfZ3JhcGg9VHJ1ZSkKICAgIGVnLCA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoZSxxLGcsY3JlYXRlX2dyYXBoPVRydWUpCiAgICBleGFjdChhZyxlZykKICAgIGFhLCA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoYWcsZyx0b3JjaC5vbmVzX2xpa2UoYWcpKQogICAgZWUsID0gdG9yY2guYXV0b2dyYWQuZ3JhZChlZyxnLHRvcmNoLm9uZXNfbGlrZShlZykpCiAgICBleGFjdChhYSxlZSkKCgpkZWYgdGVzdF9tZXRhZGF0YV9yZWplY3Rpb25zX2FuZF9zbW9rZSgpOgogICAgZnJvbSAuY3B1X2NvbnRyYWN0IGltcG9ydCBydW4KICAgIGFzc2VydCBydW4oKVsicGFzc2VkIl0KICAgIHEsayxjLHMgPSBpbnB1dHMoKQogICAgcmVhc29uID0gbGFtYmRhICphLCoqa3c6IHJvdGFyeS5fc3VwcG9ydF9yZWFzb24oKmEsY3VkYV9vbmx5PUZhbHNlLCoqa3cpCiAgICBhc3NlcnQgcmVhc29uKHEscSxjLHMpIGlzIG5vdCBOb25lCiAgICBhc3NlcnQgcmVhc29uKHRvcmNoLl9uZWdfdmlldyhxKSxrLGMscykgaXMgbm90IE5vbmUKICAgIGFzc2VydCByZWFzb24ocSxrLGMscyx1bnNxdWVlemVfZGltPTIpIGlzIG5vdCBOb25lCiAgICBhc3NlcnQgcmVhc29uKHEuaGFsZigpLGssYyxzKSBpcyBub3QgTm9uZQogICAgYXNzZXJ0IHJlYXNvbihxLGssYy5leHBhbmQoMiwtMSwtMSkscy5leHBhbmQoMiwtMSwtMSkpIGlzIG5vdCBOb25lCiAgICBhc3NlcnQgcmVhc29uKHEsayxjWzosOi0xXSxzWzosOi0xXSkgaXMgbm90IE5vbmUKCgpkZWYgdGVzdF9wYXRjaF9pZGVtcG90ZW50X2NwdV9uYXRpdmVfYW5kX3JuZyhtb25rZXlwYXRjaCk6CiAgICBmcm9tIHRyYW5zZm9ybWVycy5tb2RlbHMucXdlbjIgaW1wb3J0IG1vZGVsaW5nX3F3ZW4yIGFzIG1vZHVsZQogICAgb3JpZ2luYWwgPSBtb2R1bGUuYXBwbHlfcm90YXJ5X3Bvc19lbWIKICAgIG1vbmtleXBhdGNoLnNldGF0dHIobW9kdWxlLCJhcHBseV9yb3RhcnlfcG9zX2VtYiIsb3JpZ2luYWwpCiAgICBxLGssYyxzID0gaW5wdXRzKCkKICAgIGUgPSBvcmlnaW5hbChxLGssYyxzKQogICAgcm5nID0gdG9yY2guZ2V0X3JuZ19zdGF0ZSgpLmNsb25lKCkKICAgIGFzc2VydCByb3RhcnkucGF0Y2hfcXdlbjJfcm90YXJ5KCkKICAgIHBhdGNoZWQgPSBtb2R1bGUuYXBwbHlfcm90YXJ5X3Bvc19lbWIKICAgIGFzc2VydCByb3RhcnkucGF0Y2hfcXdlbjJfcm90YXJ5KCkKICAgIGFzc2VydCBtb2R1bGUuYXBwbHlfcm90YXJ5X3Bvc19lbWIgaXMgcGF0Y2hlZAogICAgYSA9IHBhdGNoZWQocSxrLGMscykKICAgIGZvciBhYSxlZSBpbiB6aXAoYSxlKTogZXhhY3QoYWEsZWUpCiAgICBhc3NlcnQgdG9yY2guZXF1YWwocm5nLHRvcmNoLmdldF9ybmdfc3RhdGUoKSkKCgpkZWYgdGVzdF9mbWFfYW5kX2Vhcmx5X2Nhc3RfbmVnYXRpdmVfY29udHJvbHMoKToKICAgIGEsYixjLHMgPSBbdG9yY2gudGVuc29yKHgsZHR5cGU9dG9yY2guZmxvYXQzMikgZm9yIHggaW4KICAgICAgICAgICAgICAgKC0zLjA5ODU4MzQ1OTg1NDEyNiwtMi4yOTkyNjcyOTIwMjI3MDUsMC44MjUzMzU2MjE4MzM4MDEzLDAuNTY0NjQyNDg4OTU2NDUxNCldCiAgICBlYWdlciA9IGEqYyArICgtYikqcwogICAgY29udHJhY3RlZCA9IChhLmRvdWJsZSgpKmMuZG91YmxlKCkrKC1iKnMpLmRvdWJsZSgpKS5mbG9hdCgpCiAgICBhc3NlcnQgZWFnZXIudmlldyh0b3JjaC5pbnQzMikgIT0gY29udHJhY3RlZC52aWV3KHRvcmNoLmludDMyKQogICAgYyA9IHMgPSB0b3JjaC5mdWxsKCgxLDEsMiksMioqLTAuNSkKICAgIGcgPSB0b3JjaC56ZXJvcygoMSw2LDEsMiksZHR5cGU9dG9yY2guZmxvYXQxNikKICAgIGdbOiw6Miw6LDBdPTQwMDAwO2dbOiw6Miw6LDFdPS00MDAwMAogICAgZ3JhZGllbnRzPVtdCiAgICBmb3IgZWFybHkgaW4gKEZhbHNlLFRydWUpOgogICAgICAgIGsgPSB0b3JjaC56ZXJvcygoMSwxLDEsMikscmVxdWlyZXNfZ3JhZD1UcnVlKQogICAgICAgIHkgPSByZWZlcmVuY2UoayxrLGMscylbMV0KICAgICAgICB5ID0geS5oYWxmKCkucmVwZWF0X2ludGVybGVhdmUoNiwxKSBpZiBlYXJseSBlbHNlIHkucmVwZWF0X2ludGVybGVhdmUoNiwxKS5oYWxmKCkKICAgICAgICBncmFkaWVudHMuYXBwZW5kKHRvcmNoLmF1dG9ncmFkLmdyYWQoeSxrLGcpWzBdKQogICAgYXNzZXJ0IHRvcmNoLmlzZmluaXRlKGdyYWRpZW50c1swXSkuYWxsKCkKICAgIGFzc2VydCB0b3JjaC5pc25hbihncmFkaWVudHNbMV0pLmFueSgpCgoKQEtFUk5FTApkZWYgdGVzdF9yZWFsX3F3ZW4yX2F0dGVudGlvbl9pbnRlcnByZXRlcl9yb3V0ZShtb25rZXlwYXRjaCk6CiAgICBmcm9tIHRyYW5zZm9ybWVycy5tb2RlbHMucXdlbjIgaW1wb3J0IG1vZGVsaW5nX3F3ZW4yIGFzIG1vZAogICAgZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxzLnF3ZW4yLmNvbmZpZ3VyYXRpb25fcXdlbjIgaW1wb3J0IFF3ZW4yQ29uZmlnCiAgICB0b3JjaC5tYW51YWxfc2VlZCg0NzEzMikKICAgIGNmZz1Rd2VuMkNvbmZpZyhoaWRkZW5fc2l6ZT0xNTM2LG51bV9hdHRlbnRpb25faGVhZHM9MTIsbnVtX2tleV92YWx1ZV9oZWFkcz0yLAogICAgICAgICAgICAgICAgICAgaW50ZXJtZWRpYXRlX3NpemU9MTYsbnVtX2hpZGRlbl9sYXllcnM9MSxhdHRlbnRpb25fZHJvcG91dD0wLiwKICAgICAgICAgICAgICAgICAgIHJvcGVfcGFyYW1ldGVycz17InJvcGVfdHlwZSI6ImRlZmF1bHQiLCJyb3BlX3RoZXRhIjoxXzAwMF8wMDAufSkKICAgIGNmZy5fYXR0bl9pbXBsZW1lbnRhdGlvbj0ic2RwYSIKICAgIG1vZHVsZT1tb2QuUXdlbjJBdHRlbnRpb24oY2ZnLDApLnRyYWluKCkKICAgIGVtYj1tb2QuUXdlbjJSb3RhcnlFbWJlZGRpbmcoY2ZnKQogICAgaGlkZGVuPXRvcmNoLnJhbmRuKCgxLDMsMTUzNikscmVxdWlyZXNfZ3JhZD1UcnVlKQogICAgY29lZmY9ZW1iKGhpZGRlbix0b3JjaC5hcmFuZ2UoMykudW5zcXVlZXplKDApKQogICAgdmFyaWFibGVzPShoaWRkZW4sKm1vZHVsZS5wYXJhbWV0ZXJzKCkpCiAgICBncmFkPXRvcmNoLm9uZXNfbGlrZShoaWRkZW4pKjY0CiAgICBlPW1vZHVsZShoaWRkZW4sY29lZmYsTm9uZSlbMF07ZWc9dG9yY2guYXV0b2dyYWQuZ3JhZChlLHZhcmlhYmxlcyxncmFkKQogICAgb3JpZ2luYWw9bW9kLmFwcGx5X3JvdGFyeV9wb3NfZW1iCiAgICBtb25rZXlwYXRjaC5zZXRhdHRyKG1vZCwiYXBwbHlfcm90YXJ5X3Bvc19lbWIiLG9yaWdpbmFsKQogICAgIyBUZXN0LW9ubHkgcmVwbGFjZSBDVURBIHJlcXVpcmVtZW50OyByZXRhaW4gcHJvZHVjdGlvbiBnZW9tZXRyeS9hbGlhcyBndWFyZHMuCiAgICByZWFzb249cm90YXJ5Ll9zdXBwb3J0X3JlYXNvbgogICAgZGVmIGNwdV9yZWFzb24oKmFyZ3MsKiprd2FyZ3MpOgogICAgICAgIGt3YXJnc1siY3VkYV9vbmx5Il09RmFsc2UKICAgICAgICByZXR1cm4gcmVhc29uKCphcmdzLCoqa3dhcmdzKQogICAgbW9ua2V5cGF0Y2guc2V0YXR0cihyb3RhcnksIl9zdXBwb3J0X3JlYXNvbiIsY3B1X3JlYXNvbikKICAgIGFzc2VydCByb3RhcnkucGF0Y2hfcXdlbjJfcm90YXJ5KCkKICAgIGJlZm9yZT1yb3RhcnkuX0NPVU5UU1siZm9yd2FyZF9sYXVuY2hfcmV0dXJuZWQiXQogICAgYT1tb2R1bGUoaGlkZGVuLGNvZWZmLE5vbmUpWzBdO2FnPXRvcmNoLmF1dG9ncmFkLmdyYWQoYSx2YXJpYWJsZXMsZ3JhZCkKICAgIGFzc2VydCByb3RhcnkuX0NPVU5UU1siZm9yd2FyZF9sYXVuY2hfcmV0dXJuZWQiXT09YmVmb3JlKzIKICAgIGV4YWN0KGEsZSkKICAgIGZvciBhYSxlZSBpbiB6aXAoYWcsZWcpOnRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGFhLGVlLHJ0b2w9MmUtNSxhdG9sPTJlLTUsZXF1YWxfbmFuPVRydWUpCg==', 'solution/web_preflight.py': 'IiIiV2ViIEpTT04tZmlsZSBhZGFwdGVyIGZvciB0aGUgdW5jaGFuZ2VkIG9yaWdpbmFsIGRpcmVjdG9yeS1vdXRwdXQgUm9QRSBnYXRlLgoKTm8ga2VybmVsIG9yIGFyaXRobWV0aWMgY2hhbmdlLiBUaGUgb3JpZ2luYWwgdGFyZ2V0IGNoZWNrcyBydW4gc3luY2hyb25vdXNseSBpbgp0aGlzIHNob3J0LWxpdmVkIGNoaWxkOyBhIGZhaWx1cmUsIGluY29tcGxldGUgcmVzdWx0LCBvciB3cm9uZyBzdGF0dXMgZmFpbHMgc2h1dC4KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHN5cwppbXBvcnQgdHJhY2ViYWNrCmltcG9ydCB1dWlkCgoKZGVmIF93cml0ZShwYXRoLCByZXBvcnQpOgogICAgdG1wID0gcGF0aC53aXRoX25hbWUocGF0aC5uYW1lICsgJy4nICsgdXVpZC51dWlkNCgpLmhleCArICcudG1wJykKICAgIHRtcC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LCBpbmRlbnQ9MikgKyAnXG4nKQogICAgdG1wLnJlcGxhY2UocGF0aCkKCgpkZWYgcnVuKGRlc3RpbmF0aW9uKToKICAgIGRlc3RpbmF0aW9uID0gUGF0aChkZXN0aW5hdGlvbikucmVzb2x2ZSgpCiAgICBpZiBkZXN0aW5hdGlvbi5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoZGVzdGluYXRpb24pCiAgICBkZXN0aW5hdGlvbi5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZGV0YWlscyA9IGRlc3RpbmF0aW9uLndpdGhfbmFtZShkZXN0aW5hdGlvbi5uYW1lICsgJy5yb3RhcnlfZGV0YWlscycpCiAgICByZXN1bHQgPSB7J3N0YXR1cyc6ICdGQUlMRUQnLCAnZ3B1X2V4ZWN1dGVkJzogRmFsc2UsICdzY29yZSc6IE5vbmUsCiAgICAgICAgICAgICAgJ2FkbWlzc2lvbic6ICdVTktOT1dOJywgJ29yaWdpbmFsX3ByZWZsaWdodF9kZXRhaWxzJzogc3RyKGRldGFpbHMpfQogICAgX3dyaXRlKGRlc3RpbmF0aW9uLCByZXN1bHQpCiAgICBwcmV2aW91c19hcmd2ID0gc3lzLmFyZ3YKICAgIGV4aXRfY29kZSA9IE5vbmUKICAgIHRyeToKICAgICAgICBmcm9tIC4gaW1wb3J0IGdwdV9wcmVmbGlnaHQKICAgICAgICBzeXMuYXJndiA9IFtncHVfcHJlZmxpZ2h0Ll9fZmlsZV9fLCAnLS1vdXQnLCBzdHIoZGV0YWlscyldCiAgICAgICAgdHJ5OgogICAgICAgICAgICBncHVfcHJlZmxpZ2h0Lm1haW4oKQogICAgICAgICAgICBleGl0X2NvZGUgPSAwCiAgICAgICAgZXhjZXB0IFN5c3RlbUV4aXQgYXMgZXhjOgogICAgICAgICAgICBleGl0X2NvZGUgPSAwIGlmIGV4Yy5jb2RlIGlzIE5vbmUgZWxzZSBleGMuY29kZQogICAgICAgIHJlcG9ydCA9IGpzb24ubG9hZHMoKGRldGFpbHMgLyAncHJlZmxpZ2h0Lmpzb24nKS5yZWFkX3RleHQoKSkKICAgICAgICBpZiBleGl0X2NvZGUgIT0gMCBvciByZXBvcnQuZ2V0KCdzdGF0dXMnKSAhPSAnUEFTU0VEX1BSRUZMSUdIVF9PTkxZJyBvciByZXBvcnQuZ2V0KCdncHVfZXhlY3V0ZWQnKSBpcyBub3QgVHJ1ZToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdPcmlnaW5hbCB0YXJnZXQgcHJlZmxpZ2h0IGZhaWxlZCBvciBkaWQgbm90IGNvbXBsZXRlOiAnICsgcmVwcihleGl0X2NvZGUpKQogICAgICAgIHJlc3VsdCA9IGRpY3QocmVwb3J0LCBvcmlnaW5hbF9wcmVmbGlnaHRfZGV0YWlscz1zdHIoZGV0YWlscyksIG9yaWdpbmFsX2V4aXRfY29kZT1leGl0X2NvZGUsCiAgICAgICAgICAgICAgICAgICAgICBzY29yZT1Ob25lLCBhZG1pc3Npb249J1VOS05PV04nKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHJlc3VsdC51cGRhdGUoc3RhdHVzPSdGQUlMRUQnLCBlcnJvcj1yZXByKGV4YyksIHRyYWNlYmFjaz10cmFjZWJhY2suZm9ybWF0X2V4YygpLAogICAgICAgICAgICAgICAgICAgICAgb3JpZ2luYWxfZXhpdF9jb2RlPWV4aXRfY29kZSkKICAgIGZpbmFsbHk6CiAgICAgICAgc3lzLmFyZ3YgPSBwcmV2aW91c19hcmd2CiAgICAgICAgX3dyaXRlKGRlc3RpbmF0aW9uLCByZXN1bHQpCiAgICByZXR1cm4gMCBpZiByZXN1bHRbJ3N0YXR1cyddID09ICdQQVNTRURfUFJFRkxJR0hUX09OTFknIGVsc2UgMQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLW91dCcsIHJlcXVpcmVkPVRydWUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcmFpc2UgU3lzdGVtRXhpdChydW4oYXJncy5vdXQpKQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtYWluKCkK', 'old_solution/__init__.py': 'IiIiT25seSBvcmlnaW5hbCBSb1BFIGFwcGxpY2F0aW9uIGtlcm5lbHM7IG9mZmljaWFsIHN0YXJ0ZXIga2VybmVscyBhcmUgbm90IGNsYWltZWQuIiIiCmRlZiBhcHBseSgpOgogICAgZnJvbSAucm90YXJ5IGltcG9ydCBwYXRjaF9xd2VuMl9yb3RhcnkKICAgIGlmIG5vdCBwYXRjaF9xd2VuMl9yb3RhcnkoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlBpbm5lZCBRd2VuMiByb3Rhcnkgc291cmNlIGd1YXJkIHJlZnVzZWQgaW5zdGFsbGF0aW9uIikKCgpkZWYgcXVldWVfc21va2UoKToKICAgIGZyb20gLmNwdV9jb250cmFjdCBpbXBvcnQgcnVuCiAgICByZXR1cm4gcnVuKCkK', 'old_solution/cpu_contract.py': 'IiIiU21hbGwgQ1BVLW9ubHkgcXVldWUgc21va2U6IG5vIENVREEgcXVlcmllcywga2VybmVscywgbW9kZWwgbG9hZGluZyBvciBkb3dubG9hZHMuIiIiCmltcG9ydCBpdGVydG9vbHMKaW1wb3J0IHRvcmNoCgoKZGVmIHJlZmVyZW5jZShxLCBrLCBjLCBzKToKICAgIGZyb20gdHJhbnNmb3JtZXJzLm1vZGVscy5xd2VuMi5tb2RlbGluZ19xd2VuMiBpbXBvcnQgYXBwbHlfcm90YXJ5X3Bvc19lbWIKICAgIHdoaWxlIGdldGF0dHIoYXBwbHlfcm90YXJ5X3Bvc19lbWIsICJfb3JpZ2luYWxfcm90YXJ5X2NhbmRpZGF0ZSIsIEZhbHNlKToKICAgICAgICBhcHBseV9yb3RhcnlfcG9zX2VtYiA9IGFwcGx5X3JvdGFyeV9wb3NfZW1iLl9fd3JhcHBlZF9fCiAgICByZXR1cm4gYXBwbHlfcm90YXJ5X3Bvc19lbWIocSwgaywgYywgcykKCgpkZWYgZXhhY3QoYSwgYik6CiAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhLCBiLCBhdG9sPTAsIHJ0b2w9MCwgZXF1YWxfbmFuPVRydWUpCiAgICBtYXNrID0gYiA9PSAwCiAgICBhc3NlcnQgdG9yY2guZXF1YWwodG9yY2guc2lnbmJpdChhW21hc2tdKSwgdG9yY2guc2lnbmJpdChiW21hc2tdKSkKCgpkZWYgaW5wdXRzKGI9MiwgdD03LCBocT0xMiwgaGs9MiwgZD0xMjgsIGNiPTEsIGRldmljZT0iY3B1Iik6CiAgICBnZW5lcmF0b3IgPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPWRldmljZSkubWFudWFsX3NlZWQoNDcxMzEpCiAgICBkZWYgcmFuZChzaGFwZSk6CiAgICAgICAgcmV0dXJuIHRvcmNoLnJhbmRuKHNoYXBlLCBnZW5lcmF0b3I9Z2VuZXJhdG9yLCBkZXZpY2U9ZGV2aWNlKQogICAgcSA9IHJhbmQoKGIsIHQsIGhxLCBkKSkudHJhbnNwb3NlKDEsIDIpLnJlcXVpcmVzX2dyYWRfKCkKICAgIGsgPSByYW5kKChiLCB0LCBoaywgZCkpLnRyYW5zcG9zZSgxLCAyKS5yZXF1aXJlc19ncmFkXygpCiAgICByZXR1cm4gcSwgaywgcmFuZCgoY2IsdCxkKSksIHJhbmQoKGNiLHQsZCkpCgoKZGVmIHJ1bigpOgogICAgZnJvbSAucm90YXJ5IGltcG9ydCBfc3VwcG9ydGVkLCBuYXRpdmVfYWRqb2ludAogICAgYmVmb3JlID0gdG9yY2guZ2V0X3JuZ19zdGF0ZSgpLmNsb25lKCkKICAgIHEsayxjLHMgPSBpbnB1dHMoKQogICAgeSA9IHJlZmVyZW5jZShxLGssYyxzKQogICAgZ3MgPSB0dXBsZSh0b3JjaC5vbmVzX2xpa2UoeCkgKiA2NTUzNiBmb3IgeCBpbiB5KQogICAgZXhwZWN0ZWQgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKHksIChxLGspLCBncykKICAgIGZvciBnLGUgaW4gemlwKGdzLGV4cGVjdGVkKToKICAgICAgICBleGFjdChuYXRpdmVfYWRqb2ludChnLGMscyksIGUpCiAgICBhc3NlcnQgX3N1cHBvcnRlZChxLGssYyxzLGN1ZGFfb25seT1GYWxzZSkKICAgIGFzc2VydCBub3QgX3N1cHBvcnRlZChxLGssYyxzKQogICAgYXNzZXJ0IG5vdCBfc3VwcG9ydGVkKHEscSxjLHMsY3VkYV9vbmx5PUZhbHNlKQogICAgYXNzZXJ0IG5vdCBfc3VwcG9ydGVkKHEuY29udGlndW91cygpLGssYyxzLGN1ZGFfb25seT1GYWxzZSkKICAgIGMucmVxdWlyZXNfZ3JhZF8oKQogICAgYXNzZXJ0IG5vdCBfc3VwcG9ydGVkKHEsayxjLHMsY3VkYV9vbmx5PUZhbHNlKQogICAgYXNzZXJ0IHRvcmNoLmVxdWFsKGJlZm9yZSwgdG9yY2guZ2V0X3JuZ19zdGF0ZSgpKQogICAgcmV0dXJuIGRpY3QocGFzc2VkPVRydWUsIHNjb3BlPSJDUFVfTUVUQURBVEFfQU5EX05BVElWRV9BREpPSU5UX09OTFkiLCBncHVfZXhlY3V0ZWQ9RmFsc2UsCiAgICAgICAgICAgICAgICB0cml0b25fZXhlY3V0ZWQ9RmFsc2UsIHNjb3JlPU5vbmUsIGFkbWlzc2lvbj0iVU5LTk9XTiIsIHRvcmNoPXRvcmNoLl9fdmVyc2lvbl9fKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBpbXBvcnQganNvbgogICAgcHJpbnQoanNvbi5kdW1wcyhydW4oKSwgaW5kZW50PTIpKQo=', 'old_solution/gpu_preflight.py': 'IiIiRXhwbGljaXQgZnV0dXJlIFQ0IGdhdGUsIG5vdCBydW4gZHVyaW5nIHByZXBhcmF0aW9uLiBTbWFsbCBmaXh0dXJlIG1vZGVscyBvbmx5LgoKcHl0aG9uIC1tIHNvbHV0aW9uLmdwdV9wcmVmbGlnaHQgLS1vdXQgL25ldy9wcmVmbGlnaHQvcGF0aApBbGwgc291cmNlL3ZlcnNpb25zLCBudW1lcmljYWwgYm91bmRhcnkgYW5kIFBUWCBjaGVja3MgZmFpbCBjbG9zZWQuIFRoZXJlIGlzIG5vCnNjb3JlL2FkbWlzc2lvbiBoZXJlLiBLZXJuZWwgZXhlY3V0aW9uIGlzIGEgcHJlZmxpZ2h0IHNlcGFyYXRlIGZyb20gY29sZCB0cmFpbmluZy4KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCByZQppbXBvcnQgdHJhY2ViYWNrCgoKZGVmIG1haW4oKToKICAgIHA9YXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKTtwLmFkZF9hcmd1bWVudCgiLS1vdXQiLHJlcXVpcmVkPVRydWUpO2E9cC5wYXJzZV9hcmdzKCkKICAgIG91dD1QYXRoKGEub3V0KS5yZXNvbHZlKCk7b3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1GYWxzZSkKICAgIHJlc3VsdD1kaWN0KHN0YXR1cz0iRkFJTEVEIixncHVfZXhlY3V0ZWQ9RmFsc2Usc2NvcmU9Tm9uZSxhZG1pc3Npb249IlVOS05PV04iLGNoZWNrcz1bXSkKICAgIHRyeToKICAgICAgICBpZiBvcy5lbnZpcm9uLmdldCgiVFJJVE9OX0lOVEVSUFJFVCIpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkdQVSBwcmVmbGlnaHQgcmVmdXNlcyBpbnRlcnByZXRlciBtb2RlIikKICAgICAgICBpbXBvcnQgdG9yY2gsdHJpdG9uLHRyYW5zZm9ybWVycwogICAgICAgIGZyb20gLiBpbXBvcnQgcm90YXJ5CiAgICAgICAgZnJvbSAuY3B1X2NvbnRyYWN0IGltcG9ydCBpbnB1dHMscmVmZXJlbmNlLGV4YWN0CiAgICAgICAgaWYgdHJpdG9uLl9fdmVyc2lvbl9fICE9ICIzLjYuMCIgb3IgdG9yY2guX192ZXJzaW9uX18gIT0gIjIuMTEuMCtjdTEyOCIgb3IgdHJhbnNmb3JtZXJzLl9fdmVyc2lvbl9fICE9ICI1LjE2LjEiOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkV4YWN0IHRhcmdldCBUb3JjaC9Ucml0b24vVHJhbnNmb3JtZXJzIHZlcnNpb25zIHJlcXVpcmVkIikKICAgICAgICBpZiB0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpIT0xIG9yIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApIT0iVGVzbGEgVDQiOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk9uZSB2aXNpYmxlIFRlc2xhIFQ0IHJlcXVpcmVkIikKICAgICAgICBvcy5lbnZpcm9uWyJTT1VQX1JPVEFSWV9DT0RFR0VOX0RJUiJdPXN0cihvdXQvInB0eCIpCiAgICAgICAgIyBPcmlnaW5hbCBjYWxsYWJsZSBzdGF5cyBpbnRhY3QgdW50aWwgYWxsIGFyaXRobWV0aWMgdGVzdHMgaGF2ZSBydW4uCiAgICAgICAgZm9yIGIsdCxjYiBpbiAoKDEsMSwxKSwoMiwxNywyKSwoNCw1MTEsMSksKDQsNTEyLDQpKToKICAgICAgICAgICAgcSxrLGMscz1pbnB1dHMoYj1iLHQ9dCxjYj1jYixkZXZpY2U9ImN1ZGEiKQogICAgICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KCJjdWRhIixkdHlwZT10b3JjaC5mbG9hdDE2KToKICAgICAgICAgICAgICAgIGV4cGVjdGVkPXJlZmVyZW5jZShxLGssYyxzKQogICAgICAgICAgICAgICAgYWN0dWFsPXJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLGssYyxzKQogICAgICAgICAgICBncz10dXBsZSh0b3JjaC5saW5zcGFjZSgtNjU1MzYsNjU1MzYseC5udW1lbCgpLGRldmljZT0iY3VkYSIpLnJlc2hhcGUoeC5zaGFwZSkgZm9yIHggaW4gYWN0dWFsKQogICAgICAgICAgICBlZz10b3JjaC5hdXRvZ3JhZC5ncmFkKGV4cGVjdGVkLChxLGspLGdzKQogICAgICAgICAgICBhZz10b3JjaC5hdXRvZ3JhZC5ncmFkKGFjdHVhbCwocSxrKSxncykKICAgICAgICAgICAgZm9yIGFhLGVlIGluIHppcCgoKmFjdHVhbCwqYWcpLCgqZXhwZWN0ZWQsKmVnKSk6ZXhhY3QoYWEsZWUpCiAgICAgICAgICAgIGZvciB5LHggaW4gemlwKGFjdHVhbCwocSxrKSk6YXNzZXJ0IHkuc3RyaWRlKCk9PXguc3RyaWRlKCkKICAgICAgICAgICAgcmVzdWx0WyJjaGVja3MiXS5hcHBlbmQoZGljdChuYW1lPSJzY2FsZWRfYm91bmRhcnkiLGI9Yix0PXQsY29lZmZpY2llbnRfYmF0Y2g9Y2IpKQogICAgICAgICMgRXhoYXVzdGl2ZSBzaWduZWQtemVybyBhcml0aG1ldGljOyBjYWxsIG9uZS1pbnB1dCBrZXJuZWwgYm91bmRhcnkgb25seSwKICAgICAgICAjIHdpdGhvdXQgYnJvYWRlbmluZyB0aGUgcHJvZHVjdGlvbiBkaXNwYXRjaGVyIGJleW9uZCBEMTI4L0gxMjoyLgogICAgICAgIGltcG9ydCBpdGVydG9vbHMKICAgICAgICB2PXRvcmNoLnRlbnNvcihsaXN0KGl0ZXJ0b29scy5wcm9kdWN0KFswLiwtMC4sMS4sLTEuXSxyZXBlYXQ9NikpLGRldmljZT0iY3VkYSIpCiAgICAgICAgcT10b3JjaC56ZXJvcygoMSxsZW4odiksMSwyKSxkZXZpY2U9ImN1ZGEiKS50cmFuc3Bvc2UoMSwyKS5yZXF1aXJlc19ncmFkXygpCiAgICAgICAgaz1xLmRldGFjaCgpLmNsb25lKCkucmVxdWlyZXNfZ3JhZF8oKQogICAgICAgIGMscz12WzosOjJdLnJlc2hhcGUoMSxsZW4odiksMikuY29udGlndW91cygpLHZbOiwyOjRdLnJlc2hhcGUoMSxsZW4odiksMikuY29udGlndW91cygpCiAgICAgICAgZz12WzosNDpdLnJlc2hhcGVfYXMocSkKICAgICAgICBlPXJlZmVyZW5jZShxLGssYyxzKQogICAgICAgIGFjdHVhbD10dXBsZShyb3RhcnkuX0Zyb3plblJvUEUuYXBwbHkoeCxjLHMpIGZvciB4IGluIChxLGspKQogICAgICAgIGVnPXRvcmNoLmF1dG9ncmFkLmdyYWQoZSwocSxrKSwoZyxnKSk7YWc9dG9yY2guYXV0b2dyYWQuZ3JhZChhY3R1YWwsKHEsayksKGcsZykpCiAgICAgICAgZm9yIGFhLGVlIGluIHppcCgoKmFjdHVhbCwqYWcpLCgqZSwqZWcpKTpleGFjdChhYSxlZSkKICAgICAgICByZXN1bHRbImNoZWNrcyJdLmFwcGVuZChkaWN0KG5hbWU9ImV4aGF1c3RpdmVfc2lnbmVkX3plcm8iLGNvbWJpbmF0aW9ucz1sZW4odikpKQogICAgICAgICMgRmlyc3Qtb3JkZXIgbm9uZmluaXRlIGNsYXNzZXMvc3Vibm9ybWFsIGFyaXRobWV0aWM6IG5vIE5hTiBwYXlsb2FkIGNsYWltLgogICAgICAgIHY9dG9yY2gudGVuc29yKFswLiwtMC4sZmxvYXQoImluZiIpLGZsb2F0KCItaW5mIiksZmxvYXQoIm5hbiIpLDFlLTQwLC0xZS00MCwxLl0sZGV2aWNlPSJjdWRhIikKICAgICAgICBxPXYucmVwZWF0KDgpLnJlc2hhcGUoMSw4LDEsOCkudHJhbnNwb3NlKDEsMikucmVxdWlyZXNfZ3JhZF8oKTtrPXEuZGV0YWNoKCkuY2xvbmUoKS5yZXF1aXJlc19ncmFkXygpCiAgICAgICAgYz12LnJlcGVhdCg4KS5yZXNoYXBlKDEsOCw4KS5jb250aWd1b3VzKCk7cz1jLmZsaXAoLTEpLmNvbnRpZ3VvdXMoKQogICAgICAgIGU9cmVmZXJlbmNlKHEsayxjLHMpO2FjdHVhbD10dXBsZShyb3RhcnkuX0Zyb3plblJvUEUuYXBwbHkoeCxjLHMpIGZvciB4IGluIChxLGspKQogICAgICAgIGVnPXRvcmNoLmF1dG9ncmFkLmdyYWQoZSwocSxrKSwocS5kZXRhY2goKSxrLmRldGFjaCgpKSk7YWc9dG9yY2guYXV0b2dyYWQuZ3JhZChhY3R1YWwsKHEsayksKHEuZGV0YWNoKCksay5kZXRhY2goKSkpCiAgICAgICAgZm9yIGFhLGVlIGluIHppcCgoKmFjdHVhbCwqYWcpLCgqZSwqZWcpKTpleGFjdChhYSxlZSkKICAgICAgICByZXN1bHRbImNoZWNrcyJdLmFwcGVuZChkaWN0KG5hbWU9Im5vbmZpbml0ZV9jbGFzc2VzX3N1Ym5vcm1hbF9hbmRfc2lnbmVkX3plcm8iKSkKICAgICAgICAjIEV4ZXJjaXNlIHVuY2hhbmdlZCBTRFBBIGF1dG9jYXN0L0dRQSB3aXRoIEZQMzIgcS9rIGlucHV0cywgYm90aCBtYXNrIHJvdXRlcy4KICAgICAgICBmcm9tIHRyYW5zZm9ybWVycy5pbnRlZ3JhdGlvbnMuc2RwYV9hdHRlbnRpb24gaW1wb3J0IHNkcGFfYXR0ZW50aW9uX2ZvcndhcmQKICAgICAgICBmcm9tIHR5cGVzIGltcG9ydCBTaW1wbGVOYW1lc3BhY2UKICAgICAgICBmb3IgbWFza2VkIGluIChGYWxzZSwgVHJ1ZSk6CiAgICAgICAgICAgIHEsayxjLHM9aW5wdXRzKGI9Mix0PTE3LGRldmljZT0iY3VkYSIpCiAgICAgICAgICAgIHY9dG9yY2gucmFuZG4oKDIsMTcsMiwxMjgpLGRldmljZT0iY3VkYSIpLnRyYW5zcG9zZSgxLDIpLnJlcXVpcmVzX2dyYWRfKCkKICAgICAgICAgICAgb3duZXI9U2ltcGxlTmFtZXNwYWNlKG51bV9rZXlfdmFsdWVfZ3JvdXBzPTYsaXNfY2F1c2FsPVRydWUpCiAgICAgICAgICAgIG1hc2s9dG9yY2gub25lcygoMiwxLDE3LDE3KSxkZXZpY2U9ImN1ZGEiLGR0eXBlPXRvcmNoLmJvb2wpLnRyaWwoKSBpZiBtYXNrZWQgZWxzZSBOb25lCiAgICAgICAgICAgIGRlZiBhdHRlbnRpb24oZm4pOgogICAgICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgiY3VkYSIsZHR5cGU9dG9yY2guZmxvYXQxNik6CiAgICAgICAgICAgICAgICAgICAgcXIsa3I9Zm4ocSxrLGMscykKICAgICAgICAgICAgICAgICAgICByZXR1cm4gc2RwYV9hdHRlbnRpb25fZm9yd2FyZChvd25lcixxcixrcix2LG1hc2ssZHJvcG91dD0wLixzY2FsaW5nPTEyOCoqLTAuNSlbMF0KICAgICAgICAgICAgcm5nPXRvcmNoLmN1ZGEuZ2V0X3JuZ19zdGF0ZSgpLmNsb25lKCkKICAgICAgICAgICAgZSxhY3R1YWw9YXR0ZW50aW9uKHJlZmVyZW5jZSksYXR0ZW50aW9uKHJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYikKICAgICAgICAgICAgZXhhY3QoYWN0dWFsLGUpO2Fzc2VydCBhY3R1YWwuZHR5cGU9PXRvcmNoLmZsb2F0MTYKICAgICAgICAgICAgYXNzZXJ0IHRvcmNoLmVxdWFsKHJuZyx0b3JjaC5jdWRhLmdldF9ybmdfc3RhdGUoKSkKICAgICAgICAgICAgZz10b3JjaC5vbmVzX2xpa2UoZSkqNjQKICAgICAgICAgICAgZWc9dG9yY2guYXV0b2dyYWQuZ3JhZChlLChxLGssdiksZyk7YWc9dG9yY2guYXV0b2dyYWQuZ3JhZChhY3R1YWwsKHEsayx2KSxnKQogICAgICAgICAgICBmb3IgYWEsZWUgaW4gemlwKGFnLGVnKTp0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhYSxlZSxydG9sPTJlLTUsYXRvbD0yZS01LGVxdWFsX25hbj1UcnVlKQogICAgICAgICAgICByZXN1bHRbImNoZWNrcyJdLmFwcGVuZChkaWN0KG5hbWU9InVuY2hhbmdlZF9zZHBhX2F1dG9jYXN0X2dxYSIsbWFza2VkPW1hc2tlZCkpCiAgICAgICAgIyBVbmNoYW5nZWQgcmVhbCBRd2VuMkF0dGVudGlvbiBTRFBBIGJvdW5kYXJ5LCBubyBwcmV0cmFpbmVkIG1vZGVsIGxvYWQuCiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxzLnF3ZW4yIGltcG9ydCBtb2RlbGluZ19xd2VuMiBhcyBtb2QKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycy5tb2RlbHMucXdlbjIuY29uZmlndXJhdGlvbl9xd2VuMiBpbXBvcnQgUXdlbjJDb25maWcKICAgICAgICB0b3JjaC5tYW51YWxfc2VlZCg0NzEzMikKICAgICAgICBjZmc9UXdlbjJDb25maWcoaGlkZGVuX3NpemU9MTUzNixudW1fYXR0ZW50aW9uX2hlYWRzPTEyLG51bV9rZXlfdmFsdWVfaGVhZHM9MiwKICAgICAgICAgICAgICAgICAgICAgICBpbnRlcm1lZGlhdGVfc2l6ZT0zMixudW1faGlkZGVuX2xheWVycz0xLGF0dGVudGlvbl9kcm9wb3V0PTAuLAogICAgICAgICAgICAgICAgICAgICAgIHJvcGVfcGFyYW1ldGVycz17InJvcGVfdHlwZSI6ImRlZmF1bHQiLCJyb3BlX3RoZXRhIjoxXzAwMF8wMDAufSkKICAgICAgICBjZmcuX2F0dG5faW1wbGVtZW50YXRpb249InNkcGEiCiAgICAgICAgbW9kdWxlPW1vZC5Rd2VuMkF0dGVudGlvbihjZmcsMCkuY3VkYSgpLnRyYWluKCkKICAgICAgICBlbWI9bW9kLlF3ZW4yUm90YXJ5RW1iZWRkaW5nKGNmZykuY3VkYSgpCiAgICAgICAgZm9yIG1hc2tlZCBpbiAoRmFsc2UsVHJ1ZSk6CiAgICAgICAgICAgIGhpZGRlbj10b3JjaC5yYW5kbigoMiwxNywxNTM2KSxkZXZpY2U9ImN1ZGEiLHJlcXVpcmVzX2dyYWQ9VHJ1ZSkKICAgICAgICAgICAgY29lZmY9ZW1iKGhpZGRlbix0b3JjaC5hcmFuZ2UoMTcsZGV2aWNlPSJjdWRhIikudW5zcXVlZXplKDApKQogICAgICAgICAgICBtYXNrPXRvcmNoLm9uZXMoKDIsMSwxNywxNyksZGV2aWNlPSJjdWRhIixkdHlwZT10b3JjaC5ib29sKS50cmlsKCkgaWYgbWFza2VkIGVsc2UgTm9uZQogICAgICAgICAgICB2YXJpYWJsZXM9KGhpZGRlbiwqbW9kdWxlLnBhcmFtZXRlcnMoKSk7dXBzdHJlYW09dG9yY2gucmFuZG5fbGlrZShoaWRkZW4pKjY0CiAgICAgICAgICAgIG9yaWdpbmFsPW1vZC5hcHBseV9yb3RhcnlfcG9zX2VtYgogICAgICAgICAgICBybmc9dG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkuY2xvbmUoKQogICAgICAgICAgICBlPW1vZHVsZShoaWRkZW4sY29lZmYsbWFzaylbMF07ZWc9dG9yY2guYXV0b2dyYWQuZ3JhZChlLHZhcmlhYmxlcyx1cHN0cmVhbSkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgYXNzZXJ0IHJvdGFyeS5wYXRjaF9xd2VuMl9yb3RhcnkoKQogICAgICAgICAgICAgICAgY291bnQ9cm90YXJ5Ll9DT1VOVFNbImZvcndhcmRfbGF1bmNoX3JldHVybmVkIl0KICAgICAgICAgICAgICAgIGFjdHVhbD1tb2R1bGUoaGlkZGVuLGNvZWZmLG1hc2spWzBdO2FnPXRvcmNoLmF1dG9ncmFkLmdyYWQoYWN0dWFsLHZhcmlhYmxlcyx1cHN0cmVhbSkKICAgICAgICAgICAgICAgIGFzc2VydCByb3RhcnkuX0NPVU5UU1siZm9yd2FyZF9sYXVuY2hfcmV0dXJuZWQiXT09Y291bnQrMgogICAgICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICAgICAgbW9kLmFwcGx5X3JvdGFyeV9wb3NfZW1iPW9yaWdpbmFsCiAgICAgICAgICAgIGFzc2VydCB0b3JjaC5lcXVhbChybmcsdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkpCiAgICAgICAgICAgIGV4YWN0KGFjdHVhbCxlKQogICAgICAgICAgICAjIFVwc3RyZWFtIHNoYXJlZC1hbmNlc3RyeSBzY2hlZHVsaW5nIGFuZCBTRFBBIHJlZHVjdGlvbnMgY2FuIGRpZmZlcjsKICAgICAgICAgICAgIyByYXcgc2NhbGVkIGdyYWRpZW50cyBhcmUgY29tcGFyZWQsIG5vdCBkaXZpZGVkIHRvIGhpZGUgYSBtaXNtYXRjaC4KICAgICAgICAgICAgZm9yIGFhLGVlIGluIHppcChhZyxlZyk6dG9yY2gudGVzdGluZy5hc3NlcnRfY2xvc2UoYWEsZWUscnRvbD0yZS01LGF0b2w9MmUtNSxlcXVhbF9uYW49VHJ1ZSkKICAgICAgICAgICAgcmVzdWx0WyJjaGVja3MiXS5hcHBlbmQoZGljdChuYW1lPSJyZWFsX3F3ZW4yX3NkcGFfcm91dGUiLG1hc2tlZD1tYXNrZWQpKQogICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTtyZXN1bHRbImdwdV9leGVjdXRlZCJdPVRydWUKICAgICAgICBwdHhmaWxlcz1zb3J0ZWQoKG91dC8icHR4IikuZ2xvYigiKi5wdHgiKSk7YXNzZXJ0IHB0eGZpbGVzCiAgICAgICAgcHR4Y2hlY2tzPVtdCiAgICAgICAgZm9yIHBhdGggaW4gcHR4ZmlsZXM6CiAgICAgICAgICAgIHRleHQ9cGF0aC5yZWFkX3RleHQoKQogICAgICAgICAgICBhc3NlcnQgbm90IHJlLnNlYXJjaChyIlxiZm1hKD86XC58XHMpIix0ZXh0KSxwYXRoLm5hbWUKICAgICAgICAgICAgYXNzZXJ0IG5vdCByZS5zZWFyY2gociJcYig/Om11bHxhZGQpXC5bXlxuO10qZnR6Iix0ZXh0KSxwYXRoLm5hbWUKICAgICAgICAgICAgYXNzZXJ0IHJlLnNlYXJjaChyIlxibXVsKD86XC5ybik/XC5mMzIiLHRleHQpLHBhdGgubmFtZQogICAgICAgICAgICBhc3NlcnQgcmUuc2VhcmNoKHIiXGJhZGQoPzpcLnJuKT9cLmYzMiIsdGV4dCkscGF0aC5uYW1lCiAgICAgICAgICAgIHB0eGNoZWNrcy5hcHBlbmQoZGljdChmaWxlPXBhdGgubmFtZSxzaGEyNTY9aGFzaGxpYi5zaGEyNTYocGF0aC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpKSkKICAgICAgICByZXN1bHQudXBkYXRlKHN0YXR1cz0iUEFTU0VEX1BSRUZMSUdIVF9PTkxZIixwdHhfY2hlY2tzPXB0eGNoZWNrcyxjb3VudHM9ZGljdChyb3RhcnkuX0NPVU5UUykpCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShlcnJvcj1yZXByKGV4YyksdHJhY2ViYWNrPXRyYWNlYmFjay5mb3JtYXRfZXhjKCkpCiAgICBmaW5hbGx5OgogICAgICAgIChvdXQvInByZWZsaWdodC5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdCxpbmRlbnQ9MikrIlxuIikKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0LGluZGVudD0yKSkKICAgIHJhaXNlIFN5c3RlbUV4aXQoMCBpZiByZXN1bHRbInN0YXR1cyJdPT0iUEFTU0VEX1BSRUZMSUdIVF9PTkxZIiBlbHNlIDEpCgppZiBfX25hbWVfXz09Il9fbWFpbl9fIjptYWluKCkK', 'old_solution/rotary.py': 'IiIiT3JpZ2luYWwgb3V0LW9mLXBsYWNlIEZQMzIgaGFsZi1zcGxpdCBSb1BFOyBubyBub3JtYWxpemF0aW9uIG9yIGF0dGVudGlvbiBwYXRjaC4KClByb2R1Y3Rpb24gZGlzcGF0Y2ggaXMgQ1VEQS1vbmx5IGFuZCBmaXhlZC10YXNrIGdlb21ldHJ5LiBDUFUgaW50ZXJwcmV0YXRpb24gaXMKYW4gZXhwbGljaXRseSByZXF1ZXN0ZWQgdGVzdCBwYXRoLCBuZXZlciBhIHBlcmZvcm1hbmNlIG9yIEdQVSB2YWxpZGF0aW9uIGNsYWltLgoiIiIKaW1wb3J0IGZ1bmN0b29scwppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW5zcGVjdAppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKCmltcG9ydCB0b3JjaAppbXBvcnQgdHJpdG9uCmltcG9ydCB0cml0b24ubGFuZ3VhZ2UgYXMgdGwKCkVYUEVDVEVEX1FXRU5fQkxPQiA9ICJiNGZlZDk4ZmFmYWU4ZTRlZTM5YmUzYjhjZDc2NmIxYzg1YjA3OWNkIgpfQ09VTlRTID0gQ291bnRlcigpCgoKZGVmIF9ldmVudChraW5kLCAqKmV4dHJhKToKICAgIF9DT1VOVFNba2luZF0gKz0gMQogICAgbiA9IF9DT1VOVFNba2luZF0KICAgIGlmIG4gPT0gMSBvciBuICUgMjU2ID09IDA6CiAgICAgICAgcHJpbnQoIlJPVEFSWV9FVklERU5DRSAiICsganNvbi5kdW1wcyhkaWN0KGV2ZW50PWtpbmQsIGNvdW50PW4sICoqZXh0cmEpLCBzb3J0X2tleXM9VHJ1ZSksIGZsdXNoPVRydWUpCgoKZGVmIF9zYXZlX2NvZGVnZW4oY29tcGlsZWQsIG5hbWUpOgogICAgIyBPcHQtaW4gcHJlZmxpZ2h0LW9ubHkgZXZpZGVuY2UuIE1lYXN1cmVkIHJ1bnMgbmV2ZXIgc2V0IHRoaXMgZW52aXJvbm1lbnQga2V5LgogICAgZGlyZWN0b3J5ID0gb3MuZW52aXJvbi5nZXQoIlNPVVBfUk9UQVJZX0NPREVHRU5fRElSIikKICAgIGlmIGRpcmVjdG9yeSBhbmQgY29tcGlsZWQgaXMgbm90IE5vbmU6CiAgICAgICAgcHR4ID0gY29tcGlsZWQuYXNtLmdldCgicHR4IiwgIiIpCiAgICAgICAgaWYgbm90IHB0eDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJObyBnZW5lcmF0ZWQgUFRYIGF2YWlsYWJsZSBmb3IgcmVxdWlyZWQgcHJlZmxpZ2h0IGluc3BlY3Rpb24iKQogICAgICAgIHJvb3QgPSBQYXRoKGRpcmVjdG9yeSkKICAgICAgICByb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihwdHguZW5jb2RlKCkpLmhleGRpZ2VzdCgpCiAgICAgICAgKHJvb3QgLyAobmFtZSArICJfIiArIGRpZ2VzdFs6MTZdICsgIi5wdHgiKSkud3JpdGVfdGV4dChwdHgpCgoKQHRyaXRvbi5qaXQKZGVmIF9zaWduZmxpcF9mMzIoeCk6CiAgICAjIEV4YWN0IHVuYXJ5IG5lZ2F0aW9uLCBpbmNsdWRpbmcgLTAuMC4gTm90IHRoZSBub24tZXF1aXZhbGVudCBleHByZXNzaW9uIDAteC4KICAgIHJldHVybiAoeC50byh0bC51aW50MzIsIGJpdGNhc3Q9VHJ1ZSkgXiAweDgwMDAwMDAwKS50byh0bC5mbG9hdDMyLCBiaXRjYXN0PVRydWUpCgoKQHRyaXRvbi5qaXQoZG9fbm90X3NwZWNpYWxpemU9WyJUIiwgIk4iXSkKZGVmIHJvdGFyeV9hcHBseV9mMzIoWCwgQywgUywgWSwgSDogdGwuY29uc3RleHByLCBULAogICAgICAgICAgICAgICAgICAgICBEOiB0bC5jb25zdGV4cHIsIENCOiB0bC5jb25zdGV4cHIsIE4sCiAgICAgICAgICAgICAgICAgICAgIEJMT0NLOiB0bC5jb25zdGV4cHIpOgogICAgIyBGbGF0dGVuIHRoZSBkZW5zZSBCLFQsSCwoRC8yKSBwaHlzaWNhbCBvcmRlciBvZiB0aGUgcHJvamVjdGlvbiBzdG9yYWdlLgogICAgaSA9IHRsLnByb2dyYW1faWQoMCkgKiBCTE9DSyArIHRsLmFyYW5nZSgwLCBCTE9DSykKICAgIGhhbGYgPSBEIC8vIDIKICAgIGogPSBpICUgaGFsZgogICAgaCA9IChpIC8vIGhhbGYpICUgSAogICAgdCA9IChpIC8vIChoYWxmICogSCkpICUgVAogICAgYiA9IGkgLy8gKGhhbGYgKiBIICogVCkKICAgIHhwID0gKChiICogVCArIHQpICogSCArIGgpICogRCArIGoKICAgIGNwID0gKChiIGlmIENCICE9IDEgZWxzZSAwKSAqIFQgKyB0KSAqIEQgKyBqCiAgICBtID0gaSA8IE4KICAgIGEgPSB0bC5sb2FkKFggKyB4cCwgbSwgb3RoZXI9MCkKICAgIHogPSB0bC5sb2FkKFggKyB4cCArIGhhbGYsIG0sIG90aGVyPTApCiAgICBjMCA9IHRsLmxvYWQoQyArIGNwLCBtLCBvdGhlcj0wKQogICAgYzEgPSB0bC5sb2FkKEMgKyBjcCArIGhhbGYsIG0sIG90aGVyPTApCiAgICBzMCA9IHRsLmxvYWQoUyArIGNwLCBtLCBvdGhlcj0wKQogICAgczEgPSB0bC5sb2FkKFMgKyBjcCArIGhhbGYsIG0sIG90aGVyPTApCiAgICAjIGVuYWJsZV9mcF9mdXNpb249RmFsc2UgYXQgbGF1bmNoOiB0d28gcm91bmRlZCBwcm9kdWN0cyB0aGVuIHJvdW5kZWQgYWRkLgogICAgeTAgPSB0bC5hZGQodGwubXVsKGEsIGMwKSwgdGwubXVsKF9zaWduZmxpcF9mMzIoeiksIHMwKSkKICAgIHkxID0gdGwuYWRkKHRsLm11bCh6LCBjMSksIHRsLm11bChhLCBzMSkpCiAgICB0bC5zdG9yZShZICsgeHAsIHkwLCBtKQogICAgdGwuc3RvcmUoWSArIHhwICsgaGFsZiwgeTEsIG0pCgoKQHRyaXRvbi5qaXQoZG9fbm90X3NwZWNpYWxpemU9WyJUIiwgIk4iLCAiR0IiLCAiR0giLCAiR1QiLCAiR0QiXSkKZGVmIHJvdGFyeV9hZGpvaW50X2YzMihHLCBDLCBTLCBEWCwgSDogdGwuY29uc3RleHByLCBULAogICAgICAgICAgICAgICAgICAgICAgIEQ6IHRsLmNvbnN0ZXhwciwgQ0I6IHRsLmNvbnN0ZXhwciwgTiwKICAgICAgICAgICAgICAgICAgICAgICBHQiwgR0gsIEdULAogICAgICAgICAgICAgICAgICAgICAgIEdELCBCTE9DSzogdGwuY29uc3RleHByKToKICAgIGkgPSB0bC5wcm9ncmFtX2lkKDApICogQkxPQ0sgKyB0bC5hcmFuZ2UoMCwgQkxPQ0spCiAgICBoYWxmID0gRCAvLyAyCiAgICBqID0gaSAlIGhhbGYKICAgIGggPSAoaSAvLyBoYWxmKSAlIEgKICAgIHQgPSAoaSAvLyAoaGFsZiAqIEgpKSAlIFQKICAgIGIgPSBpIC8vIChoYWxmICogSCAqIFQpCiAgICB4cCA9ICgoYiAqIFQgKyB0KSAqIEggKyBoKSAqIEQgKyBqCiAgICBncCA9IGIgKiBHQiArIGggKiBHSCArIHQgKiBHVCArIGogKiBHRAogICAgY3AgPSAoKGIgaWYgQ0IgIT0gMSBlbHNlIDApICogVCArIHQpICogRCArIGoKICAgIG0gPSBpIDwgTgogICAgZzAgPSB0bC5sb2FkKEcgKyBncCwgbSwgb3RoZXI9MCkKICAgIGcxID0gdGwubG9hZChHICsgZ3AgKyBoYWxmICogR0QsIG0sIG90aGVyPTApCiAgICBjMCA9IHRsLmxvYWQoQyArIGNwLCBtLCBvdGhlcj0wKQogICAgYzEgPSB0bC5sb2FkKEMgKyBjcCArIGhhbGYsIG0sIG90aGVyPTApCiAgICBzMCA9IHRsLmxvYWQoUyArIGNwLCBtLCBvdGhlcj0wKQogICAgczEgPSB0bC5sb2FkKFMgKyBjcCArIGhhbGYsIG0sIG90aGVyPTApCiAgICAjIE5hdGl2ZSBzbGljZS1iYWNrd2FyZCBwYWRzIHdpdGggKzAgdGhlbiBhY2N1bXVsYXRlcyB0d28gZGlzam9pbnQgc2xpY2VzLgogICAgIyBJdCBjYW5vbmljYWxpemVzIGEgemVyby12YWx1ZWQgcm90YXRlZCBjb250cmlidXRpb24gdG8gcG9zaXRpdmUgemVyby4KICAgIHowID0gdGwubXVsKGcxLCBzMSkKICAgIHoxID0gX3NpZ25mbGlwX2YzMih0bC5tdWwoZzAsIHMwKSkKICAgIHowID0gdGwud2hlcmUoejAgPT0gMC4wLCAwLjAsIHowKQogICAgejEgPSB0bC53aGVyZSh6MSA9PSAwLjAsIDAuMCwgejEpCiAgICBkeDAgPSB0bC5hZGQodGwubXVsKGcwLCBjMCksIHowKQogICAgZHgxID0gdGwuYWRkKHRsLm11bChnMSwgYzEpLCB6MSkKICAgIHRsLnN0b3JlKERYICsgeHAsIGR4MCwgbSkKICAgIHRsLnN0b3JlKERYICsgeHAgKyBoYWxmLCBkeDEsIG0pCgoKZGVmIF9kZW5zZV9wcm9qZWN0aW9uKHgpOgogICAgaWYgeC5uZGltICE9IDQ6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICBiLCBoLCB0LCBkID0geC5zaGFwZQogICAgcmV0dXJuIGIgPiAwIGFuZCBoID4gMCBhbmQgdCA+IDAgYW5kIGQgPiAwIGFuZCBkICUgMiA9PSAwIGFuZCB4LnN0cmlkZSgpID09ICh0KmgqZCwgZCwgaCpkLCAxKQoKCmRlZiBfc3VwcG9ydF9yZWFzb24ocSwgaywgY29zLCBzaW4sIHVuc3F1ZWV6ZV9kaW09MSwgKiwgY3VkYV9vbmx5PVRydWUsIHRhc2tfb25seT1UcnVlKToKICAgIHhzID0gKHEsIGssIGNvcywgc2luKQogICAgaWYgYW55KHR5cGUoeCkgaXMgbm90IHRvcmNoLlRlbnNvciBmb3IgeCBpbiB4cyk6CiAgICAgICAgcmV0dXJuICJ0ZW5zb3Jfc3ViY2xhc3MiCiAgICBpZiB1bnNxdWVlemVfZGltICE9IDE6CiAgICAgICAgcmV0dXJuICJheGlzIgogICAgaWYgYW55KHguZHR5cGUgIT0gdG9yY2guZmxvYXQzMiBvciB4LmRldmljZSAhPSBxLmRldmljZSBvciB4LmlzX25lZygpIG9yIHguaXNfY29uaigpIGZvciB4IGluIHhzKToKICAgICAgICByZXR1cm4gImR0eXBlX2RldmljZV9vcl9sYXp5X3ZpZXciCiAgICBpZiBjdWRhX29ubHkgYW5kIHEuZGV2aWNlLnR5cGUgIT0gImN1ZGEiOgogICAgICAgIHJldHVybiAibm90X2N1ZGEiCiAgICBpZiBhbnkoeC5sYXlvdXQgIT0gdG9yY2guc3RyaWRlZCBmb3IgeCBpbiB4cyk6CiAgICAgICAgcmV0dXJuICJsYXlvdXQiCiAgICBpZiBub3QgX2RlbnNlX3Byb2plY3Rpb24ocSkgb3Igbm90IF9kZW5zZV9wcm9qZWN0aW9uKGspOgogICAgICAgIHJldHVybiAicHJvamVjdGlvbl9zdHJpZGUiCiAgICBiLCBocSwgdCwgZCA9IHEuc2hhcGUKICAgIGlmIGsuc2hhcGVbMF0gIT0gYiBvciBrLnNoYXBlWzI6XSAhPSAodCwgZCk6CiAgICAgICAgcmV0dXJuICJxa19nZW9tZXRyeSIKICAgIGlmIHRhc2tfb25seSBhbmQgbm90IChiIDw9IDQgYW5kIHQgPD0gNTEyIGFuZCBocSA9PSAxMiBhbmQgay5zaGFwZVsxXSA9PSAyIGFuZCBkID09IDEyOCk6CiAgICAgICAgcmV0dXJuICJvdXRzaWRlX2ZpeGVkX3Rhc2tfZ2VvbWV0cnkiCiAgICBpZiBjb3MubmRpbSAhPSAzIG9yIHNpbi5zaGFwZSAhPSBjb3Muc2hhcGUgb3IgY29zLnNoYXBlWzBdIG5vdCBpbiAoMSwgYikgb3IgY29zLnNoYXBlWzE6XSAhPSAodCwgZCk6CiAgICAgICAgcmV0dXJuICJjb2VmZmljaWVudF9nZW9tZXRyeSIKICAgIGlmIGNvcy5yZXF1aXJlc19ncmFkIG9yIHNpbi5yZXF1aXJlc19ncmFkOgogICAgICAgIHJldHVybiAidHJhaW5hYmxlX2NvZWZmaWNpZW50IgogICAgaWYgbm90IGNvcy5pc19jb250aWd1b3VzKCkgb3Igbm90IHNpbi5pc19jb250aWd1b3VzKCk6CiAgICAgICAgcmV0dXJuICJjb2VmZmljaWVudF9zdHJpZGUiCiAgICBzdG9yYWdlID0gW3gudW50eXBlZF9zdG9yYWdlKCkuZGF0YV9wdHIoKSBmb3IgeCBpbiB4c10KICAgIGlmIHN0b3JhZ2VbMF0gPT0gc3RvcmFnZVsxXSBvciBhbnkocyBpbiBzdG9yYWdlWzI6XSBmb3IgcyBpbiBzdG9yYWdlWzoyXSk6CiAgICAgICAgcmV0dXJuICJzaGFyZWRfYWN0aXZhdGlvbl9zdG9yYWdlIgogICAgcmV0dXJuIE5vbmUKCgpkZWYgX3N1cHBvcnRlZCgqYXJncywgKiprd2FyZ3MpOgogICAgcmV0dXJuIF9zdXBwb3J0X3JlYXNvbigqYXJncywgKiprd2FyZ3MpIGlzIE5vbmUKCgpkZWYgbmF0aXZlX2Fkam9pbnQoZywgY29zLCBzaW4pOgogICAgIiIiRGlmZmVyZW50aWFibGUgZmFsbGJhY2sgZm9yIGhpZ2hlci1vcmRlciByZXF1ZXN0czsgZXhwbGljaXQgbmF0aXZlIHNsaWNlIGdyYXBoLiIiIgogICAgZCA9IGcuc2hhcGVbLTFdIC8vIDIKICAgIGMsIHMgPSBjb3MudW5zcXVlZXplKDEpLCBzaW4udW5zcXVlZXplKDEpCiAgICByb3RhdGVkX2dyYWQgPSBnICogcwogICAgYSwgYiA9IHJvdGF0ZWRfZ3JhZFsuLi4sIDpkXSwgcm90YXRlZF9ncmFkWy4uLiwgZDpdCiAgICAjIFR3byBzbGljZS1iYWNrd2FyZHMgcmVhbGx5IGRvIGFkZCBwYWRkZWQgdGVuc29yczsgcHJlc2VydmUgdGhlaXIgemVybyBzaWducy4KICAgIHphID0gdG9yY2guY2F0KCh0b3JjaC56ZXJvc19saWtlKGEpLCAtYSksIGRpbT0tMSkKICAgIHpiID0gdG9yY2guY2F0KChiLCB0b3JjaC56ZXJvc19saWtlKGIpKSwgZGltPS0xKQogICAgcmV0dXJuIGcgKiBjICsgKHphICsgemIpCgoKY2xhc3MgX0Zyb3plblJvUEUodG9yY2guYXV0b2dyYWQuRnVuY3Rpb24pOgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGZvcndhcmQoY3R4LCB4LCBjb3MsIHNpbik6CiAgICAgICAgY3R4LnNldF9tYXRlcmlhbGl6ZV9ncmFkcyhGYWxzZSkKICAgICAgICBjdHguc2F2ZV9mb3JfYmFja3dhcmQoY29zLCBzaW4pCiAgICAgICAgY3R4LnNoYXBlLCBjdHguc3RyaWRlID0gdHVwbGUoeC5zaGFwZSksIHguc3RyaWRlKCkKICAgICAgICB5ID0gdG9yY2guZW1wdHlfc3RyaWRlZCh4LnNoYXBlLCB4LnN0cmlkZSgpLCBkdHlwZT14LmR0eXBlLCBkZXZpY2U9eC5kZXZpY2UpCiAgICAgICAgYiwgaCwgdCwgZCA9IHguc2hhcGUKICAgICAgICBuID0geC5udW1lbCgpIC8vIDIKICAgICAgICBjb21waWxlZCA9IHJvdGFyeV9hcHBseV9mMzJbKHRyaXRvbi5jZGl2KG4sIDI1NiksKV0oeCwgY29zLCBzaW4sIHksIGgsIHQsIGQsIGNvcy5zaGFwZVswXSwgbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgQkxPQ0s9MjU2LCBudW1fd2FycHM9NCwgZW5hYmxlX2ZwX2Z1c2lvbj1GYWxzZSkKICAgICAgICBfc2F2ZV9jb2RlZ2VuKGNvbXBpbGVkLCAicm90YXJ5X2FwcGx5X2YzMiIpCiAgICAgICAgX2V2ZW50KCJmb3J3YXJkX2xhdW5jaF9yZXR1cm5lZCIsIHNoYXBlPWxpc3QoeC5zaGFwZSksIGR0eXBlPXN0cih4LmR0eXBlKSwgc3RyaWRlPWxpc3QoeC5zdHJpZGUoKSkpCiAgICAgICAgcmV0dXJuIHkKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgYmFja3dhcmQoY3R4LCBnKToKICAgICAgICBpZiBnIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBOb25lLCBOb25lLCBOb25lCiAgICAgICAgY29zLCBzaW4gPSBjdHguc2F2ZWRfdGVuc29ycwogICAgICAgIGlmIHRvcmNoLmlzX2dyYWRfZW5hYmxlZCgpOgogICAgICAgICAgICBfZXZlbnQoImhpZ2hlcl9vcmRlcl9uYXRpdmVfYWRqb2ludCIpCiAgICAgICAgICAgIHJldHVybiBuYXRpdmVfYWRqb2ludChnLCBjb3MsIHNpbiksIE5vbmUsIE5vbmUKICAgICAgICAjIFBvaW50ZXIgbG9hZHMgZG8gbm90IGltcGxlbWVudCBQeVRvcmNoJ3MgbGF6eSBuZWdhdGl2ZS9jb25qdWdhdGUgYml0cy4KICAgICAgICBnID0gZy5yZXNvbHZlX25lZygpLnJlc29sdmVfY29uaigpCiAgICAgICAgZHggPSB0b3JjaC5lbXB0eV9zdHJpZGVkKGN0eC5zaGFwZSwgY3R4LnN0cmlkZSwgZHR5cGU9Zy5kdHlwZSwgZGV2aWNlPWcuZGV2aWNlKQogICAgICAgIGIsIGgsIHQsIGQgPSBjdHguc2hhcGUKICAgICAgICBuID0gYipoKnQqZC8vMgogICAgICAgIGNvbXBpbGVkID0gcm90YXJ5X2Fkam9pbnRfZjMyWyh0cml0b24uY2RpdihuLCAyNTYpLCldKGcsIGNvcywgc2luLCBkeCwgaCwgdCwgZCwgY29zLnNoYXBlWzBdLCBuLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKmcuc3RyaWRlKCksIEJMT0NLPTI1NiwgbnVtX3dhcnBzPTQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbmFibGVfZnBfZnVzaW9uPUZhbHNlKQogICAgICAgIF9zYXZlX2NvZGVnZW4oY29tcGlsZWQsICJyb3RhcnlfYWRqb2ludF9mMzIiKQogICAgICAgIF9ldmVudCgiYmFja3dhcmRfbGF1bmNoX3JldHVybmVkIiwgc2hhcGU9bGlzdChjdHguc2hhcGUpLCB1cHN0cmVhbV9zdHJpZGU9bGlzdChnLnN0cmlkZSgpKSkKICAgICAgICByZXR1cm4gZHgsIE5vbmUsIE5vbmUKCgpkZWYgZnVzZWRfYXBwbHlfcm90YXJ5X3Bvc19lbWIocSwgaywgY29zLCBzaW4sIHVuc3F1ZWV6ZV9kaW09MSk6CiAgICBpbnRlcnByZXRpbmcgPSBvcy5lbnZpcm9uLmdldCgiVFJJVE9OX0lOVEVSUFJFVCIpID09ICIxIgogICAgcmVhc29uID0gX3N1cHBvcnRfcmVhc29uKHEsIGssIGNvcywgc2luLCB1bnNxdWVlemVfZGltLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGN1ZGFfb25seT1ub3QgaW50ZXJwcmV0aW5nLCB0YXNrX29ubHk9bm90IGludGVycHJldGluZykKICAgIGlmIHJlYXNvbjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJVbnN1cHBvcnRlZCBmcm96ZW4gRlAzMiByb3RhcnkgYm91bmRhcnk6ICIgKyByZWFzb24pCiAgICAjIFNlcGFyYXRlIEZ1bmN0aW9ucyBwcmVzZXJ2ZSBpbmRlcGVuZGVudGx5IGZyb3plbi91bnVzZWQgUS9LIGJyYW5jaGVzLgogICAgcmV0dXJuIF9Gcm96ZW5Sb1BFLmFwcGx5KHEsIGNvcywgc2luKSwgX0Zyb3plblJvUEUuYXBwbHkoaywgY29zLCBzaW4pCgoKZGVmIHBhdGNoX3F3ZW4yX3JvdGFyeSgpOgogICAgZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxzLnF3ZW4yIGltcG9ydCBtb2RlbGluZ19xd2VuMiBhcyBtb2R1bGUKICAgIG9yaWdpbmFsID0gbW9kdWxlLmFwcGx5X3JvdGFyeV9wb3NfZW1iCiAgICBpZiBnZXRhdHRyKG9yaWdpbmFsLCAiX29yaWdpbmFsX3JvdGFyeV9jYW5kaWRhdGUiLCBGYWxzZSk6CiAgICAgICAgcmV0dXJuIFRydWUKICAgIHJhdyA9IG9wZW4oaW5zcGVjdC5nZXRmaWxlKG1vZHVsZSksICJyYiIpLnJlYWQoKQogICAgYmxvYiA9IGhhc2hsaWIuc2hhMShiImJsb2IgIiArIHN0cihsZW4ocmF3KSkuZW5jb2RlKCkgKyBiIlwwIiArIHJhdykuaGV4ZGlnZXN0KCkKICAgIGlmIGJsb2IgIT0gRVhQRUNURURfUVdFTl9CTE9COgogICAgICAgIF9ldmVudCgicGF0Y2hfcmVmdXNlZF9zb3VyY2VfbWlzbWF0Y2giLCBvYnNlcnZlZD1ibG9iKQogICAgICAgIHJldHVybiBGYWxzZQoKICAgIEBmdW5jdG9vbHMud3JhcHMob3JpZ2luYWwpCiAgICBkZWYgZ3VhcmRlZChxLCBrLCBjb3MsIHNpbiwgdW5zcXVlZXplX2RpbT0xKToKICAgICAgICByZWFzb24gPSBfc3VwcG9ydF9yZWFzb24ocSwgaywgY29zLCBzaW4sIHVuc3F1ZWV6ZV9kaW0pCiAgICAgICAgaWYgcmVhc29uIGlzIG5vdCBOb25lOgogICAgICAgICAgICBfZXZlbnQoIm5hdGl2ZV9mYWxsYmFja18iICsgcmVhc29uKQogICAgICAgICAgICByZXR1cm4gb3JpZ2luYWwocSwgaywgY29zLCBzaW4sIHVuc3F1ZWV6ZV9kaW09dW5zcXVlZXplX2RpbSkKICAgICAgICByZXR1cm4gZnVzZWRfYXBwbHlfcm90YXJ5X3Bvc19lbWIocSwgaywgY29zLCBzaW4sIHVuc3F1ZWV6ZV9kaW0pCgogICAgZ3VhcmRlZC5fb3JpZ2luYWxfcm90YXJ5X2NhbmRpZGF0ZSA9IFRydWUKICAgIG1vZHVsZS5hcHBseV9yb3RhcnlfcG9zX2VtYiA9IGd1YXJkZWQKICAgIF9ldmVudCgicGF0Y2hfaW5zdGFsbGVkIiwgc291cmNlX2Jsb2I9YmxvYiwgb3duZWRfa2VybmVscz1bInJvdGFyeV9hcHBseV9mMzIiLCAicm90YXJ5X2Fkam9pbnRfZjMyIl0pCiAgICByZXR1cm4gVHJ1ZQo=', 'old_solution/test_rotary.py': 'IiIiT3JpZ2luYWwgZml4dHVyZXM7IGludGVycHJldGVyIHJlc3VsdHMgYXJlIG5vdCBDVURBIGNvZGUtZ2VuZXJhdGlvbiBldmlkZW5jZS4iIiIKaW1wb3J0IGl0ZXJ0b29scwppbXBvcnQgb3MKaW1wb3J0IHB5dGVzdAppbXBvcnQgdG9yY2gKZnJvbSAuY3B1X2NvbnRyYWN0IGltcG9ydCBpbnB1dHMsIHJlZmVyZW5jZSwgZXhhY3QKZnJvbSAuIGltcG9ydCByb3RhcnkKCklOVEVSUFJFVCA9IG9zLmVudmlyb24uZ2V0KCJUUklUT05fSU5URVJQUkVUIikgPT0gIjEiCktFUk5FTCA9IHB5dGVzdC5tYXJrLnNraXBpZihub3QgSU5URVJQUkVULCByZWFzb249IkNQVSBpbnRlcnByZXRlciBleHBsaWNpdGx5IGRpc2FibGVkIikKCgpAcHl0ZXN0Lm1hcmsucGFyYW1ldHJpemUoImNiIiwgWzEsMl0pCkBweXRlc3QubWFyay5wYXJhbWV0cml6ZSgic2NhbGUiLCBbMS4sNjU1MzYuXSkKQEtFUk5FTApkZWYgdGVzdF9maW5pdGVfc3RhZ2VkX2FyaXRobWV0aWMoY2Isc2NhbGUpOgogICAgcSxrLGMscyA9IGlucHV0cyhjYj1jYikKICAgIHZlcnNpb25zID0gW3guX3ZlcnNpb24gZm9yIHggaW4gKHEsayxjLHMpXQogICAgc25hcHNob3RzID0gW3guZGV0YWNoKCkuY2xvbmUoKSBmb3IgeCBpbiAocSxrLGMscyldCiAgICBybmcgPSB0b3JjaC5nZXRfcm5nX3N0YXRlKCkuY2xvbmUoKQogICAgZXhwZWN0ZWQgPSByZWZlcmVuY2UocSxrLGMscykKICAgIGFjdHVhbCA9IHJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLGssYyxzKQogICAgZ3MgPSB0dXBsZSh0b3JjaC5saW5zcGFjZSgtc2NhbGUsc2NhbGUseC5udW1lbCgpKS5yZXNoYXBlKHguc2hhcGUpIGZvciB4IGluIGV4cGVjdGVkKQogICAgZWcgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKGV4cGVjdGVkLChxLGspLGdzKQogICAgYWcgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKGFjdHVhbCwocSxrKSxncykKICAgIGZvciBhLGUseCBpbiB6aXAoYWN0dWFsLGV4cGVjdGVkLChxLGspKToKICAgICAgICBleGFjdChhLGUpCiAgICAgICAgYXNzZXJ0IGEuc3RyaWRlKCkgPT0geC5zdHJpZGUoKQogICAgICAgIGFzc2VydCBhLmRhdGFfcHRyKCkgIT0geC5kYXRhX3B0cigpCiAgICBmb3IgYSxlIGluIHppcChhZyxlZyk6IGV4YWN0KGEsZSkKICAgIGZvciB4LHZlcnNpb24sc25hcCBpbiB6aXAoKHEsayxjLHMpLHZlcnNpb25zLHNuYXBzaG90cyk6CiAgICAgICAgYXNzZXJ0IHguX3ZlcnNpb24gPT0gdmVyc2lvbgogICAgICAgIGV4YWN0KHgsc25hcCkKICAgIGFzc2VydCB0b3JjaC5lcXVhbChybmcsdG9yY2guZ2V0X3JuZ19zdGF0ZSgpKQoKCkBLRVJORUwKQHB5dGVzdC5tYXJrLnBhcmFtZXRyaXplKCJsYXlvdXQiLCBbImNvbnRpZ3VvdXMiLCJleHBhbmRlZCIsInNsaWNlZCIsIm5lZ2F0aXZlX3ZpZXciXSkKZGVmIHRlc3RfdXBzdHJlYW1fbGF5b3V0KGxheW91dCk6CiAgICBxLGssYyxzID0gaW5wdXRzKHQ9MykKICAgIGRlZiBncmFkaWVudCh4KToKICAgICAgICBpZiBsYXlvdXQgPT0gImV4cGFuZGVkIjogcmV0dXJuIHRvcmNoLm9uZXMoKDEsMSwxLDEpKS5leHBhbmRfYXMoeCkKICAgICAgICBpZiBsYXlvdXQgPT0gImNvbnRpZ3VvdXMiOiByZXR1cm4gdG9yY2gub25lcyh4LnNoYXBlKQogICAgICAgIGcgPSB0b3JjaC5vbmVzKCgqeC5zaGFwZVs6LTFdLCB4LnNoYXBlWy0xXSoyKSlbLi4uLDo6Ml0KICAgICAgICByZXR1cm4gdG9yY2guX25lZ192aWV3KGcpIGlmIGxheW91dCA9PSAibmVnYXRpdmVfdmlldyIgZWxzZSBnCiAgICBncyA9IHR1cGxlKGdyYWRpZW50KHgpIGZvciB4IGluIChxLGspKQogICAgZSA9IHRvcmNoLmF1dG9ncmFkLmdyYWQocmVmZXJlbmNlKHEsayxjLHMpLChxLGspLGdzKQogICAgYSA9IHRvcmNoLmF1dG9ncmFkLmdyYWQocm90YXJ5LmZ1c2VkX2FwcGx5X3JvdGFyeV9wb3NfZW1iKHEsayxjLHMpLChxLGspLGdzKQogICAgZm9yIGFhLGVlIGluIHppcChhLGUpOiBleGFjdChhYSxlZSkKCgpAS0VSTkVMCkBweXRlc3QubWFyay5wYXJhbWV0cml6ZSgicnEscmssdXNlZCIsIFsoVHJ1ZSxUcnVlLDApLChUcnVlLFRydWUsMSksKEZhbHNlLFRydWUsMSksKFRydWUsRmFsc2UsMCldKQpkZWYgdGVzdF9pbmRlcGVuZGVudF91bnVzZWRfb3V0cHV0cyhycSxyayx1c2VkKToKICAgIHEsayxjLHMgPSBpbnB1dHMoYj0xLHQ9MSkKICAgIHEucmVxdWlyZXNfZ3JhZF8ocnEpO2sucmVxdWlyZXNfZ3JhZF8ocmspCiAgICB2YXJpYWJsZXMgPSB0dXBsZSh4IGZvciB4IGluIChxLGspIGlmIHgucmVxdWlyZXNfZ3JhZCkKICAgIGV4cGVjdGVkLGFjdHVhbCA9IHJlZmVyZW5jZShxLGssYyxzKSxyb3RhcnkuZnVzZWRfYXBwbHlfcm90YXJ5X3Bvc19lbWIocSxrLGMscykKICAgIGFzc2VydCBbeC5yZXF1aXJlc19ncmFkIGZvciB4IGluIGFjdHVhbF0gPT0gW3JxLHJrXQogICAgZSA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoZXhwZWN0ZWRbdXNlZF0uc3VtKCksIHZhcmlhYmxlcywgYWxsb3dfdW51c2VkPVRydWUpCiAgICBhID0gdG9yY2guYXV0b2dyYWQuZ3JhZChhY3R1YWxbdXNlZF0uc3VtKCksIHZhcmlhYmxlcywgYWxsb3dfdW51c2VkPVRydWUpCiAgICBmb3IgYWEsZWUgaW4gemlwKGEsZSk6CiAgICAgICAgaWYgZWUgaXMgTm9uZTogYXNzZXJ0IGFhIGlzIE5vbmUKICAgICAgICBlbHNlOiBleGFjdChhYSxlZSkKCgpAS0VSTkVMCmRlZiB0ZXN0X2V4aGF1c3RpdmVfc2lnbmVkX3plcm9fbmF0aXZlX3NsaWNlX2Fkam9pbnQoKToKICAgIHZhbHVlcyA9IHRvcmNoLnRlbnNvcihsaXN0KGl0ZXJ0b29scy5wcm9kdWN0KFswLiwtMC4sMS4sLTEuXSxyZXBlYXQ9NikpKQogICAgbiA9IGxlbih2YWx1ZXMpCiAgICBxID0gdG9yY2guemVyb3MoKDEsbiwxLDIpKS50cmFuc3Bvc2UoMSwyKS5yZXF1aXJlc19ncmFkXygpCiAgICBrID0gcS5kZXRhY2goKS5jbG9uZSgpLnJlcXVpcmVzX2dyYWRfKCkKICAgIGMscyA9IHZhbHVlc1s6LDoyXS5yZXNoYXBlKDEsbiwyKS5jb250aWd1b3VzKCksIHZhbHVlc1s6LDI6NF0ucmVzaGFwZSgxLG4sMikuY29udGlndW91cygpCiAgICBnID0gdmFsdWVzWzosNDpdLnJlc2hhcGVfYXMocSkKICAgIGUsYSA9IHJlZmVyZW5jZShxLGssYyxzKSxyb3RhcnkuZnVzZWRfYXBwbHlfcm90YXJ5X3Bvc19lbWIocSxrLGMscykKICAgIGZvciBhYSxlZSBpbiB6aXAoYSxlKTogZXhhY3QoYWEsZWUpCiAgICBlZyA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoZSwocSxrKSwoZyxnKSkKICAgIGFnID0gdG9yY2guYXV0b2dyYWQuZ3JhZChhLChxLGspLChnLGcpKQogICAgZm9yIGFhLGVlIGluIHppcChhZyxlZyk6IGV4YWN0KGFhLGVlKQogICAgZXhhY3Qocm90YXJ5Lm5hdGl2ZV9hZGpvaW50KGcsYyxzKSxlZ1swXSkKCgpAS0VSTkVMCmRlZiB0ZXN0X25vbmZpbml0ZV9jbGFzc2VzX2FuZF9zdWJub3JtYWxzKCk6CiAgICB2YWxzID0gdG9yY2gudGVuc29yKFswLiwtMC4sZmxvYXQoJ2luZicpLGZsb2F0KCctaW5mJyksZmxvYXQoJ25hbicpLDFlLTQwLC0xZS00MCwxLl0pCiAgICBxID0gdmFscy5yZXBlYXQoOCkucmVzaGFwZSgxLDgsMSw4KS50cmFuc3Bvc2UoMSwyKS5yZXF1aXJlc19ncmFkXygpCiAgICBrID0gcS5kZXRhY2goKS5jbG9uZSgpLnJlcXVpcmVzX2dyYWRfKCkKICAgIGMgPSB2YWxzLnJlcGVhdCg4KS5yZXNoYXBlKDEsOCw4KS5jb250aWd1b3VzKCkKICAgIHMgPSB0b3JjaC5mbGlwKGMsWy0xXSkuY29udGlndW91cygpCiAgICBlLGEgPSByZWZlcmVuY2UocSxrLGMscykscm90YXJ5LmZ1c2VkX2FwcGx5X3JvdGFyeV9wb3NfZW1iKHEsayxjLHMpCiAgICBncyA9IChxLmRldGFjaCgpLGsuZGV0YWNoKCkpCiAgICBlZyxhZyA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoZSwocSxrKSxncyksdG9yY2guYXV0b2dyYWQuZ3JhZChhLChxLGspLGdzKQogICAgZm9yIGFhLGVlIGluIHppcCgoKmEsKmFnKSwoKmUsKmVnKSk6IGV4YWN0KGFhLGVlKQoKCkBLRVJORUwKZGVmIHRlc3RfY29lZmZpY2llbnRfbXV0YXRpb25fcmVqZWN0ZWRfYnlfYXV0b2dyYWQoKToKICAgIHEsayxjLHMgPSBpbnB1dHModD0xKQogICAgYSA9IHJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLGssYyxzKQogICAgYy5hZGRfKDEpCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoUnVudGltZUVycm9yLG1hdGNoPSJtb2RpZmllZCBieSBhbiBpbnBsYWNlIG9wZXJhdGlvbiIpOgogICAgICAgIGFbMF0uc3VtKCkuYmFja3dhcmQoKQoKCkBLRVJORUwKZGVmIHRlc3RfaGlnaGVyX29yZGVyX25hdGl2ZV9mYWxsYmFjaygpOgogICAgcSxrLGMscyA9IGlucHV0cyh0PTEpCiAgICBnID0gdG9yY2gub25lc19saWtlKHEscmVxdWlyZXNfZ3JhZD1UcnVlKQogICAgYSA9IHJvdGFyeS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYihxLGssYyxzKVswXQogICAgZSA9IHJlZmVyZW5jZShxLGssYyxzKVswXQogICAgYWcsID0gdG9yY2guYXV0b2dyYWQuZ3JhZChhLHEsZyxjcmVhdGVfZ3JhcGg9VHJ1ZSkKICAgIGVnLCA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoZSxxLGcsY3JlYXRlX2dyYXBoPVRydWUpCiAgICBleGFjdChhZyxlZykKICAgIGFhLCA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoYWcsZyx0b3JjaC5vbmVzX2xpa2UoYWcpKQogICAgZWUsID0gdG9yY2guYXV0b2dyYWQuZ3JhZChlZyxnLHRvcmNoLm9uZXNfbGlrZShlZykpCiAgICBleGFjdChhYSxlZSkKCgpkZWYgdGVzdF9tZXRhZGF0YV9yZWplY3Rpb25zX2FuZF9zbW9rZSgpOgogICAgZnJvbSAuY3B1X2NvbnRyYWN0IGltcG9ydCBydW4KICAgIGFzc2VydCBydW4oKVsicGFzc2VkIl0KICAgIHEsayxjLHMgPSBpbnB1dHMoKQogICAgcmVhc29uID0gbGFtYmRhICphLCoqa3c6IHJvdGFyeS5fc3VwcG9ydF9yZWFzb24oKmEsY3VkYV9vbmx5PUZhbHNlLCoqa3cpCiAgICBhc3NlcnQgcmVhc29uKHEscSxjLHMpIGlzIG5vdCBOb25lCiAgICBhc3NlcnQgcmVhc29uKHRvcmNoLl9uZWdfdmlldyhxKSxrLGMscykgaXMgbm90IE5vbmUKICAgIGFzc2VydCByZWFzb24ocSxrLGMscyx1bnNxdWVlemVfZGltPTIpIGlzIG5vdCBOb25lCiAgICBhc3NlcnQgcmVhc29uKHEuaGFsZigpLGssYyxzKSBpcyBub3QgTm9uZQogICAgYXNzZXJ0IHJlYXNvbihxLGssYy5leHBhbmQoMiwtMSwtMSkscy5leHBhbmQoMiwtMSwtMSkpIGlzIG5vdCBOb25lCiAgICBhc3NlcnQgcmVhc29uKHEsayxjWzosOi0xXSxzWzosOi0xXSkgaXMgbm90IE5vbmUKCgpkZWYgdGVzdF9wYXRjaF9pZGVtcG90ZW50X2NwdV9uYXRpdmVfYW5kX3JuZyhtb25rZXlwYXRjaCk6CiAgICBmcm9tIHRyYW5zZm9ybWVycy5tb2RlbHMucXdlbjIgaW1wb3J0IG1vZGVsaW5nX3F3ZW4yIGFzIG1vZHVsZQogICAgb3JpZ2luYWwgPSBtb2R1bGUuYXBwbHlfcm90YXJ5X3Bvc19lbWIKICAgIG1vbmtleXBhdGNoLnNldGF0dHIobW9kdWxlLCJhcHBseV9yb3RhcnlfcG9zX2VtYiIsb3JpZ2luYWwpCiAgICBxLGssYyxzID0gaW5wdXRzKCkKICAgIGUgPSBvcmlnaW5hbChxLGssYyxzKQogICAgcm5nID0gdG9yY2guZ2V0X3JuZ19zdGF0ZSgpLmNsb25lKCkKICAgIGFzc2VydCByb3RhcnkucGF0Y2hfcXdlbjJfcm90YXJ5KCkKICAgIHBhdGNoZWQgPSBtb2R1bGUuYXBwbHlfcm90YXJ5X3Bvc19lbWIKICAgIGFzc2VydCByb3RhcnkucGF0Y2hfcXdlbjJfcm90YXJ5KCkKICAgIGFzc2VydCBtb2R1bGUuYXBwbHlfcm90YXJ5X3Bvc19lbWIgaXMgcGF0Y2hlZAogICAgYSA9IHBhdGNoZWQocSxrLGMscykKICAgIGZvciBhYSxlZSBpbiB6aXAoYSxlKTogZXhhY3QoYWEsZWUpCiAgICBhc3NlcnQgdG9yY2guZXF1YWwocm5nLHRvcmNoLmdldF9ybmdfc3RhdGUoKSkKCgpkZWYgdGVzdF9mbWFfYW5kX2Vhcmx5X2Nhc3RfbmVnYXRpdmVfY29udHJvbHMoKToKICAgIGEsYixjLHMgPSBbdG9yY2gudGVuc29yKHgsZHR5cGU9dG9yY2guZmxvYXQzMikgZm9yIHggaW4KICAgICAgICAgICAgICAgKC0zLjA5ODU4MzQ1OTg1NDEyNiwtMi4yOTkyNjcyOTIwMjI3MDUsMC44MjUzMzU2MjE4MzM4MDEzLDAuNTY0NjQyNDg4OTU2NDUxNCldCiAgICBlYWdlciA9IGEqYyArICgtYikqcwogICAgY29udHJhY3RlZCA9IChhLmRvdWJsZSgpKmMuZG91YmxlKCkrKC1iKnMpLmRvdWJsZSgpKS5mbG9hdCgpCiAgICBhc3NlcnQgZWFnZXIudmlldyh0b3JjaC5pbnQzMikgIT0gY29udHJhY3RlZC52aWV3KHRvcmNoLmludDMyKQogICAgYyA9IHMgPSB0b3JjaC5mdWxsKCgxLDEsMiksMioqLTAuNSkKICAgIGcgPSB0b3JjaC56ZXJvcygoMSw2LDEsMiksZHR5cGU9dG9yY2guZmxvYXQxNikKICAgIGdbOiw6Miw6LDBdPTQwMDAwO2dbOiw6Miw6LDFdPS00MDAwMAogICAgZ3JhZGllbnRzPVtdCiAgICBmb3IgZWFybHkgaW4gKEZhbHNlLFRydWUpOgogICAgICAgIGsgPSB0b3JjaC56ZXJvcygoMSwxLDEsMikscmVxdWlyZXNfZ3JhZD1UcnVlKQogICAgICAgIHkgPSByZWZlcmVuY2UoayxrLGMscylbMV0KICAgICAgICB5ID0geS5oYWxmKCkucmVwZWF0X2ludGVybGVhdmUoNiwxKSBpZiBlYXJseSBlbHNlIHkucmVwZWF0X2ludGVybGVhdmUoNiwxKS5oYWxmKCkKICAgICAgICBncmFkaWVudHMuYXBwZW5kKHRvcmNoLmF1dG9ncmFkLmdyYWQoeSxrLGcpWzBdKQogICAgYXNzZXJ0IHRvcmNoLmlzZmluaXRlKGdyYWRpZW50c1swXSkuYWxsKCkKICAgIGFzc2VydCB0b3JjaC5pc25hbihncmFkaWVudHNbMV0pLmFueSgpCgoKQEtFUk5FTApkZWYgdGVzdF9yZWFsX3F3ZW4yX2F0dGVudGlvbl9pbnRlcnByZXRlcl9yb3V0ZShtb25rZXlwYXRjaCk6CiAgICBmcm9tIHRyYW5zZm9ybWVycy5tb2RlbHMucXdlbjIgaW1wb3J0IG1vZGVsaW5nX3F3ZW4yIGFzIG1vZAogICAgZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxzLnF3ZW4yLmNvbmZpZ3VyYXRpb25fcXdlbjIgaW1wb3J0IFF3ZW4yQ29uZmlnCiAgICB0b3JjaC5tYW51YWxfc2VlZCg0NzEzMikKICAgIGNmZz1Rd2VuMkNvbmZpZyhoaWRkZW5fc2l6ZT0xNTM2LG51bV9hdHRlbnRpb25faGVhZHM9MTIsbnVtX2tleV92YWx1ZV9oZWFkcz0yLAogICAgICAgICAgICAgICAgICAgaW50ZXJtZWRpYXRlX3NpemU9MTYsbnVtX2hpZGRlbl9sYXllcnM9MSxhdHRlbnRpb25fZHJvcG91dD0wLiwKICAgICAgICAgICAgICAgICAgIHJvcGVfcGFyYW1ldGVycz17InJvcGVfdHlwZSI6ImRlZmF1bHQiLCJyb3BlX3RoZXRhIjoxXzAwMF8wMDAufSkKICAgIGNmZy5fYXR0bl9pbXBsZW1lbnRhdGlvbj0ic2RwYSIKICAgIG1vZHVsZT1tb2QuUXdlbjJBdHRlbnRpb24oY2ZnLDApLnRyYWluKCkKICAgIGVtYj1tb2QuUXdlbjJSb3RhcnlFbWJlZGRpbmcoY2ZnKQogICAgaGlkZGVuPXRvcmNoLnJhbmRuKCgxLDMsMTUzNikscmVxdWlyZXNfZ3JhZD1UcnVlKQogICAgY29lZmY9ZW1iKGhpZGRlbix0b3JjaC5hcmFuZ2UoMykudW5zcXVlZXplKDApKQogICAgdmFyaWFibGVzPShoaWRkZW4sKm1vZHVsZS5wYXJhbWV0ZXJzKCkpCiAgICBncmFkPXRvcmNoLm9uZXNfbGlrZShoaWRkZW4pKjY0CiAgICBlPW1vZHVsZShoaWRkZW4sY29lZmYsTm9uZSlbMF07ZWc9dG9yY2guYXV0b2dyYWQuZ3JhZChlLHZhcmlhYmxlcyxncmFkKQogICAgb3JpZ2luYWw9bW9kLmFwcGx5X3JvdGFyeV9wb3NfZW1iCiAgICBtb25rZXlwYXRjaC5zZXRhdHRyKG1vZCwiYXBwbHlfcm90YXJ5X3Bvc19lbWIiLG9yaWdpbmFsKQogICAgIyBUZXN0LW9ubHkgcmVwbGFjZSBDVURBIHJlcXVpcmVtZW50OyByZXRhaW4gcHJvZHVjdGlvbiBnZW9tZXRyeS9hbGlhcyBndWFyZHMuCiAgICByZWFzb249cm90YXJ5Ll9zdXBwb3J0X3JlYXNvbgogICAgZGVmIGNwdV9yZWFzb24oKmFyZ3MsKiprd2FyZ3MpOgogICAgICAgIGt3YXJnc1siY3VkYV9vbmx5Il09RmFsc2UKICAgICAgICByZXR1cm4gcmVhc29uKCphcmdzLCoqa3dhcmdzKQogICAgbW9ua2V5cGF0Y2guc2V0YXR0cihyb3RhcnksIl9zdXBwb3J0X3JlYXNvbiIsY3B1X3JlYXNvbikKICAgIGFzc2VydCByb3RhcnkucGF0Y2hfcXdlbjJfcm90YXJ5KCkKICAgIGJlZm9yZT1yb3RhcnkuX0NPVU5UU1siZm9yd2FyZF9sYXVuY2hfcmV0dXJuZWQiXQogICAgYT1tb2R1bGUoaGlkZGVuLGNvZWZmLE5vbmUpWzBdO2FnPXRvcmNoLmF1dG9ncmFkLmdyYWQoYSx2YXJpYWJsZXMsZ3JhZCkKICAgIGFzc2VydCByb3RhcnkuX0NPVU5UU1siZm9yd2FyZF9sYXVuY2hfcmV0dXJuZWQiXT09YmVmb3JlKzIKICAgIGV4YWN0KGEsZSkKICAgIGZvciBhYSxlZSBpbiB6aXAoYWcsZWcpOnRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGFhLGVlLHJ0b2w9MmUtNSxhdG9sPTJlLTUsZXF1YWxfbmFuPVRydWUpCg==', 'old_solution/web_preflight.py': 'IiIiV2ViIEpTT04tZmlsZSBhZGFwdGVyIGZvciB0aGUgdW5jaGFuZ2VkIG9yaWdpbmFsIGRpcmVjdG9yeS1vdXRwdXQgUm9QRSBnYXRlLgoKTm8ga2VybmVsIG9yIGFyaXRobWV0aWMgY2hhbmdlLiBUaGUgb3JpZ2luYWwgdGFyZ2V0IGNoZWNrcyBydW4gc3luY2hyb25vdXNseSBpbgp0aGlzIHNob3J0LWxpdmVkIGNoaWxkOyBhIGZhaWx1cmUsIGluY29tcGxldGUgcmVzdWx0LCBvciB3cm9uZyBzdGF0dXMgZmFpbHMgc2h1dC4KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHN5cwppbXBvcnQgdHJhY2ViYWNrCmltcG9ydCB1dWlkCgoKZGVmIF93cml0ZShwYXRoLCByZXBvcnQpOgogICAgdG1wID0gcGF0aC53aXRoX25hbWUocGF0aC5uYW1lICsgJy4nICsgdXVpZC51dWlkNCgpLmhleCArICcudG1wJykKICAgIHRtcC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LCBpbmRlbnQ9MikgKyAnXG4nKQogICAgdG1wLnJlcGxhY2UocGF0aCkKCgpkZWYgcnVuKGRlc3RpbmF0aW9uKToKICAgIGRlc3RpbmF0aW9uID0gUGF0aChkZXN0aW5hdGlvbikucmVzb2x2ZSgpCiAgICBpZiBkZXN0aW5hdGlvbi5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoZGVzdGluYXRpb24pCiAgICBkZXN0aW5hdGlvbi5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZGV0YWlscyA9IGRlc3RpbmF0aW9uLndpdGhfbmFtZShkZXN0aW5hdGlvbi5uYW1lICsgJy5yb3RhcnlfZGV0YWlscycpCiAgICByZXN1bHQgPSB7J3N0YXR1cyc6ICdGQUlMRUQnLCAnZ3B1X2V4ZWN1dGVkJzogRmFsc2UsICdzY29yZSc6IE5vbmUsCiAgICAgICAgICAgICAgJ2FkbWlzc2lvbic6ICdVTktOT1dOJywgJ29yaWdpbmFsX3ByZWZsaWdodF9kZXRhaWxzJzogc3RyKGRldGFpbHMpfQogICAgX3dyaXRlKGRlc3RpbmF0aW9uLCByZXN1bHQpCiAgICBwcmV2aW91c19hcmd2ID0gc3lzLmFyZ3YKICAgIGV4aXRfY29kZSA9IE5vbmUKICAgIHRyeToKICAgICAgICBmcm9tIC4gaW1wb3J0IGdwdV9wcmVmbGlnaHQKICAgICAgICBzeXMuYXJndiA9IFtncHVfcHJlZmxpZ2h0Ll9fZmlsZV9fLCAnLS1vdXQnLCBzdHIoZGV0YWlscyldCiAgICAgICAgdHJ5OgogICAgICAgICAgICBncHVfcHJlZmxpZ2h0Lm1haW4oKQogICAgICAgICAgICBleGl0X2NvZGUgPSAwCiAgICAgICAgZXhjZXB0IFN5c3RlbUV4aXQgYXMgZXhjOgogICAgICAgICAgICBleGl0X2NvZGUgPSAwIGlmIGV4Yy5jb2RlIGlzIE5vbmUgZWxzZSBleGMuY29kZQogICAgICAgIHJlcG9ydCA9IGpzb24ubG9hZHMoKGRldGFpbHMgLyAncHJlZmxpZ2h0Lmpzb24nKS5yZWFkX3RleHQoKSkKICAgICAgICBpZiBleGl0X2NvZGUgIT0gMCBvciByZXBvcnQuZ2V0KCdzdGF0dXMnKSAhPSAnUEFTU0VEX1BSRUZMSUdIVF9PTkxZJyBvciByZXBvcnQuZ2V0KCdncHVfZXhlY3V0ZWQnKSBpcyBub3QgVHJ1ZToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdPcmlnaW5hbCB0YXJnZXQgcHJlZmxpZ2h0IGZhaWxlZCBvciBkaWQgbm90IGNvbXBsZXRlOiAnICsgcmVwcihleGl0X2NvZGUpKQogICAgICAgIHJlc3VsdCA9IGRpY3QocmVwb3J0LCBvcmlnaW5hbF9wcmVmbGlnaHRfZGV0YWlscz1zdHIoZGV0YWlscyksIG9yaWdpbmFsX2V4aXRfY29kZT1leGl0X2NvZGUsCiAgICAgICAgICAgICAgICAgICAgICBzY29yZT1Ob25lLCBhZG1pc3Npb249J1VOS05PV04nKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHJlc3VsdC51cGRhdGUoc3RhdHVzPSdGQUlMRUQnLCBlcnJvcj1yZXByKGV4YyksIHRyYWNlYmFjaz10cmFjZWJhY2suZm9ybWF0X2V4YygpLAogICAgICAgICAgICAgICAgICAgICAgb3JpZ2luYWxfZXhpdF9jb2RlPWV4aXRfY29kZSkKICAgIGZpbmFsbHk6CiAgICAgICAgc3lzLmFyZ3YgPSBwcmV2aW91c19hcmd2CiAgICAgICAgX3dyaXRlKGRlc3RpbmF0aW9uLCByZXN1bHQpCiAgICByZXR1cm4gMCBpZiByZXN1bHRbJ3N0YXR1cyddID09ICdQQVNTRURfUFJFRkxJR0hUX09OTFknIGVsc2UgMQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLW91dCcsIHJlcXVpcmVkPVRydWUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcmFpc2UgU3lzdGVtRXhpdChydW4oYXJncy5vdXQpKQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtYWluKCkK', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'runtime_probe.py': 'IiIiUmVhZC1vbmx5LCBzaG9ydC1saXZlZCBkZXBsb3llZC10ZW1wbGF0ZSBnYXRlLiBOZXZlciBwcmludHMgc2VjcmV0IHZhbHVlcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YSxqc29uLG9zLHBsYXRmb3JtLHN1YnByb2Nlc3Msc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApFWFBFQ1RFRD17J3RvcmNoJzonMi4xMS4wK2N1MTI4JywndHJpdG9uJzonMy42LjAnLCdudW1weSc6JzIuMy41Jywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQoKZGVmIG1haW4oKToKIG91dD1QYXRoKHN5cy5hcmd2WzFdKTtyPXsncGFzc2VkJzpGYWxzZSwnZ3B1X3dvcmtsb2FkX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogdHJ5OgogIHZlcnNpb25zPXtrOm1ldGFkYXRhLnZlcnNpb24oaykgZm9yIGsgaW4gRVhQRUNURUR9CiAgci51cGRhdGUocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCkscGFja2FnZXM9dmVyc2lvbnMsdWlkPW9zLmdldHVpZCgpLGNwdV9hZmZpbml0eT1zb3J0ZWQob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpKQogIGFzc2VydCBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpPT0nMy4xMy4xNScsJ1B5dGhvbiBwaW4gbWlzbWF0Y2gnCiAgYXNzZXJ0IHZlcnNpb25zPT1FWFBFQ1RFRCwnRXhhY3QgcGFja2FnZSBwaW5zIG1pc21hdGNoJwogIGFzc2VydCBvcy5nZXR1aWQoKT09MTAwMDEsJ0RlcGxveWVkIHJ1bnRpbWUgdWlkIG1pc21hdGNoJwogIGFzc2VydCBsZW4ob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpPT00LCdGb3VyIGFzc2lnbmVkIENQVXMgcmVxdWlyZWQnCiAgZm9yIHAsd2FudGVkIGluIFsoJ21lbW9yeS5tYXgnLHN0cigzMCoxMDI0KiozKSksKCdtZW1vcnkuc3dhcC5tYXgnLCcwJyksKCdjcHVzZXQuY3B1cy5lZmZlY3RpdmUnLCcwLTMnKV06CiAgIGFjdHVhbD0oUGF0aCgnL3N5cy9mcy9jZ3JvdXAnKS9wKS5yZWFkX3RleHQoKS5zdHJpcCgpO3JbcF09YWN0dWFsCiAgIGFzc2VydCBhY3R1YWw9PXdhbnRlZCwnRGVwbG95ZWQgY2dyb3VwIG1pc21hdGNoOiAnK3AKICBhc3NlcnQgbm90IFBhdGgoJy92YXIvcnVuL2RvY2tlci5zb2NrJykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgRG9ja2VyIHNvY2tldCcKICBhc3NlcnQgbm90IFBhdGgoJy9ldGMvc291cC1xdWV1ZS93b3JrZXIuZW52JykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgd29ya2VyIGVudmlyb25tZW50IGZpbGUnCiAgYXNzZXJ0IG5vdCBhbnkoJ1RPS0VOJyBpbiBrIGFuZCBrIT0nVE9LRU5JWkVSU19QQVJBTExFTElTTScgZm9yIGsgaW4gb3MuZW52aXJvbiksJ1Rva2VuLWJlYXJpbmcgZW52aXJvbm1lbnQgcmVqZWN0ZWQgd2l0aG91dCByZWFkaW5nIGl0cyB2YWx1ZScKICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyksJ0ludGVycHJldGVyIG1vZGUgZm9yYmlkZGVuJwogIGltcG9ydCB0b3JjaCx0cml0b24sbnVtcHkKICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnIGFuZCB0b3JjaC52ZXJzaW9uLmN1ZGE9PScxMi44JwogIGFzc2VydCB0cml0b24uX192ZXJzaW9uX189PSczLjYuMCcgYW5kIG51bXB5Ll9fdmVyc2lvbl9fPT0nMi4zLjUnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogIHIudXBkYXRlKGdwdV9uYW1lPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLGN1ZGFfYnVpbGQ9dG9yY2gudmVyc2lvbi5jdWRhLHRvcmNoX2dpdD10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uKQogIGZyb20gaHVnZ2luZ2ZhY2VfaHViLmNvbnN0YW50cyBpbXBvcnQgSEZfSFVCX0NBQ0hFCiAgclsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddPXN0cihQYXRoKEhGX0hVQl9DQUNIRSkucmVzb2x2ZSgpKQogIHJbJ3Bhc3NlZCddPVRydWUKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6clsnZXJyb3InXT1yZXByKGUpCiBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHIsaW5kZW50PTIpKydcbicpO3ByaW50KGpzb24uZHVtcHMocixpbmRlbnQ9MikpCiByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgclsncGFzc2VkJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'official_manifest.json': 'ewogICJoYXJuZXNzL190cmFpbi5weSI6ICI0NWE2MjY0MmRjZWFiMzJjZjNlZmNkYTUwNGRlODdhOWMyNWVhMjhiMTE5YjE4MDZmNTZkMTRmZjExNDZhODlkIiwKICAiaGFybmVzcy9jaGVja19ydWxlcy5weSI6ICI4NzlhYjE4NjE2NzRmNmU3MTRlYWRkMDBhYWYwNDQyMzVkOTg1MjZkZDc1ZmFiMTAzNWRlZWMzNDAxMTIxZWY1IiwKICAiaGFybmVzcy9jb21tb24ucHkiOiAiYzgzMDY2ZjI0ZjI2NmRhYjYyZWJjYzljMmQ0NGY1YTY1OTAyNzYwYjUzZGE5MTJlNjVlYmZlNTExNTc5YzJkYiIsCiAgImhhcm5lc3MvZXZhbF9hZGFwdGVyLnB5IjogIjQ3Y2RkYmI4ZTQ1NzljNjlmOTUzNWIwOTlmZjIzNzFiMWMzYWYwOGJmYmQ5M2QxNGJmNzlhYWRkNzRkMzg0ODIiLAogICJoYXJuZXNzL2V4YW1wbGVfcmVmZXJlbmNlLnB5IjogIjZhMTEwNTVmNGExODIyZTUzOTcxYzIyOGM2ZTNlM2FhNmI4OGVkZDllYjZkYjgxYTM4OGUyNmE3MGEwMzBjZWIiLAogICJoYXJuZXNzL3J1bl90YXNrLnB5IjogImQ5MzVhODk1OWYzNDg5NzczMTNmMmVlZjM2ODcwZmYyNjk5YjliYTAxM2E3OTc4OTE4ZmYzNWUxNDg5NjhhNzkiLAogICJoYXJuZXNzL3Njb3JlLnB5IjogIjczMWM5YjY0NGRjYzQwNjliMGIwNjk2MTZmNDM4MDY0ZDFmN2YzNzZhMjAyZmViMTUyMDFmZDkxMGVjNjg4YzUiLAogICJ0YXNrL3ByZXBhcmVfZGF0YS5weSI6ICJhODU0M2YxMWE1NWFmZjgwZGJkYzcxNGEwZTYyMzkxMjQ0YTUyOTY2MTZlZWEyZTM0ZGMyZWIzZmYzZWMxOGY3IiwKICAidGFzay90YXNrLnlhbWwiOiAiZGQ3MGM3MzlmY2QxNDNlMDZhZWYzYmIxZWZjYzkzODZmNjgyYmYxZGUyNmYzMGUwODQwOWQ4NmE5N2ZmNDdjZSIsCiAgInRlc3RzL2NvbmZ0ZXN0LnB5IjogImE4MmVhYmFiNDY4ZWEwZWYxZTAwM2NjY2Q3YzM4MjdjZTJmNGQwMGQwOWMzMjk0OGEzNGZiNDgyMjFmNzMzN2EiLAogICJ0ZXN0cy90ZXN0X2NhbGxiYWNrLnB5IjogIjE5MTkzMDNhMzY3MGMwYTMzYWJiZmNlOWIzM2VhYjA5MTNiYjY0YTNhMjFjMzMxMmQ4NjUzMWMzMmI1YmY0OTAiLAogICJ0ZXN0cy90ZXN0X2NvbmZpZy5weSI6ICJkYjgwOGM0NDQ3MGQxNDRiN2NmMThiNzMyOTQ5MmY0MWY3NmUzN2NmYzI0MzA2ZTdjYmEwNDg4YmNhOWQ3ODdjIiwKICAidGVzdHMvdGVzdF9mb3JtYXRzLnB5IjogIjVkNjU2ZWQ1MGM1OWY3OTRlOTM0NDczZDQ1OWVhODU2MzVmODkyYmEwMjhlMTkyNDE0NDc0ZTliNWQ2NTkxOWUiLAogICJ0ZXN0cy90ZXN0X2xvYWRlci5weSI6ICIzZGFkMWJlYTJkMzNhYWI5N2U1MmI3Y2Q0NDg4OGQ3NjhiYzk1MTlhZjQ0ZmJlNmNmM2U5NmMyNjAwMjY2MzJmIiwKICAidGVzdHMvdGVzdF90cmFpbmVyX2luaXQucHkiOiAiYjc2MDhkNjJhNDA5ODNiZjdjOWU1YjI3MGUwZTUyN2I2NDFhODM3ZTQ2MmQ2ZGYyOTA5ZmVjODhmNjA3ZTgyMSIKfQo=', 'reviewed_support.py': 'aW1wb3J0IGFyZ3BhcnNlLGZjbnRsLGhhc2hsaWIsanNvbixtYXRoLG9zLHNodXRpbCxzeXMsdGhyZWFkaW5nLHRpbWUsdXVpZCx6aXBmaWxlCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwcm9jZXNzX2d1YXJkCgppbXBvcnQgb3duZWRfcHJvY2Vzc190cmVlIGFzIHRyZWUKCmRlZiBzaGEocCk6cmV0dXJuIGhhc2hsaWIuc2hhMjU2KFBhdGgocCkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQoKZGVmIHJlYWQocCk6cmV0dXJuIGpzb24ubG9hZHMoUGF0aChwKS5yZWFkX3RleHQoKSkKCmRlZiB3cml0ZShwLHZhbHVlKToKIHA9UGF0aChwKTtwZW5kaW5nPXAud2l0aF9uYW1lKHAubmFtZSsnLnBlbmRpbmcnKTtwZW5kaW5nLndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpO3BlbmRpbmcucmVwbGFjZShwKQoKZGVmIHZlcmlmeV9vZmZpY2lhbChyZXBvLG1hbmlmZXN0KToKIGlmIG5vdCByZXBvLmlzX2RpcigpIG9yIHJlcG8uaXNfc3ltbGluaygpOnJhaXNlIFJ1bnRpbWVFcnJvcignU09VUF9SRVBPIG11c3QgYmUgYSByZWFsIHByZXBhcmVkIGpvYiBkaXJlY3RvcnknKQogZm9yIG5hbWUsd2FudGVkIGluIG1hbmlmZXN0Lml0ZW1zKCk6CiAgcD1yZXBvL25hbWUKICBpZiBub3QgcC5pc19maWxlKCkgb3IgcC5pc19zeW1saW5rKCkgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgUnVudGltZUVycm9yKCdQcm90ZWN0ZWQgb2ZmaWNpYWwgc291cmNlIG1pc21hdGNoOiAnK25hbWUpCiBhY3R1YWw9e3N0cihwLnJlbGF0aXZlX3RvKHJlcG8pKSBmb3IgZm9sZGVyIGluICgnaGFybmVzcycsJ3Rhc2snLCd0ZXN0cycpIGZvciBwIGluIChyZXBvL2ZvbGRlcikucmdsb2IoJyonKQogICAgICAgICBpZiBwLmlzX2ZpbGUoKSBhbmQgJ19fcHljYWNoZV9fJyBub3QgaW4gcC5wYXJ0cyBhbmQgcC5zdWZmaXggbm90IGluICgnLnB5YycsJy5weW8nKX0KIGlmIGFjdHVhbCE9c2V0KG1hbmlmZXN0KTpyYWlzZSBSdW50aW1lRXJyb3IoJ1VuZXhwZWN0ZWQgcHJvdGVjdGVkIG9mZmljaWFsIGZpbGVzJykKCkVYUE9SVF9CVURHRVQ9NTYqMTAyNCoqMgoKRVhQT1JUX09NSVRURUQ9e30KCmRlZiBleHBvcnRfc21hbGwocnVuLGV4cG9ydCk6CiAiIiJSZWFkLW9ubHkgaW5jcmVtZW50YWwgZXhwb3J0OyBubyBhZGFwdGVycywgd2VpZ2h0cyBvciBjb21waWxlciBjYWNoZXMuIiIiCiB1c2VkPXN1bShwLnN0YXQoKS5zdF9zaXplIGZvciBwIGluIGV4cG9ydC5yZ2xvYignKicpIGlmIHAuaXNfZmlsZSgpKQogZXhjbHVkZWQ9eydhZGFwdGVyJywndHJpdG9uX2NhY2hlJywnaW5kdWN0b3JfY2FjaGUnLCdwcmVmbGlnaHRfdHJpdG9uJywncHJlZmxpZ2h0X2luZHVjdG9yJywnb3JpZ2luYWxfc29sdXRpb24nfQogZm9yIGJhc2UsZGlycyxmaWxlcyBpbiBvcy53YWxrKHJ1bik6CiAgZGlyc1s6XT1bZCBmb3IgZCBpbiBkaXJzIGlmIGQgbm90IGluIGV4Y2x1ZGVkIGFuZCBub3QgZC5zdGFydHN3aXRoKCdjaGVja3BvaW50LScpIGFuZCBub3QgKFBhdGgoYmFzZSkvZCkuaXNfc3ltbGluaygpXQogIGZvciBuYW1lIGluIGZpbGVzOgogICBwYXRoPVBhdGgoYmFzZSkvbmFtZTtyZWw9cGF0aC5yZWxhdGl2ZV90byhydW4pCiAgIHRyeToKICAgIGlmIG5vdCBwYXRoLmlzX3N5bWxpbmsoKSBhbmQgcGF0aC5zdWZmaXggaW4gKCcuanNvbicsJy5qc29ubCcsJy5sb2cnLCcueWFtbCcpOgogICAgIGlmIHBhdGguc3RhdCgpLnN0X3NpemU+PTQqMTAyNCoqMjoKICAgICAgRVhQT1JUX09NSVRURURbc3RyKHJlbCldPSdwZXJfZmlsZV9saW1pdF80TWlCJztjb250aW51ZQogICAgIGRlc3Q9ZXhwb3J0LydyYXcnL3JlbDtkZXN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICBwcmlvcj1kZXN0LnN0YXQoKS5zdF9zaXplIGlmIGRlc3QuZXhpc3RzKCkgZWxzZSAwCiAgICAgc2l6ZT1wYXRoLnN0YXQoKS5zdF9zaXplCiAgICAgaWYgdXNlZC1wcmlvcitzaXplPkVYUE9SVF9CVURHRVQ6CiAgICAgIEVYUE9SVF9PTUlUVEVEW3N0cihyZWwpXT0nYWdncmVnYXRlX2J1ZGdldF81Nk1pQic7Y29udGludWUKICAgICB0ZW1wPWRlc3Qud2l0aF9uYW1lKGRlc3QubmFtZSsnLmNvcHlpbmcnKTtzaHV0aWwuY29weWZpbGUocGF0aCx0ZW1wKTt0ZW1wLnJlcGxhY2UoZGVzdCk7dXNlZD11c2VkLXByaW9yK3NpemUKICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvcixPU0Vycm9yKTpwYXNzCgpjbGFzcyBFeHBvcnRlcjoKIGRlZiBfX2luaXRfXyhzZWxmLHJ1bixleHBvcnQpOnNlbGYucnVuPXJ1bjtzZWxmLmV4cG9ydD1leHBvcnQ7c2VsZi5zdG9wPXRocmVhZGluZy5FdmVudCgpO3NlbGYudGhyZWFkPU5vbmUKIGRlZiBfX2VudGVyX18oc2VsZik6CiAgZGVmIGxvb3AoKToKICAgd2hpbGUgbm90IHNlbGYuc3RvcC53YWl0KDEpOmV4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKICBzZWxmLnRocmVhZD10aHJlYWRpbmcuVGhyZWFkKHRhcmdldD1sb29wLGRhZW1vbj1UcnVlKTtzZWxmLnRocmVhZC5zdGFydCgpO3JldHVybiBzZWxmCiBkZWYgX19leGl0X18oc2VsZiwqZXhjKTpzZWxmLnN0b3Auc2V0KCk7c2VsZi50aHJlYWQuam9pbigpO2V4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKCmRlZiBpZGxlX2NoZWNrKGVudixyZXBvLHJ1bixsb2NrZmQpOgogY29kZT0nJydpbXBvcnQganNvbixzdWJwcm9jZXNzLHN5cwpwPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktY29tcHV0ZS1hcHBzPXBpZCx1c2VkX2dwdV9tZW1vcnknLCctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlLGNoZWNrPVRydWUpCmFzc2VydCBub3QgcC5zdGRvdXQuc3RyaXAoKSwnR1BVIHByb2Nlc3NlcyBzdGlsbCBhY3RpdmUgYWZ0ZXIgcHJvYmUvcHJlZmxpZ2h0JwpyPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktZ3B1PWluZGV4LG1lbW9yeS51c2VkJywnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSxjYXB0dXJlX291dHB1dD1UcnVlLHRleHQ9VHJ1ZSxjaGVjaz1UcnVlKQpyb3dzPVtsaW5lLnNwbGl0KCcsJykgZm9yIGxpbmUgaW4gci5zdGRvdXQuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KYXNzZXJ0IHJvd3MgYW5kIHN1bShpbnQoeFsxXSkgZm9yIHggaW4gcm93cyk8MTAwMCwnR1BVIHBoeXNpY2FsIGludmVudG9yeSBub3QgaWRsZScKcHJpbnQoanNvbi5kdW1wcyh7J2lkbGUnOlRydWUsJ3BoeXNpY2FsX21lbW9yeV9yb3dzJzpyb3dzfSkpJycnCiByZXR1cm4gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctYycsY29kZV0sY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydpZGxlLmxvZycsdGltZW91dD0yMCxsb2NrX2ZkPWxvY2tmZCkK', 'worker.py': 'IiIiTUlDUk9CRU5DSF9PTkxZLiBObyBtb2RlbC90cmFpbmVyL2hhcm5lc3Mvb3B0aW1pemVyLCBuZXR3b3JrIG9yIGluc3RhbGxhdGlvbnMuCgpDb2xkIGZpcnN0LWNhbGwgd2FsbCBpbmNsdWRlcyBjb21waWxhdGlvbiB3aGVyZSBuZWVkZWQsIGRpc3BhdGNoLCBhbGxvY2F0aW9ucywKZXhlY3V0aW9uIGFuZCBzeW5jaHJvbml6YXRpb246IE5PVCBhIHB1cmUgY29tcGlsYXRpb24gdGltZXIuIFdhcm0gQ1VEQS1ldmVudAppbnRlcnZhbHMgaW5jbHVkZSBzdHJlYW0gaWRsZSBnYXBzIGJldHdlZW4gUHl0aG9uIGxhdW5jaGVzLCBub3QgcHJvZmlsZXIga2VybmVsCnN1bXMuIEV4YWN0bHkgdGhlIG9yaWdpbmFsIG5hdGl2ZS9jYW5kaWRhdGUgcHVibGljIHJvdGFyeSB3cmFwcGVyIGlzIHRpbWVkLgoiIiIKaW1wb3J0IGFyZ3BhcnNlLCBoYXNobGliLCBpbnNwZWN0LCBqc29uLCBvcywgc3RhdGlzdGljcywgc3lzLCB0aW1lLCB0cmFjZWJhY2sKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpUUyA9ICgxMzMsIDIzNSwgMzE3LCA0ODUsIDUxMikKV0FSTVVQID0gMwpHUk9VUFMgPSA1ClJFUEVUSVRJT05TID0gOAoKZGVmIHdyaXRlKHBhdGgsIHZhbHVlKToKICAgIHRlbXAgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAnLnBlbmRpbmcnKQogICAgdGVtcC53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yLCBhbGxvd19uYW49RmFsc2UpICsgJ1xuJykKICAgIHRlbXAucmVwbGFjZShwYXRoKQoKZGVmIG1haW4oKToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tbWV0aG9kJywgY2hvaWNlcz0oJ25hdGl2ZScsICdvbGQnLCAncnVudGltZScpLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1vdXQnLCByZXF1aXJlZD1UcnVlKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIG91dCA9IFBhdGgoYXJncy5vdXQpLnJlc29sdmUoKQogICAgaWYgb3V0LmV4aXN0cygpOiByYWlzZSBGaWxlRXhpc3RzRXJyb3Iob3V0KQogICAgcmVwb3J0ID0gZGljdChzdGF0dXM9J0ZBSUxFRCcsIHNjb3BlPSdNSUNST0JFTkNIX09OTFknLCBtZXRob2Q9YXJncy5tZXRob2QsCiAgICAgICAgICAgICAgICAgIGdwdV9leGVjdXRlZD1GYWxzZSwgdHJhaW5pbmdfZXhlY3V0ZWQ9RmFsc2UsIGNvbXBsZXRlX2ZpeGVkX3Rhc2s9RmFsc2UsCiAgICAgICAgICAgICAgICAgIG1ldGhvZF92YXJpYW50cz17J25hdGl2ZSc6J25hdGl2ZV9xd2VuMicsJ29sZCc6J3J1bnRpbWVfVF9zdHJpZGVzXzFkJywncnVudGltZSc6J3J1bnRpbWVfVF9zdHJpZGVzXzJkX2dyaWQnfSwKICAgICAgICAgICAgICAgICAgc2NvcmU9Tm9uZSwgYWRtaXNzaW9uPSdVTktOT1dOJywgcm93cz1bXSwKICAgICAgICAgICAgICAgICAgbWVhc3VyZW1lbnRfY29udHJhY3Q9ewogICAgICAgICAgICAgICAgICAgICdjb2xkJzonc3luY2hyb25pemVkIGZpcnN0IGNhbGwgcGVyIHNoYXBlIGluIHRoaXMgZnJlc2ggcHJvY2VzcyBhbmQgcHJpdmF0ZSBlbXB0eSBUcml0b24gY2FjaGU7IGluY2x1ZGVzIGNvbXBpbGUgd2hlbiBuZWVkZWQsIGFsbG9jYXRpb25zLCBkaXNwYXRjaCBhbmQgZGV2aWNlIHdvcmsnLAogICAgICAgICAgICAgICAgICAgICd3YXJtJzonQ1VEQS1ldmVudCBzdHJlYW0gaW50ZXJ2YWwgcGVyIHJlcGVhdGVkIHdyYXBwZXIgY2FsbCwgd2l0aCBQeXRob24gbGF1bmNoIGdhcHM7IG5vIHB1cmUta2VybmVsIG9yIHRyYWluaW5nLXdhbGwgY2xhaW0nLAogICAgICAgICAgICAgICAgICAgICdjYWNoZSc6J25vdCByZXNldCBiZXR3ZWVuIHNoYXBlczsgcnVudGltZSBrZXlzIGFyZSBpbnRlbmRlZCB0byBiZSByZXVzZWQnLAogICAgICAgICAgICAgICAgICAgICdvcmRlcic6J3NpbmdsZSBmaXhlZCBuYXRpdmUvb2xkL3J1bnRpbWUgcHJvY2VzcyBvcmRlcjsgbm8gc3RhdGlzdGljYWwgY291bnRlcmJhbGFuY2VkIHBlcmZvcm1hbmNlIGNsYWltJywKICAgICAgICAgICAgICAgICAgICAnZml4ZWRfcmVwZXRpdGlvbnMnOlJFUEVUSVRJT05TLCdncm91cHMnOkdST1VQUywnd2FybXVwX2NhbGxzJzpXQVJNVVAsCiAgICAgICAgICAgICAgICAgICAgJ2NvZWZmaWNpZW50X2JhdGNoJzoxLCdjb2VmZmljaWVudF9iYXRjaF9pc19vYnNlcnZlZCc6RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgJ3NoYXBlX3Byb3ZlbmFuY2UnOidUIHZhbHVlcyBvYnNlcnZlZCBpbiBvcmlnaW5hbCBzb3VyY2UtYm91bmQgdHJhaW5pbmcgbG9nOyBwYWlyZWQgUS9LIHN5bnRoZXRpYyBmaXh0dXJlcyB1c2UgdGFzayBIMTIvSDIgZ2VvbWV0cnknLAogICAgICAgICAgICAgICAgICAgICdiYWNrd2FyZF9sYXlvdXQnOidRIHByb2plY3Rpb24tb3JkZXIgYW5kIEsgY29udGlndW91cywgYm90aCBsYXlvdXQgZmFtaWxpZXMgb2JzZXJ2ZWQgaW4gb3JpZ2luYWwgam9iOyBubyBjbGFpbSBldmVyeSBUL2xheW91dCBwYWlyIHdhcyByZWNvcmRlZCcsCiAgICAgICAgICAgICAgICAgICAgJ3RlbGVtZXRyeSc6J29yaWdpbmFsIGNhbmRpZGF0ZSB0ZWxlbWV0cnkgcmVtYWlucyBlbmFibGVkIGFuZCBpdHMgaG9zdCBvdmVyaGVhZCBpcyBpbmNsdWRlZCB3aGVyZSBpdCBvY2N1cnMnfSkKICAgIHdyaXRlKG91dCwgcmVwb3J0KQogICAgdHJ5OgogICAgICAgIGFzc2VydCBub3Qgb3MuZW52aXJvbi5nZXQoJ1RSSVRPTl9JTlRFUlBSRVQnKSwgJ0ludGVycHJldGVyIGZvcmJpZGRlbiBmb3IgR1BVIG1pY3JvYmVuY2htYXJrJwogICAgICAgIGNhY2hlID0gUGF0aChvcy5lbnZpcm9uWydUUklUT05fQ0FDSEVfRElSJ10pCiAgICAgICAgYXNzZXJ0IGNhY2hlLmlzX2RpcigpIGFuZCBub3QgYW55KGNhY2hlLml0ZXJkaXIoKSksICdQcml2YXRlIGVtcHR5IGNhY2hlIHJlcXVpcmVkIGJlZm9yZSBpbXBvcnRpbmcgVHJpdG9uJwogICAgICAgIGltcG9ydCB0b3JjaCwgdHJpdG9uLCB0cmFuc2Zvcm1lcnMKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycy5tb2RlbHMucXdlbjIgaW1wb3J0IG1vZGVsaW5nX3F3ZW4yIGFzIG5hdGl2ZV9tb2R1bGUKICAgICAgICBmcm9tIHNvbHV0aW9uLmNwdV9jb250cmFjdCBpbXBvcnQgaW5wdXRzLCBleGFjdAogICAgICAgIGZyb20gc29sdXRpb24gaW1wb3J0IHJvdGFyeSBhcyBydW50aW1lX21vZHVsZQogICAgICAgIGZyb20gb2xkX3NvbHV0aW9uIGltcG9ydCByb3RhcnkgYXMgb2xkX21vZHVsZQogICAgICAgIGFzc2VydCB0b3JjaC5fX3ZlcnNpb25fXyA9PSAnMi4xMS4wK2N1MTI4JyBhbmQgdG9yY2gudmVyc2lvbi5jdWRhID09ICcxMi44JwogICAgICAgIGFzc2VydCB0cml0b24uX192ZXJzaW9uX18gPT0gJzMuNi4wJyBhbmQgdHJhbnNmb3JtZXJzLl9fdmVyc2lvbl9fID09ICc1LjE2LjEnCiAgICAgICAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkgPT0gMSBhbmQgdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkgPT0gJ1Rlc2xhIFQ0JwogICAgICAgIGFzc2VydCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eSgwKSA9PSAoNyw1KQogICAgICAgIHJhdyA9IFBhdGgoaW5zcGVjdC5nZXRmaWxlKG5hdGl2ZV9tb2R1bGUpKS5yZWFkX2J5dGVzKCkKICAgICAgICBibG9iID0gaGFzaGxpYi5zaGExKGInYmxvYiAnICsgc3RyKGxlbihyYXcpKS5lbmNvZGUoKSArIGInXDAnICsgcmF3KS5oZXhkaWdlc3QoKQogICAgICAgIGFzc2VydCBibG9iID09IHJ1bnRpbWVfbW9kdWxlLkVYUEVDVEVEX1FXRU5fQkxPQiA9PSBvbGRfbW9kdWxlLkVYUEVDVEVEX1FXRU5fQkxPQgogICAgICAgIG5hdGl2ZSA9IG5hdGl2ZV9tb2R1bGUuYXBwbHlfcm90YXJ5X3Bvc19lbWIKICAgICAgICBhc3NlcnQgbm90IGdldGF0dHIobmF0aXZlLCAnX29yaWdpbmFsX3JvdGFyeV9jYW5kaWRhdGUnLCBGYWxzZSksICdOYXRpdmUgY2FsbGFibGUgYWxyZWFkeSBwYXRjaGVkJwogICAgICAgIG1vZHVsZSA9IHsnb2xkJzpvbGRfbW9kdWxlLCAncnVudGltZSc6cnVudGltZV9tb2R1bGV9LmdldChhcmdzLm1ldGhvZCkKICAgICAgICBmbiA9IG5hdGl2ZSBpZiBtb2R1bGUgaXMgTm9uZSBlbHNlIG1vZHVsZS5mdXNlZF9hcHBseV9yb3RhcnlfcG9zX2VtYgogICAgICAgIGRlZiBrZXlfY291bnRzKCk6CiAgICAgICAgICAgIGlmIG1vZHVsZSBpcyBOb25lOiByZXR1cm4gTm9uZQogICAgICAgICAgICByZXR1cm4ge25hbWU6c3VtKGxlbihlbnRyeVswXSkgZm9yIGVudHJ5IGluIGdldGF0dHIobW9kdWxlLG5hbWUpLmRldmljZV9jYWNoZXMudmFsdWVzKCkpCiAgICAgICAgICAgICAgICAgICAgZm9yIG5hbWUgaW4gKCdyb3RhcnlfYXBwbHlfZjMyJywncm90YXJ5X2Fkam9pbnRfZjMyJyl9CiAgICAgICAgZGVmIHdhbGwoY2FsbCk6CiAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgICAgICAgICBzdGFydCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICAgICAgdmFsdWUgPSBjYWxsKCkKICAgICAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCiAgICAgICAgICAgIHJldHVybiB2YWx1ZSwgdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0CiAgICAgICAgZGVmIHdhcm0oY2FsbCk6CiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKFdBUk1VUCk6IGNhbGwoKQogICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgICAgICAgICAgc2FtcGxlcyA9IFtdCiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKEdST1VQUyk6CiAgICAgICAgICAgICAgICBiZWdpbiwgZW5kID0gdG9yY2guY3VkYS5FdmVudChlbmFibGVfdGltaW5nPVRydWUpLCB0b3JjaC5jdWRhLkV2ZW50KGVuYWJsZV90aW1pbmc9VHJ1ZSkKICAgICAgICAgICAgICAgIGJlZ2luLnJlY29yZCgpCiAgICAgICAgICAgICAgICBmb3IgXyBpbiByYW5nZShSRVBFVElUSU9OUyk6IGNhbGwoKQogICAgICAgICAgICAgICAgZW5kLnJlY29yZCgpOyBlbmQuc3luY2hyb25pemUoKQogICAgICAgICAgICAgICAgc2FtcGxlcy5hcHBlbmQoYmVnaW4uZWxhcHNlZF90aW1lKGVuZCkgLyBSRVBFVElUSU9OUykKICAgICAgICAgICAgYXNzZXJ0IGFsbCh2ID4gMCBmb3IgdiBpbiBzYW1wbGVzKQogICAgICAgICAgICByZXR1cm4gZGljdChjdWRhX2V2ZW50X21zX3Blcl9jYWxsPXNhbXBsZXMsIG1lZGlhbl9tcz1zdGF0aXN0aWNzLm1lZGlhbihzYW1wbGVzKSwKICAgICAgICAgICAgICAgICAgICAgICAgbWluX21zPW1pbihzYW1wbGVzKSwgbWF4X21zPW1heChzYW1wbGVzKSkKICAgICAgICAjIEluaXRpYWxpemUgQ1VEQSBjb250ZXh0IGFuZCBhIHRpbnkgdW5yZWxhdGVkIGVhZ2VyIG9wZXJhdGlvbiBiZWZvcmUgdGltaW5nLgogICAgICAgICMgTm8gUm9QRSBjYW5kaWRhdGUga2VybmVsIGFuZCBubyByZXByZXNlbnRhdGl2ZSBvcGVyYXRpb24gaXMgcHJld2FybWVkLgogICAgICAgIHNjcmF0Y2ggPSB0b3JjaC5vbmVzKDE2LCBkZXZpY2U9J2N1ZGEnKTsgc2NyYXRjaC5hZGRfKDEpOyB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7IGRlbCBzY3JhdGNoCiAgICAgICAgcmVwb3J0WydncHVfZXhlY3V0ZWQnXT1UcnVlOyB3cml0ZShvdXQscmVwb3J0KQogICAgICAgIGZvciB0IGluIFRTOgogICAgICAgICAgICBxLGssYyxzID0gaW5wdXRzKGI9NCx0PXQsY2I9MSxkZXZpY2U9J2N1ZGEnKQogICAgICAgICAgICBncyA9ICh0b3JjaC5lbXB0eV9zdHJpZGVkKHEuc2hhcGUscS5zdHJpZGUoKSxkZXZpY2U9J2N1ZGEnKS5maWxsXyg2NTUzNiksCiAgICAgICAgICAgICAgICAgIHRvcmNoLmZ1bGwoay5zaGFwZSwtNjU1MzYuLGRldmljZT0nY3VkYScpKQogICAgICAgICAgICBhc3NlcnQgcS5zdHJpZGUoKSA9PSAodCoxNTM2LDEyOCwxNTM2LDEpCiAgICAgICAgICAgIGFzc2VydCBrLnN0cmlkZSgpID09ICh0KjI1NiwxMjgsMjU2LDEpCiAgICAgICAgICAgIGFzc2VydCBnc1swXS5zdHJpZGUoKSA9PSBxLnN0cmlkZSgpIGFuZCBnc1sxXS5zdHJpZGUoKSA9PSAodCoyNTYsdCoxMjgsMTI4LDEpCiAgICAgICAgICAgIHZlcnNpb25zID0gW3guX3ZlcnNpb24gZm9yIHggaW4gKHEsayxjLHMpXQogICAgICAgICAgICBzbmFwc2hvdHMgPSBbeC5kZXRhY2goKS5jbG9uZSgpIGZvciB4IGluIChxLGssYyxzKV0KICAgICAgICAgICAgcm5nID0gdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkuY2xvbmUoKQogICAgICAgICAgICBiZWZvcmUgPSBrZXlfY291bnRzKCkKICAgICAgICAgICAgeSwgZm9yd2FyZF9zID0gd2FsbChsYW1iZGE6IGZuKHEsayxjLHMpKQogICAgICAgICAgICBhZnRlcl9mb3J3YXJkID0ga2V5X2NvdW50cygpCiAgICAgICAgICAgIGJhY2t3YXJkID0gbGFtYmRhOiB0b3JjaC5hdXRvZ3JhZC5ncmFkKHksKHEsayksZ3MscmV0YWluX2dyYXBoPVRydWUpCiAgICAgICAgICAgIGR5LCBiYWNrd2FyZF9zID0gd2FsbChiYWNrd2FyZCkKICAgICAgICAgICAgYWZ0ZXJfYmFja3dhcmQgPSBrZXlfY291bnRzKCkKICAgICAgICAgICAgIyBHYXRlcyBvY2N1ciBhZnRlciB0aGlzIHNoYXBlJ3MgZmlyc3QtY2FsbCB0aW1pbmcsIGJlZm9yZSBhbnkgd2FybSB0aW1pbmcuCiAgICAgICAgICAgIGV4cGVjdGVkID0gbmF0aXZlKHEsayxjLHMpCiAgICAgICAgICAgIGV4cGVjdGVkX2dyYWQgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKGV4cGVjdGVkLChxLGspLGdzKQogICAgICAgICAgICBmb3IgYWN0dWFsLCB3YW50ZWQgaW4gemlwKCgqeSwqZHkpLCgqZXhwZWN0ZWQsKmV4cGVjdGVkX2dyYWQpKTogZXhhY3QoYWN0dWFsLHdhbnRlZCkKICAgICAgICAgICAgZm9yIGFjdHVhbCwgc291cmNlIGluIHppcCh5LChxLGspKToKICAgICAgICAgICAgICAgIGFzc2VydCBhY3R1YWwuZGF0YV9wdHIoKSAhPSBzb3VyY2UuZGF0YV9wdHIoKQogICAgICAgICAgICBpZiBtb2R1bGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBmb3IgYWN0dWFsLCBzb3VyY2UgaW4gemlwKHksKHEsaykpOiBhc3NlcnQgYWN0dWFsLnN0cmlkZSgpID09IHNvdXJjZS5zdHJpZGUoKQogICAgICAgICAgICBmb3IgdGVuc29yLCB2ZXJzaW9uLCBzbmFwIGluIHppcCgocSxrLGMscyksdmVyc2lvbnMsc25hcHNob3RzKToKICAgICAgICAgICAgICAgIGFzc2VydCB0ZW5zb3IuX3ZlcnNpb24gPT0gdmVyc2lvbgogICAgICAgICAgICAgICAgZXhhY3QodGVuc29yLHNuYXApCiAgICAgICAgICAgIGFzc2VydCB0b3JjaC5lcXVhbChybmcsdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkpCiAgICAgICAgICAgIHJvdyA9IGRpY3QoVD10LEI9NCxIcT0xMixIaz0yLEQ9MTI4LGNvZWZmaWNpZW50X2JhdGNoPTEsCiAgICAgICAgICAgICAgICAgICAgICAgaW5wdXRfc3RyaWRlcz1bbGlzdChxLnN0cmlkZSgpKSxsaXN0KGsuc3RyaWRlKCkpXSwKICAgICAgICAgICAgICAgICAgICAgICB1cHN0cmVhbV9zdHJpZGVzPVtsaXN0KGcuc3RyaWRlKCkpIGZvciBnIGluIGdzXSwKICAgICAgICAgICAgICAgICAgICAgICBleGFjdF92YWx1ZV9jbGFzc19zaWduZWR6ZXJvX2ZvcndhcmRfYmFja3dhcmQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICBjb2xkX2ZpcnN0X2NhbGxfd2FsbF9zPXsnZm9yd2FyZCc6Zm9yd2FyZF9zLCdiYWNrd2FyZCc6YmFja3dhcmRfc30sCiAgICAgICAgICAgICAgICAgICAgICAgaml0X2NhY2hlX2VudHJpZXM9eydiZWZvcmUnOmJlZm9yZSwnYWZ0ZXJfZm9yd2FyZCc6YWZ0ZXJfZm9yd2FyZCwnYWZ0ZXJfYmFja3dhcmQnOmFmdGVyX2JhY2t3YXJkfSkKICAgICAgICAgICAgcm93Wyd3YXJtX2ZvcndhcmQnXSA9IHdhcm0obGFtYmRhOiBmbihxLGssYyxzKSkKICAgICAgICAgICAgcm93Wyd3YXJtX2JhY2t3YXJkJ10gPSB3YXJtKGJhY2t3YXJkKQogICAgICAgICAgICBhc3NlcnQgdG9yY2guZXF1YWwocm5nLHRvcmNoLmN1ZGEuZ2V0X3JuZ19zdGF0ZSgpKQogICAgICAgICAgICByZXBvcnRbJ3Jvd3MnXS5hcHBlbmQocm93KTsgd3JpdGUob3V0LHJlcG9ydCkKICAgICAgICAgICAgZGVsIHksZHksZXhwZWN0ZWQsZXhwZWN0ZWRfZ3JhZCxiYWNrd2FyZCxxLGssYyxzLGdzLHNuYXBzaG90cwogICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgICAgIHJlcG9ydC51cGRhdGUoc3RhdHVzPSdQQVNTRURfTUlDUk9CRU5DSF9PTkxZJyxncHVfZXhlY3V0ZWQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgIHRvcmNoPXRvcmNoLl9fdmVyc2lvbl9fLHRyaXRvbj10cml0b24uX192ZXJzaW9uX18sbmF0aXZlX3NvdXJjZV9ibG9iPWJsb2IsCiAgICAgICAgICAgICAgICAgICAgICBmaW5hbF9qaXRfY2FjaGVfZW50cmllcz1rZXlfY291bnRzKCkpCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVwb3J0LnVwZGF0ZShlcnJvcj1yZXByKGV4YyksdHJhY2ViYWNrPXRyYWNlYmFjay5mb3JtYXRfZXhjKCkpCiAgICBmaW5hbGx5OgogICAgICAgIHdyaXRlKG91dCxyZXBvcnQpCiAgICBwcmludChqc29uLmR1bXBzKHsnc2NvcGUnOidNSUNST0JFTkNIX09OTFknLCdtZXRob2QnOmFyZ3MubWV0aG9kLCdzdGF0dXMnOnJlcG9ydFsnc3RhdHVzJ10sCiAgICAgICAgICAgICAgICAgICAgICAncm93cyc6bGVuKHJlcG9ydFsncm93cyddKSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfSksZmx1c2g9VHJ1ZSkKICAgIHJhaXNlIFN5c3RlbUV4aXQoMCBpZiByZXBvcnRbJ3N0YXR1cyddPT0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWScgZWxzZSAxKQoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzogbWFpbigpCg==', 'driver.py': 'IiIiQm91bmRlZCBuYXRpdmUvb2xkL3J1bnRpbWUgcm90YXJ5IE1JQ1JPQkVOQ0hfT05MWSBvcmNoZXN0cmF0aW9uLgoKTmV2ZXIgaW5zdGFsbHMgaW50byBTT1VQX1JFUE8gb3IgY2FsbHMgYSB0cmFpbmluZyBoYXJuZXNzLiBFeGlzdGluZyBzb3VyY2UvcnVudGltZSwKbG9jaywgcGlkZmQvc3VicmVhcGVyLCBxdWFyYW50aW5lIGFuZCBpZGxlLWNoZWNrIGhlbHBlcnMgYXJlIHByZXNlcnZlZCB1bmNoYW5nZWQuClRoZSBkZXBsb3llZCBzZXJ2aWNlJ3MgbGltaXRzIHJlbWFpbiBhdXRob3JpdGF0aXZlOyBlYWNoIEdQVSBjaGlsZCBnZXRzIDw9ODAgcy4KIiIiCmltcG9ydCBmY250bCwgaGFzaGxpYiwganNvbiwgb3MsIHN5cywgdGltZSwgdXVpZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHByb2Nlc3NfZ3VhcmQKaW1wb3J0IG93bmVkX3Byb2Nlc3NfdHJlZSBhcyB0cmVlCmZyb20gcmV2aWV3ZWRfc3VwcG9ydCBpbXBvcnQgdmVyaWZ5X29mZmljaWFsLCBpZGxlX2NoZWNrLCBFeHBvcnRlciwgZXhwb3J0X3NtYWxsLCByZWFkLCB3cml0ZSwgc2hhCgpMT0NLID0gUGF0aCgnL3RtcC9zb3VwLWdwdTAubG9jaycpCk1BUktFUiA9IFBhdGgoJy90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCcpCgpkZWYgbWFpbigpOgogICAgaGVyZSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKICAgIGlmIG5vdCBvcy5lbnZpcm9uLmdldCgnU09VUF9SRVBPJykgb3Igbm90IG9zLmVudmlyb24uZ2V0KCdTT1VQX1JFU1VMVFMnKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCdSZXF1aXJlcyBkZXBsb3llZCBTT1VQX1JFUE8vU09VUF9SRVNVTFRTLCBubyBtYW51YWwgZmFsbGJhY2snKQogICAgcmVwbywgcmVzdWx0cyA9IFBhdGgob3MuZW52aXJvblsnU09VUF9SRVBPJ10pLmFic29sdXRlKCksIFBhdGgob3MuZW52aXJvblsnU09VUF9SRVNVTFRTJ10pLmFic29sdXRlKCkKICAgIGlmIHJlcG8uaXNfc3ltbGluaygpIG9yIHJlc3VsdHMuaXNfc3ltbGluaygpIG9yIHJlcG8gPT0gcmVzdWx0czoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCdQcmVwYXJlZCBjaGVja291dC9yZXN1bHRzIG11c3QgYmUgc2VwYXJhdGUgcmVhbCBkaXJlY3RvcmllcycpCiAgICBhc3NlcnQgUGF0aCgnL2thZ2dsZS93b3JraW5nJykuaXNfZGlyKCkKICAgIHJlc3VsdHMubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBpZGVudGl0eSA9ICdyb3BlLW1pY3JvYmVuY2gtb25seS0nICsgdXVpZC51dWlkNCgpLmhleFs6MTJdCiAgICBydW4sIGV4cG9ydCA9IFBhdGgoJy9rYWdnbGUvd29ya2luZycpL2lkZW50aXR5LCByZXN1bHRzL2lkZW50aXR5CiAgICBydW4ubWtkaXIoZXhpc3Rfb2s9RmFsc2UpOyBleHBvcnQubWtkaXIoZXhpc3Rfb2s9RmFsc2UpCiAgICBzdGF0dXMgPSBkaWN0KHN0YXR1cz0nUFJFUEFSSU5HX01JQ1JPQkVOQ0hfT05MWScsc2NvcGU9J01JQ1JPQkVOQ0hfT05MWScsCiAgICAgICAgICAgICAgICAgIHRyYWluaW5nX2V4ZWN1dGVkPUZhbHNlLGNvbXBsZXRlX2ZpeGVkX3Rhc2s9RmFsc2Usc2NvcmU9Tm9uZSxhZG1pc3Npb249J1VOS05PV04nLAogICAgICAgICAgICAgICAgICBtZXRob2RfdmFyaWFudHM9eyduYXRpdmUnOiduYXRpdmVfcXdlbjInLCdvbGQnOidydW50aW1lX1Rfc3RyaWRlc18xZCcsJ3J1bnRpbWUnOidydW50aW1lX1Rfc3RyaWRlc18yZF9ncmlkJ30sCiAgICAgICAgICAgICAgICAgIHBlcl9ncHVfY2hpbGRfd2FsbF9jYXBfcz04MCxjYXBfYnlwYXNzPUZhbHNlLAogICAgICAgICAgICAgICAgICBjYW5kaWRhdGVfc291cmNlX3NoYTI1Nj1yZWFkKGhlcmUvJ3NvdXJjZV9tYW5pZmVzdC5qc29uJyksCiAgICAgICAgICAgICAgICAgIG1ldGhvZHNfaW5fZXhlY3V0aW9uX29yZGVyPVsnbmF0aXZlJywnb2xkJywncnVudGltZSddLHByb2Nlc3Nlcz17fSkKICAgIHdyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgIGxvY2tmZCA9IE5vbmU7IHN0YXJ0ZWQgPSB0aW1lLm1vbm90b25pYygpCiAgICB0cnk6CiAgICAgICAgZm9yIG5hbWUsZGlnZXN0IGluIHJlYWQoaGVyZS8ncGF5bG9hZF9tYW5pZmVzdC5qc29uJykuaXRlbXMoKToKICAgICAgICAgICAgcCA9IGhlcmUvbmFtZQogICAgICAgICAgICBhc3NlcnQgbm90IHAuaXNfc3ltbGluaygpIGFuZCBwLmlzX2ZpbGUoKSBhbmQgc2hhKHApPT1kaWdlc3QsICdQYXlsb2FkIGNoYW5nZWQ6ICcrbmFtZQogICAgICAgIGlmIE1BUktFUi5leGlzdHMoKTogcmFpc2UgUnVudGltZUVycm9yKCdHUFUgcXVhcmFudGluZSByZXF1aXJlcyBvcGVyYXRvciByZWNvbmNpbGlhdGlvbicpCiAgICAgICAgbG9ja2ZkID0gb3Mub3BlbihMT0NLLG9zLk9fQ1JFQVR8b3MuT19SRFdSfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICAgICAgZmNudGwuZmxvY2sobG9ja2ZkLGZjbnRsLkxPQ0tfRVh8ZmNudGwuTE9DS19OQikKICAgICAgICBpZiBNQVJLRVIuZXhpc3RzKCk6IHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVIHF1YXJhbnRpbmUgYXBwZWFyZWQgd2hpbGUgbG9ja2luZycpCiAgICAgICAgdHJlZS5wcmVwYXJlKCk7IHRyZWUuYmVnaW5fcXVhcmFudGluZShNQVJLRVIpCiAgICAgICAgb2ZmaWNpYWwgPSByZWFkKGhlcmUvJ29mZmljaWFsX21hbmlmZXN0Lmpzb24nKQogICAgICAgIHZlcmlmeV9vZmZpY2lhbChyZXBvLG9mZmljaWFsKQogICAgICAgIGVudiA9IGRpY3Qob3MuZW52aXJvbixIRl9IVUJfT0ZGTElORT0nMScsSEZfREFUQVNFVFNfT0ZGTElORT0nMScsVFJBTlNGT1JNRVJTX09GRkxJTkU9JzEnLAogICAgICAgICAgICAgICAgICAgUFlUSE9ORE9OVFdSSVRFQllURUNPREU9JzEnLFBZVEhPTlVOQlVGRkVSRUQ9JzEnKQogICAgICAgIGVudi5wb3AoJ1NPVVBfUk9UQVJZX0NPREVHRU5fRElSJyxOb25lKQogICAgICAgIHdpdGggRXhwb3J0ZXIocnVuLGV4cG9ydCk6CiAgICAgICAgICAgIHByb2JlID0gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctQicsc3RyKGhlcmUvJ3J1bnRpbWVfcHJvYmUucHknKSxzdHIocnVuLydydW50aW1lX3Byb2JlLmpzb24nKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydydW50aW1lX3Byb2JlLmxvZycsdGltZW91dD00MCxsb2NrX2ZkPWxvY2tmZCkKICAgICAgICAgICAgc3RhdHVzWydwcm9jZXNzZXMnXVsncnVudGltZV9wcm9iZSddID0gcHJvYmUKICAgICAgICAgICAgaWYgcHJvYmVbJ2V4aXRfY29kZSddIG9yIG5vdCByZWFkKHJ1bi8ncnVudGltZV9wcm9iZS5qc29uJykuZ2V0KCdwYXNzZWQnKToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignRXhhY3QgZGVwbG95ZWQgcnVudGltZSBnYXRlIGZhaWxlZDsgbm8gYmVuY2htYXJrJykKICAgICAgICAgICAgZGVmIGlkbGUobGFiZWwpOgogICAgICAgICAgICAgICAgZm9sZGVyPXJ1bi8oJ2lkbGVfJytsYWJlbCk7Zm9sZGVyLm1rZGlyKGV4aXN0X29rPUZhbHNlKQogICAgICAgICAgICAgICAgcmVzdWx0PWlkbGVfY2hlY2soZW52LHJlcG8sZm9sZGVyLGxvY2tmZCkKICAgICAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bJ2lkbGVfJytsYWJlbF09cmVzdWx0CiAgICAgICAgICAgICAgICBpZiByZXN1bHRbJ2V4aXRfY29kZSddOnJhaXNlIFJ1bnRpbWVFcnJvcignSWRsZSBnYXRlIGZhaWxlZDogJytsYWJlbCkKICAgICAgICAgICAgaWRsZSgnYmVmb3JlX3ByZWZsaWdodCcpCiAgICAgICAgICAgICMgRXhpc3Rpbmcgb3JpZ2luYWwgbnVtZXJpY2FsL1BUWCBnYXRlLCBpc29sYXRlZCBmcm9tIGV2ZXJ5IG1lYXN1cmVkIGNhY2hlLgogICAgICAgICAgICBwcmVlbnYgPSBkaWN0KGVudixUUklUT05fQ0FDSEVfRElSPXN0cihydW4vJ3ByZWZsaWdodF90cml0b24nKSwKICAgICAgICAgICAgICAgICAgICAgICAgICBUT1JDSElORFVDVE9SX0NBQ0hFX0RJUj1zdHIocnVuLydwcmVmbGlnaHRfaW5kdWN0b3InKSkKICAgICAgICAgICAgcHJlID0gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctQicsJy1tJywnc29sdXRpb24ud2ViX3ByZWZsaWdodCcsJy0tb3V0JyxzdHIocnVuLydwcmVmbGlnaHQuanNvbicpXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3dkPWhlcmUsZW52PXByZWVudixsb2c9cnVuLydwcmVmbGlnaHQubG9nJyx0aW1lb3V0PTgwLGxvY2tfZmQ9bG9ja2ZkKQogICAgICAgICAgICBzdGF0dXNbJ3Byb2Nlc3NlcyddWydwcmVmbGlnaHQnXSA9IHByZQogICAgICAgICAgICBpZiBwcmVbJ2V4aXRfY29kZSddIG9yIHJlYWQocnVuLydwcmVmbGlnaHQuanNvbicpLmdldCgnc3RhdHVzJykhPSdQQVNTRURfUFJFRkxJR0hUX09OTFknOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdSdW50aW1lIHZhcmlhbnQgb3JpZ2luYWwgbnVtZXJpY2FsL1BUWCBnYXRlIGZhaWxlZDsgbm8gYmVuY2htYXJrJykKICAgICAgICAgICAgc3RhdHVzWydzdGF0dXMnXT0nUlVOTklOR19NSUNST0JFTkNIX09OTFknO3dyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgICAgICAgICAgaWRsZSgncHJlZmxpZ2h0JykKICAgICAgICAgICAgZm9yIG1ldGhvZCBpbiBzdGF0dXNbJ21ldGhvZHNfaW5fZXhlY3V0aW9uX29yZGVyJ106CiAgICAgICAgICAgICAgICBjYWNoZT1ydW4vJ3RyaXRvbl9jYWNoZScvbWV0aG9kO2NhY2hlLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1GYWxzZSkKICAgICAgICAgICAgICAgIGNoaWxkZW52PWRpY3QoZW52LFRSSVRPTl9DQUNIRV9ESVI9c3RyKGNhY2hlKSxUT1JDSElORFVDVE9SX0NBQ0hFX0RJUj1zdHIocnVuLydpbmR1Y3Rvcl9jYWNoZScvbWV0aG9kKSkKICAgICAgICAgICAgICAgIHByb2Nlc3M9cHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctQicsc3RyKGhlcmUvJ3dvcmtlci5weScpLCctLW1ldGhvZCcsbWV0aG9kLCctLW91dCcsc3RyKHJ1bi8obWV0aG9kKycuanNvbicpKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGN3ZD1oZXJlLGVudj1jaGlsZGVudixsb2c9cnVuLyhtZXRob2QrJy5sb2cnKSx0aW1lb3V0PTgwLGxvY2tfZmQ9bG9ja2ZkKQogICAgICAgICAgICAgICAgc3RhdHVzWydwcm9jZXNzZXMnXVttZXRob2RdPXByb2Nlc3MKICAgICAgICAgICAgICAgIGlmIHByb2Nlc3NbJ2V4aXRfY29kZSddIG9yIHJlYWQocnVuLyhtZXRob2QrJy5qc29uJykpLmdldCgnc3RhdHVzJykhPSdQQVNTRURfTUlDUk9CRU5DSF9PTkxZJzoKICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0luY29tcGxldGUvZmFpbGVkL2NlbnNvcmVkIG1pY3JvYmVuY2htYXJrOiAnK21ldGhvZCkKICAgICAgICAgICAgICAgIGlkbGUobWV0aG9kKQogICAgICAgICAgICAgICAgd3JpdGUoZXhwb3J0LydzdGF0dXMuanNvbicsc3RhdHVzKQogICAgICAgICAgICB2ZXJpZnlfb2ZmaWNpYWwocmVwbyxvZmZpY2lhbCkKICAgICAgICAgICAgcmVwb3J0cz17bWV0aG9kOnJlYWQocnVuLyhtZXRob2QrJy5qc29uJykpIGZvciBtZXRob2QgaW4gc3RhdHVzWydtZXRob2RzX2luX2V4ZWN1dGlvbl9vcmRlciddfQogICAgICAgICAgICBhc3NlcnQgYWxsKGxlbihyWydyb3dzJ10pPT01IGFuZCByWyd0cmFpbmluZ19leGVjdXRlZCddIGlzIEZhbHNlIGFuZCByWydjb21wbGV0ZV9maXhlZF90YXNrJ10gaXMgRmFsc2UgZm9yIHIgaW4gcmVwb3J0cy52YWx1ZXMoKSkKICAgICAgICAgICAgd3JpdGUocnVuLydjb21wYXJpc29uLmpzb24nLGRpY3Qoc2NvcGU9J01JQ1JPQkVOQ0hfT05MWScscmVwb3J0cz1yZXBvcnRzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNjb3JlPU5vbmUsYWRtaXNzaW9uPSdVTktOT1dOJyxjb21wbGV0ZV9maXhlZF90YXNrPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5vdGU9J1JhdyBmaXJzdC1jYWxsIHdhbGwgYW5kIHdhcm0gQ1VEQS1ldmVudCBzYW1wbGVzLCBub3QgcHVyZSBjb21waWxlIGNvc3Qgb3IgdGFzayBzcGVlZHVwOyBubyByYXRpb3Mgb3Igb3duZXJzaGlwLXNoYXJlIGluZmVyZW5jZS4nKSkKICAgICAgICAgICAgc3RhdHVzWydzdGF0dXMnXT0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWScKICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBzdGF0dXMudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9GQUlMRURfT1JfQ0VOU09SRURfTUlDUk9CRU5DSF9PTkxZJyxlcnJvcj1yZXByKGV4YykpCiAgICAgICAgaWYgbG9ja2ZkIGlzIG5vdCBOb25lOnRyZWUucXVhcmFudGluZSgnUm9QRSBtaWNyb2JlbmNobWFyayBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgZmluYWxseToKICAgICAgICBzdGF0dXNbJ29yY2hlc3RyYXRvcl93YWxsX3MnXT10aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQKICAgICAgICBleHBvcnRfc21hbGwocnVuLGV4cG9ydCk7d3JpdGUoZXhwb3J0LydzdGF0dXMuanNvbicsc3RhdHVzKQogICAgICAgIGlmIGxvY2tmZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgdHJlZS5maW5pc2hfcXVhcmFudGluZSgpO29zLmNsb3NlKGxvY2tmZCkKICAgIHByaW50KGpzb24uZHVtcHMoeydzY29wZSc6J01JQ1JPQkVOQ0hfT05MWScsJ3N0YXR1cyc6c3RhdHVzWydzdGF0dXMnXSwnZXhwb3J0JzpzdHIoZXhwb3J0KSwKICAgICAgICAgICAgICAgICAgICAgICdzY29yZSc6Tm9uZSwnYWRtaXNzaW9uJzonVU5LTk9XTid9KSxmbHVzaD1UcnVlKQogICAgcmFpc2UgU3lzdGVtRXhpdCgwIGlmIHN0YXR1c1snc3RhdHVzJ109PSdQQVNTRURfTUlDUk9CRU5DSF9PTkxZJyBlbHNlIDEpCgppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'source_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICI0MGIyNjQ2MTRlYWY2YTA3MzhiOWYyNWYwZmY5MTUxYzJlMTAzMmU4NDA0M2EwZGRmYjYxOGZjNDA1YzRiOGExIiwKICAic29sdXRpb24vY3B1X2NvbnRyYWN0LnB5IjogIjYzNWM3Y2U3MTcxNjM0ZWUxOGZmNjEwMGU0OGY5MDE2Y2M3NjczNGM2YjMzZWI5MjZiNDVkYjM4MmZiYTE1NTAiLAogICJzb2x1dGlvbi9ncHVfcHJlZmxpZ2h0LnB5IjogIjNjYjYyZGVlNWI4OTdmZDY4OWQ5MWJmYTdmZGU1ZGUyNzllYjcyOTI3YmIxYTNhOWRlZjZkODJmODZmOTY3ZmQiLAogICJzb2x1dGlvbi9yb3RhcnkucHkiOiAiYTA3NzIwZjMxMDZhOTQxNDRiNzRhOTcxOTUwNTgzNTc0Yjc4OWZhMDE4OGNlNzMwMGM0Y2JlMzBhMWU4M2MwNyIsCiAgInNvbHV0aW9uL3Rlc3Rfcm90YXJ5LnB5IjogIjA1ZGZlMDlhOGEyZjJhOWJhYTg3MmQ3OTU4MzhjMWNlMmY1OGJhYWQ4YjJiMzE4N2FhYTQyZTY0NjlhMzgwNjIiLAogICJzb2x1dGlvbi93ZWJfcHJlZmxpZ2h0LnB5IjogIjFkOTcwZWEwZTg0ODhmZDEyMjIzNThiMWViZmZkNzc5NzI2NGZkNWZmYmFiZTI1MjNmNjZkMjAzZDMxMmY0OTEiLAogICJvbGRfc29sdXRpb24vX19pbml0X18ucHkiOiAiNDBiMjY0NjE0ZWFmNmEwNzM4YjlmMjVmMGZmOTE1MWMyZTEwMzJlODQwNDNhMGRkZmI2MThmYzQwNWM0YjhhMSIsCiAgIm9sZF9zb2x1dGlvbi9jcHVfY29udHJhY3QucHkiOiAiNjM1YzdjZTcxNzE2MzRlZTE4ZmY2MTAwZTQ4ZjkwMTZjYzc2NzM0YzZiMzNlYjkyNmI0NWRiMzgyZmJhMTU1MCIsCiAgIm9sZF9zb2x1dGlvbi9ncHVfcHJlZmxpZ2h0LnB5IjogIjNjYjYyZGVlNWI4OTdmZDY4OWQ5MWJmYTdmZGU1ZGUyNzllYjcyOTI3YmIxYTNhOWRlZjZkODJmODZmOTY3ZmQiLAogICJvbGRfc29sdXRpb24vcm90YXJ5LnB5IjogImNiZWE1YzdhNjRjNzA2OTVmYTAwOWE3Yjk4MTI5ODg5ZDllYzQzNDk5M2RhNGY5ZDEyM2E4NzgwOGVlNzY2YjAiLAogICJvbGRfc29sdXRpb24vdGVzdF9yb3RhcnkucHkiOiAiMDVkZmUwOWE4YTJmMmE5YmFhODcyZDc5NTgzOGMxY2UyZjU4YmFhZDhiMmIzMTg3YWFhNDJlNjQ2OWEzODA2MiIsCiAgIm9sZF9zb2x1dGlvbi93ZWJfcHJlZmxpZ2h0LnB5IjogIjFkOTcwZWEwZTg0ODhmZDEyMjIzNThiMWViZmZkNzc5NzI2NGZkNWZmYmFiZTI1MjNmNjZkMjAzZDMxMmY0OTEiCn0K', 'payload_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICI0MGIyNjQ2MTRlYWY2YTA3MzhiOWYyNWYwZmY5MTUxYzJlMTAzMmU4NDA0M2EwZGRmYjYxOGZjNDA1YzRiOGExIiwKICAic29sdXRpb24vY3B1X2NvbnRyYWN0LnB5IjogIjYzNWM3Y2U3MTcxNjM0ZWUxOGZmNjEwMGU0OGY5MDE2Y2M3NjczNGM2YjMzZWI5MjZiNDVkYjM4MmZiYTE1NTAiLAogICJzb2x1dGlvbi9ncHVfcHJlZmxpZ2h0LnB5IjogIjNjYjYyZGVlNWI4OTdmZDY4OWQ5MWJmYTdmZGU1ZGUyNzllYjcyOTI3YmIxYTNhOWRlZjZkODJmODZmOTY3ZmQiLAogICJzb2x1dGlvbi9yb3RhcnkucHkiOiAiYTA3NzIwZjMxMDZhOTQxNDRiNzRhOTcxOTUwNTgzNTc0Yjc4OWZhMDE4OGNlNzMwMGM0Y2JlMzBhMWU4M2MwNyIsCiAgInNvbHV0aW9uL3Rlc3Rfcm90YXJ5LnB5IjogIjA1ZGZlMDlhOGEyZjJhOWJhYTg3MmQ3OTU4MzhjMWNlMmY1OGJhYWQ4YjJiMzE4N2FhYTQyZTY0NjlhMzgwNjIiLAogICJzb2x1dGlvbi93ZWJfcHJlZmxpZ2h0LnB5IjogIjFkOTcwZWEwZTg0ODhmZDEyMjIzNThiMWViZmZkNzc5NzI2NGZkNWZmYmFiZTI1MjNmNjZkMjAzZDMxMmY0OTEiLAogICJvbGRfc29sdXRpb24vX19pbml0X18ucHkiOiAiNDBiMjY0NjE0ZWFmNmEwNzM4YjlmMjVmMGZmOTE1MWMyZTEwMzJlODQwNDNhMGRkZmI2MThmYzQwNWM0YjhhMSIsCiAgIm9sZF9zb2x1dGlvbi9jcHVfY29udHJhY3QucHkiOiAiNjM1YzdjZTcxNzE2MzRlZTE4ZmY2MTAwZTQ4ZjkwMTZjYzc2NzM0YzZiMzNlYjkyNmI0NWRiMzgyZmJhMTU1MCIsCiAgIm9sZF9zb2x1dGlvbi9ncHVfcHJlZmxpZ2h0LnB5IjogIjNjYjYyZGVlNWI4OTdmZDY4OWQ5MWJmYTdmZGU1ZGUyNzllYjcyOTI3YmIxYTNhOWRlZjZkODJmODZmOTY3ZmQiLAogICJvbGRfc29sdXRpb24vcm90YXJ5LnB5IjogImNiZWE1YzdhNjRjNzA2OTVmYTAwOWE3Yjk4MTI5ODg5ZDllYzQzNDk5M2RhNGY5ZDEyM2E4NzgwOGVlNzY2YjAiLAogICJvbGRfc29sdXRpb24vdGVzdF9yb3RhcnkucHkiOiAiMDVkZmUwOWE4YTJmMmE5YmFhODcyZDc5NTgzOGMxY2UyZjU4YmFhZDhiMmIzMTg3YWFhNDJlNjQ2OWEzODA2MiIsCiAgIm9sZF9zb2x1dGlvbi93ZWJfcHJlZmxpZ2h0LnB5IjogIjFkOTcwZWEwZTg0ODhmZDEyMjIzNThiMWViZmZkNzc5NzI2NGZkNWZmYmFiZTI1MjNmNjZkMjAzZDMxMmY0OTEiLAogICJwcm9jZXNzX2d1YXJkLnB5IjogIjgyNDY1OTQzYTNhOWFmMzgzMWE1Mzk5YTNhNTVkMDlhYzI5MGJhNzM2NWM0ZTgwZDZiYzVlNzlmYWI2MmE2ZmYiLAogICJvd25lZF9wcm9jZXNzX3RyZWUucHkiOiAiNjkzZGRlNzA4ZGE1YzA2MDUzYWYyMzk0NjM5NTIxMTVlZjVlODg2NWY1MGJjYmM1MWEyMDM4MjM3Y2E1N2Q2OSIsCiAgInJ1bnRpbWVfcHJvYmUucHkiOiAiZWZlMWI3Y2YzZjNjZjFlMzdiYmMxNjlhODQ2ODg1OWM1MTI3ZWMzMjgxYTM3ZTM5MDY4NzAzYjJlYjNjMTQ4MCIsCiAgIm9mZmljaWFsX21hbmlmZXN0Lmpzb24iOiAiZDAxM2UxNjMzOTQwYzI3ZDcwOGUyNDc0YTgyODA5OTAwMDBlMzMwMThlODg5ZWQxNDY1NWM4NTYzMjlmZjFkOCIsCiAgInJldmlld2VkX3N1cHBvcnQucHkiOiAiN2M1ZDYwMmQwNzEzODNlZjA3ZmRkZDk5NjcwZWQxY2NlMzE0MWZlYWJkNWFmODM5NGYwNTNjNmI0ZTA3MTIyYyIsCiAgIndvcmtlci5weSI6ICIxMjQ4N2Y5NDlmNDkwZGI5NWQwMDcyNDE5OTQ5Y2VhODNmODFlMTA4OWE1MWMwOTAwMzBhNGMzOTFhYjkxNzIzIiwKICAiZHJpdmVyLnB5IjogImUyYmRhMzg0OGEyMTkzZjliNDE4ZDkyZTJlNTgzZWEwYTdjMjZjNDY1MmYyYWU5MDQ2NjkwNDAwMTBiYmI3ZTgiLAogICJzb3VyY2VfbWFuaWZlc3QuanNvbiI6ICI0ZjQzMTBmM2ZkNTc3OTE2ZjJkY2E2MGViMjRhMmVjODMwNDc4NTRjMDhmMjQwOGQwOWU4NGQ1MmYwNTA3Y2I5Igp9Cg=='}
PAYLOAD_SHA256 = {'solution/__init__.py': '40b264614eaf6a0738b9f25f0ff9151c2e1032e84043a0ddfb618fc405c4b8a1', 'solution/cpu_contract.py': '635c7ce7171634ee18ff6100e48f9016cc76734c6b33eb926b45db382fba1550', 'solution/gpu_preflight.py': '3cb62dee5b897fd689d91bfa7fde5de279eb72927bb1a3a9def6d82f86f967fd', 'solution/rotary.py': 'a07720f3106a94144b74a971950583574b789fa0188ce7300c4cbe30a1e83c07', 'solution/test_rotary.py': '05dfe09a8a2f2a9baa872d795838c1ce2f58baad8b2b3187aaa42e6469a38062', 'solution/web_preflight.py': '1d970ea0e8488fd1222358b1ebffd7797264fd5ffbabe2523f66d203d312f491', 'old_solution/__init__.py': '40b264614eaf6a0738b9f25f0ff9151c2e1032e84043a0ddfb618fc405c4b8a1', 'old_solution/cpu_contract.py': '635c7ce7171634ee18ff6100e48f9016cc76734c6b33eb926b45db382fba1550', 'old_solution/gpu_preflight.py': '3cb62dee5b897fd689d91bfa7fde5de279eb72927bb1a3a9def6d82f86f967fd', 'old_solution/rotary.py': 'cbea5c7a64c70695fa009a7b98129889d9ec434993da4f9d123a87808ee766b0', 'old_solution/test_rotary.py': '05dfe09a8a2f2a9baa872d795838c1ce2f58baad8b2b3187aaa42e6469a38062', 'old_solution/web_preflight.py': '1d970ea0e8488fd1222358b1ebffd7797264fd5ffbabe2523f66d203d312f491', 'process_guard.py': '82465943a3a9af3831a5399a3a55d09ac290ba7365c4e80d6bc5e79fab62a6ff', 'owned_process_tree.py': '693dde708da5c06053af239463952115ef5e8865f50bcbc51a2038237ca57d69', 'runtime_probe.py': 'efe1b7cf3f3cf1e37bbc169a8468859c5127ec3281a37e39068703b2eb3c1480', 'official_manifest.json': 'd013e1633940c27d708e2474a8280990000e33018e889ed14655c856329ff1d8', 'reviewed_support.py': '7c5d602d071383ef07fddd99670ed1cce3141feabd5af8394f053c6b4e07122c', 'worker.py': '12487f949f490db95d0072419949cea83f81e1089a51c090030a4c391ab91723', 'driver.py': 'e2bda3848a2193f9b418d92e2e583ea0a7c26c4652f2ae904669040010bbb7e8', 'source_manifest.json': '4f4310f3fd577916f2dca60eb24a2ec83047854c08f2408d09e84d52f0507cb9', 'payload_manifest.json': '4b6b01642e6d56e915b39b2f9d8f687524bdc95ccc220506bce9a18a8714d17e'}
STAGED=Path(tempfile.mkdtemp(prefix='rope-microbench-',dir='/kaggle/working'))
for name,encoded in PAYLOAD.items():
    parts=Path(name).parts
    assert not Path(name).is_absolute() and '..' not in parts
    raw=base64.b64decode(encoded,validate=True)
    assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA256[name]
    target=STAGED/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
print('Prepared MICROBENCH_ONLY payload:',STAGED)


In [ ]:
import subprocess,sys,signal
process=subprocess.Popen([sys.executable,'-B',str(STAGED/'driver.py')],start_new_session=True)
try:
    return_code=process.wait()
except BaseException:
    if process.poll() is None:process.send_signal(signal.SIGTERM)
    while process.poll() is None:
        try:process.wait()
        except KeyboardInterrupt:pass
    raise
if return_code:raise RuntimeError('MICROBENCH_ONLY failed or censored; inspect exported status. No automatic retry.')
